In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

assert tf.config.list_physical_devices("GPU"), "GPU not available! Please enable GPU runtime."

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")
DATASET_DIR = PROJECT_DIR / "dataset"

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Dataset folder:", DATASET_DIR)

In [ ]:
!pip -q install -U kaggle==2.2.2

from google.colab import userdata
import os

token = userdata.get("KAGGLE_API_TOKEN")

if token is None:
    raise RuntimeError(
        "KAGGLE_API_TOKEN not found. Add it in Colab Secrets and turn Notebook access ON."
    )

os.environ["KAGGLE_API_TOKEN"] = token

print("Kaggle setup complete ✅")
!kaggle --version

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")
DATASET_DIR = PROJECT_DIR / "dataset"

classes = ["Benign", "InSitu", "Invasive", "Normal"]

for class_name in classes:
    (DATASET_DIR / class_name).mkdir(parents=True, exist_ok=True)

print("Persistent dataset location:")
print(DATASET_DIR)

for class_name in classes:
    folder = DATASET_DIR / class_name
    print(f"{class_name}: {len(list(folder.glob('*.tif')))} existing TIFFs")

In [ ]:
import subprocess
import shutil
import zipfile
from pathlib import Path
from PIL import Image

DATASET = "truthisneverlinear/bach-breast-cancer-histology-images"

TEMP_DIR = Path("/content/bach_download_tmp")
TEMP_DIR.mkdir(parents=True, exist_ok=True)

target_dir = DATASET_DIR / "Benign"
target_file = target_dir / "b001.tif"

remote_file = (
    "ICIAR2018_BACH_Challenge/"
    "ICIAR2018_BACH_Challenge/"
    "Photos/Benign/b001.tif"
)

# Clean any previous test download
for p in TEMP_DIR.glob("*"):
    if p.is_file():
        p.unlink()

# -------------------------
# 1. Download ZIP from Kaggle
# -------------------------

result = subprocess.run(
    [
        "kaggle", "datasets", "download",
        "-d", DATASET,
        "-f", remote_file,
        "-p", str(TEMP_DIR),
        "-o"
    ],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Kaggle download failed.")

# -------------------------
# 2. Find the downloaded ZIP
# -------------------------

zip_file = TEMP_DIR / "b001.tif.zip"

if not zip_file.exists():
    raise FileNotFoundError(f"ZIP file was not found: {zip_file}")

print("ZIP downloaded:", zip_file)

# -------------------------
# 3. Extract the TIFF
# -------------------------

with zipfile.ZipFile(zip_file, "r") as z:
    print("ZIP contents:", z.namelist())
    z.extractall(TEMP_DIR)

temp_file = TEMP_DIR / "b001.tif"

if not temp_file.exists():
    raise FileNotFoundError(
        f"TIFF was not extracted: {temp_file}"
    )

# -------------------------
# 4. Verify the TIFF
# -------------------------

with Image.open(temp_file) as img:
    print("Verified TIFF:")
    print("  Size:", img.size)
    print("  Mode:", img.mode)
    print("  Format:", img.format)
    img.verify()

# -------------------------
# 5. Copy verified TIFF to Drive
# -------------------------

shutil.copy2(temp_file, target_file)

# -------------------------
# 6. Verify Drive copy
# -------------------------

if not target_file.exists():
    raise RuntimeError("Drive copy failed.")

with Image.open(target_file) as img:
    print("\nDrive copy verified:")
    print("  File:", target_file)
    print("  Size:", img.size)
    print("  Mode:", img.mode)
    print("  Format:", img.format)

print("\n✅ b001.tif successfully stored in Google Drive.")

In [ ]:
print(list((DATASET_DIR / "Benign").glob("*.tif")))

In [ ]:
import subprocess
import zipfile
import shutil
from pathlib import Path
from PIL import Image

# ==========================================
# CONFIG
# ==========================================

DATASET = "truthisneverlinear/bach-breast-cancer-histology-images"

DRIVE_DATASET_DIR = Path(
    "/content/drive/MyDrive/BACH_Project/dataset"
)

TEMP_DIR = Path("/content/bach_download_tmp")
TEMP_DIR.mkdir(parents=True, exist_ok=True)

CLASSES = {
    "Benign": "b",
    "InSitu": "is",
    "Invasive": "iv",
    "Normal": "n"
}

REMOTE_ROOT = (
    "ICIAR2018_BACH_Challenge/"
    "ICIAR2018_BACH_Challenge/"
    "Photos"
)

# ==========================================
# FUNCTION: DOWNLOAD ONE IMAGE
# ==========================================

def download_image(class_name, prefix, number):

    filename = f"{prefix}{number:03d}.tif"

    drive_class_dir = DRIVE_DATASET_DIR / class_name
    drive_class_dir.mkdir(parents=True, exist_ok=True)

    final_file = drive_class_dir / filename

    # --------------------------------------
    # Already downloaded? Skip it.
    # --------------------------------------

    if final_file.exists():
        return "skipped"

    remote_file = (
        f"{REMOTE_ROOT}/{class_name}/{filename}"
    )

    zip_file = TEMP_DIR / f"{filename}.zip"
    temp_tif = TEMP_DIR / filename

    # Clean leftovers from a previous interrupted attempt
    if zip_file.exists():
        zip_file.unlink()

    if temp_tif.exists():
        temp_tif.unlink()

    # --------------------------------------
    # Download from Kaggle
    # --------------------------------------

    result = subprocess.run(
        [
            "kaggle",
            "datasets",
            "download",
            "-d", DATASET,
            "-f", remote_file,
            "-p", str(TEMP_DIR),
            "-o"
        ],
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        print(result.stderr)
        return "failed"

    # --------------------------------------
    # Confirm ZIP exists
    # --------------------------------------

    if not zip_file.exists():
        print(f"ZIP missing: {filename}")
        return "failed"

    # --------------------------------------
    # Extract
    # --------------------------------------

    try:
        with zipfile.ZipFile(zip_file, "r") as z:
            z.extractall(TEMP_DIR)

    except Exception as e:
        print(f"Extraction failed: {filename}")
        print(e)
        return "failed"

    # --------------------------------------
    # Confirm TIFF exists
    # --------------------------------------

    if not temp_tif.exists():
        print(f"TIFF missing after extraction: {filename}")
        return "failed"

    # --------------------------------------
    # Verify TIFF can actually be read
    # --------------------------------------

    try:
        with Image.open(temp_tif) as img:
            img.load()

            if img.format != "TIFF":
                print(f"Not TIFF: {filename}")
                return "failed"

            if img.mode != "RGB":
                print(f"Unexpected mode for {filename}: {img.mode}")
                return "failed"

            if img.size != (2048, 1536):
                print(
                    f"Unexpected size for {filename}: "
                    f"{img.size}"
                )
                return "failed"

    except Exception as e:
        print(f"Invalid TIFF: {filename}")
        print(e)
        return "failed"

    # --------------------------------------
    # Copy verified TIFF to Google Drive
    # --------------------------------------

    try:
        shutil.copy2(temp_tif, final_file)

    except Exception as e:
        print(f"Drive copy failed: {filename}")
        print(e)
        return "failed"

    # --------------------------------------
    # Confirm Drive copy exists
    # --------------------------------------

    if not final_file.exists():
        print(f"Drive file missing: {filename}")
        return "failed"

    # --------------------------------------
    # Clean temporary files
    # --------------------------------------

    if zip_file.exists():
        zip_file.unlink()

    if temp_tif.exists():
        temp_tif.unlink()

    return "downloaded"


# ==========================================
# MAIN DOWNLOAD LOOP
# ==========================================

downloaded = 0
skipped = 0
failed = []

for class_name, prefix in CLASSES.items():

    print(f"\n{'=' * 15} {class_name} {'=' * 15}")

    for number in range(1, 101):

        filename = f"{prefix}{number:03d}.tif"

        result = download_image(
            class_name,
            prefix,
            number
        )

        if result == "downloaded":
            downloaded += 1

        elif result == "skipped":
            skipped += 1

        else:
            failed.append(
                f"{class_name}/{filename}"
            )

        # Show progress every 10 images
        completed = downloaded + skipped

        if completed % 10 == 0:
            print(
                f"Progress: {completed}/400 "
                f"| New: {downloaded} "
                f"| Existing: {skipped} "
                f"| Failed: {len(failed)}"
            )

# ==========================================
# FINAL PHYSICAL VERIFICATION
# ==========================================

print("\n")
print("=" * 45)
print("FINAL DATASET VERIFICATION")
print("=" * 45)

grand_total = 0

for class_name in CLASSES:

    class_dir = DRIVE_DATASET_DIR / class_name

    files = list(class_dir.glob("*.tif"))

    count = len(files)
    grand_total += count

    print(f"{class_name:10s}: {count}/100")

print("-" * 45)
print(f"TOTAL:      {grand_total}/400")

if failed:
    print("\nFAILED FILES:")
    for filename in failed:
        print(" -", filename)

elif grand_total == 400:
    print("\n✅ ALL 400 TIFF FILES ARE PHYSICALLY IN GOOGLE DRIVE")

else:
    print(
        "\n⚠️ Download completed, "
        "but the physical file count is not 400."
    )

In [ ]:
from pathlib import Path

DATASET_DIR = Path("/content/drive/MyDrive/BACH_Project/dataset")

classes = ["Benign", "InSitu", "Invasive", "Normal"]

for class_name in classes:
    count = len(list((DATASET_DIR / class_name).glob("*.tif")))
    print(f"{class_name}: {count}")

print("TOTAL:", sum(
    len(list((DATASET_DIR / c).glob("*.tif")))
    for c in classes
))

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

test_file = DATASET_DIR / "Benign" / "b001.tif"

with Image.open(test_file) as img:
    print("Size:", img.size)
    print("Mode:", img.mode)
    print("Format:", img.format)

    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis("off")
    plt.title("BACH — Benign / b001")
    plt.show()

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

In [ ]:
from pathlib import Path

DATASET_DIR = Path("/content/drive/MyDrive/BACH_Project/dataset")

for class_name in ["Benign", "InSitu", "Invasive", "Normal"]:
    print(
        class_name,
        len(list((DATASET_DIR / class_name).glob("*.tif")))
    )

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

DATASET_DIR = Path("/content/drive/MyDrive/BACH_Project/dataset")

classes = ["Benign", "InSitu", "Invasive", "Normal"]

for class_name in classes:
    count = len(list((DATASET_DIR / class_name).glob("*.tif")))
    print(f"{class_name}: {count}")

total = sum(
    len(list((DATASET_DIR / c).glob("*.tif")))
    for c in classes
)

print("TOTAL:", total)

In [ ]:
from pathlib import Path
from PIL import Image

DATASET_DIR = Path("/content/drive/MyDrive/BACH_Project/dataset")

classes = ["Benign", "InSitu", "Invasive", "Normal"]

for class_name in classes:
    files = list((DATASET_DIR / class_name).glob("*.tif"))

    with Image.open(files[0]) as img:
        print(
            f"{class_name}: "
            f"{len(files)} images | "
            f"size={img.size} | "
            f"mode={img.mode} | "
            f"format={img.format}"
        )

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, class_name in zip(axes, classes):

    image_path = next((DATASET_DIR / class_name).glob("*.tif"))

    with Image.open(image_path) as img:
        ax.imshow(img)

    ax.set_title(class_name, fontsize=14)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
#integrity check

In [ ]:
from pathlib import Path
from PIL import Image

DATASET_DIR = Path("/content/drive/MyDrive/BACH_Project/dataset")

classes = ["Benign", "InSitu", "Invasive", "Normal"]

total_checked = 0
errors = []
sizes = set()
modes = set()

for class_name in classes:

    files = sorted((DATASET_DIR / class_name).glob("*.tif"))

    for image_path in files:
        try:
            with Image.open(image_path) as img:
                sizes.add(img.size)
                modes.add(img.mode)
                img.verify()

            total_checked += 1

        except Exception as e:
            errors.append((str(image_path), str(e)))

print("Images checked:", total_checked)
print("Image sizes:", sizes)
print("Image modes:", modes)
print("Errors:", len(errors))

if errors:
    print("\nFirst few errors:")
    for path, error in errors[:10]:
        print(path, "->", error)
else:
    print("✅ All labeled images passed the integrity check.")

In [ ]:
from pathlib import Path
import pandas as pd

from sklearn.model_selection import train_test_split

DATASET_DIR = Path("/content/drive/MyDrive/BACH_Project/dataset")
PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")

classes = ["Benign", "InSitu", "Invasive", "Normal"]

# --------------------------------------------------
# 1. Build a manifest of all 400 labeled images
# --------------------------------------------------

records = []

for class_name in classes:
    class_dir = DATASET_DIR / class_name

    for image_path in sorted(class_dir.glob("*.tif")):
        records.append({
            "path": str(image_path),
            "label": class_name
        })

df = pd.DataFrame(records)

print("Total images:", len(df))
print("\nClass counts:")
print(df["label"].value_counts().sort_index())

# --------------------------------------------------
# 2. 70% training, 30% temporary
# --------------------------------------------------

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)

# --------------------------------------------------
# 3. Split the remaining 30% equally:
#    15% validation + 15% internal test
# --------------------------------------------------

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

# Add split information
train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "validation"
test_df["split"] = "internal_test"

# Combine everything
split_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

# --------------------------------------------------
# 4. Save the split permanently in Google Drive
# --------------------------------------------------

manifest_path = PROJECT_DIR / "split_manifest.csv"
split_df.to_csv(manifest_path, index=False)

print("\nSplit sizes:")
print(split_df["split"].value_counts())

print("\nClass distribution:")
print(
    pd.crosstab(
        split_df["split"],
        split_df["label"]
    )
)

print("\nSaved manifest to:")
print(manifest_path)

In [ ]:
#224 x 224

In [ ]:
from pathlib import Path
import pandas as pd
from PIL import Image

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")
MANIFEST_PATH = PROJECT_DIR / "split_manifest.csv"
PROCESSED_DIR = PROJECT_DIR / "processed_224"

# Load our fixed split
df = pd.read_csv(MANIFEST_PATH)

print("Images in manifest:", len(df))
print(df["split"].value_counts())

# Create output folders
for split in ["train", "validation", "internal_test"]:
    for class_name in ["Benign", "InSitu", "Invasive", "Normal"]:
        (PROCESSED_DIR / split / class_name).mkdir(
            parents=True,
            exist_ok=True
        )

print("\nProcessing images...")

processed = 0
errors = []

for _, row in df.iterrows():

    source = Path(row["path"])
    split = row["split"]
    class_name = row["label"]

    output = (
        PROCESSED_DIR
        / split
        / class_name
        / (source.stem + ".jpg")
    )

    # Resume-safe
    if output.exists():
        processed += 1
        continue

    try:
        with Image.open(source) as img:

            # Ensure RGB
            img = img.convert("RGB")

            # Preserve aspect ratio and pad to 224x224
            img.thumbnail((224, 224), Image.Resampling.LANCZOS)

            # White canvas
            canvas = Image.new(
                "RGB",
                (224, 224),
                (255, 255, 255)
            )

            # Center the resized image
            x = (224 - img.width) // 2
            y = (224 - img.height) // 2

            canvas.paste(img, (x, y))

            canvas.save(
                output,
                "JPEG",
                quality=95
            )

        processed += 1

    except Exception as e:
        errors.append((str(source), str(e)))

    if processed % 50 == 0:
        print(f"Processed: {processed}/400")

print("\n==============================")
print("PROCESSING COMPLETE")
print("==============================")
print("Processed:", processed)
print("Errors:", len(errors))

if errors:
    print("\nFirst errors:")
    for path, error in errors[:10]:
        print(path, "->", error)
else:
    print("✅ All images processed successfully.")

In [ ]:
!du -sh "/content/drive/MyDrive/BACH_Project/processed_224"

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

processed_dir = Path("/content/drive/MyDrive/BACH_Project/processed_224")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, class_name in zip(axes, classes):

    image_path = next(
        (processed_dir / "train" / class_name).glob("*.jpg")
    )

    img = Image.open(image_path)

    ax.imshow(img)
    ax.set_title(class_name)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
#RESIZE + PADDING + JPEG COMPRESS.   VS    NORMAL HIGH RES

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

original_path = DATASET_DIR / "Benign" / "b001.tif"
processed_path = (
    Path("/content/drive/MyDrive/BACH_Project/processed_224")
    / "train" / "Benign" / "b001.jpg"
)

print("Original exists:", original_path.exists())
print("Processed exists:", processed_path.exists())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

with Image.open(original_path) as img:
    axes[0].imshow(img)

axes[0].set_title("Original b001 — 2048×1536")
axes[0].axis("off")

with Image.open(processed_path) as img:
    axes[1].imshow(img)

axes[1].set_title("Processed b001 — 224×224")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import shutil

DRIVE_PROCESSED = Path(
    "/content/drive/MyDrive/BACH_Project/processed_224"
)

LOCAL_PROCESSED = Path(
    "/content/BACH_processed_224"
)

# Remove an old temporary copy if one exists
if LOCAL_PROCESSED.exists():
    shutil.rmtree(LOCAL_PROCESSED)

# Copy processed dataset from Drive → Colab local storage
shutil.copytree(
    DRIVE_PROCESSED,
    LOCAL_PROCESSED
)

print("Copied processed dataset to local Colab storage ✅")
print("Location:", LOCAL_PROCESSED)

In [ ]:
for split in ["train", "validation", "internal_test"]:

    total = 0

    for class_name in ["Benign", "InSitu", "Invasive", "Normal"]:
        folder = LOCAL_PROCESSED / split / class_name
        count = len(list(folder.glob("*.jpg")))
        total += count
        print(f"{split:14s} {class_name:10s}: {count}")

    print(f"{split:14s} TOTAL: {total}\n")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import pandas as pd
import tensorflow as tf

# Project locations in persistent Google Drive
PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")
DATASET_DIR = PROJECT_DIR / "dataset"
PROCESSED_DIR = PROJECT_DIR / "processed_224"
MANIFEST_PATH = PROJECT_DIR / "split_manifest.csv"

CLASSES = ["Benign", "InSitu", "Invasive", "Normal"]
SPLITS = ["train", "validation", "internal_test"]

# Check required files and folders
print("=== PROJECT FILES ===")
for path in [DATASET_DIR, PROCESSED_DIR, MANIFEST_PATH]:
    print(f"{path.name}: {'FOUND ✅' if path.exists() else 'MISSING ❌'}")

if not all(p.exists() for p in [DATASET_DIR, PROCESSED_DIR, MANIFEST_PATH]):
    raise FileNotFoundError(
        "A required project folder/file is missing. Stop here and show me the output."
    )

# Count original TIFF images
print("\n=== ORIGINAL TIFF IMAGES ===")
original_total = 0

for class_name in CLASSES:
    count = len(list((DATASET_DIR / class_name).glob("*.tif")))
    original_total += count
    print(f"{class_name}: {count}")

print("Original total:", original_total)

# Count processed JPEGs in each split
print("\n=== PROCESSED JPEG IMAGES ===")
for split in SPLITS:
    split_total = 0

    for class_name in CLASSES:
        folder = PROCESSED_DIR / split / class_name
        count = len(list(folder.glob("*.jpg")))
        split_total += count
        print(f"{split:14s} {class_name:10s}: {count}")

    print(f"{split:14s} TOTAL: {split_total}")

# Check the saved CSV
manifest = pd.read_csv(MANIFEST_PATH)

print("\n=== SAVED SPLIT CSV ===")
print("Rows:", len(manifest))
print(manifest["split"].value_counts().to_string())

# Check GPU availability
print("\n=== GPU ===")
print(tf.config.list_physical_devices("GPU"))

In [ ]:
from pathlib import Path
import shutil

DRIVE_PROCESSED = Path(
    "/content/drive/MyDrive/BACH_Project/processed_224"
)
LOCAL_PROCESSED = Path("/content/BACH_processed_224")

# Replace only the temporary local copy, never the Drive copy
if LOCAL_PROCESSED.exists():
    shutil.rmtree(LOCAL_PROCESSED)

shutil.copytree(DRIVE_PROCESSED, LOCAL_PROCESSED)

# Verify the copied images
classes = ["Benign", "InSitu", "Invasive", "Normal"]
splits = {
    "train": 70,
    "validation": 15,
    "internal_test": 15
}

all_correct = True

for split, expected_per_class in splits.items():
    split_total = 0

    for class_name in classes:
        folder = LOCAL_PROCESSED / split / class_name
        count = len(list(folder.glob("*.jpg")))
        expected = expected_per_class

        print(f"{split:14s} {class_name:10s}: {count}")

        if count != expected:
            all_correct = False

        split_total += count

    print(f"{split:14s} TOTAL: {split_total}\n")

if all_correct:
    print("✅ Local copy verified: 400 processed images.")
else:
    raise RuntimeError("Image counts do not match. Stop and investigate.")

print("Temporary training folder:", LOCAL_PROCESSED)

TENSORFLOW PIPELINE AFTER PREPROCESSING

In [ ]:
import tensorflow as tf
from pathlib import Path

# Configuration
LOCAL_PROCESSED = Path("/content/BACH_processed_224")

CLASSES = ["Benign", "InSitu", "Invasive", "Normal"]
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

# Function to load one dataset split
def load_split(split, shuffle=False):
    dataset = tf.keras.utils.image_dataset_from_directory(
        str(LOCAL_PROCESSED / split),
        labels="inferred",
        label_mode="int",
        class_names=CLASSES,
        color_mode="rgb",
        image_size=(224, 224),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=42
    )

    return dataset.prefetch(buffer_size=AUTOTUNE)

# Create the three datasets
train_ds = load_split("train", shuffle=True)
val_ds = load_split("validation", shuffle=False)
test_ds = load_split("internal_test", shuffle=False)

print("\n✅ TensorFlow datasets created!")

print("\nClass-to-label mapping:")
for index, name in enumerate(CLASSES):
    print(f"{index} = {name}")

print("\nNumber of batches:")
print("Training:", len(train_ds))
print("Validation:", len(val_ds))
print("Internal test:", len(test_ds))

# Inspect one training batch
images, labels = next(iter(train_ds))

print("\nFirst training batch:")
print("Image shape:", images.shape)
print("Image dtype:", images.dtype)
print("Label shape:", labels.shape)
print("Pixel range:", float(tf.reduce_min(images)),
      "to", float(tf.reduce_max(images)))

print("\nGPU available:",
      bool(tf.config.list_physical_devices("GPU")))

In [ ]:
import tensorflow as tf
from pathlib import Path

# =========================================
# 1. REPRODUCIBILITY AND PATHS
# =========================================

tf.keras.utils.set_random_seed(42)

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODEL_DIR / "baseline_mobilenetv2_best.keras"
LATEST_MODEL_PATH = MODEL_DIR / "baseline_mobilenetv2_last.keras"
LOG_PATH = PROJECT_DIR / "baseline_training_log.csv"

CLASSES = ["Benign", "InSitu", "Invasive", "Normal"]

# Confirm GPU is available
gpus = tf.config.list_physical_devices("GPU")
if not gpus:
    raise RuntimeError("No GPU detected. Stop before training.")

print("GPU:", gpus[0])
print("Saving models to:", MODEL_DIR)

# =========================================
# 2. DATA AUGMENTATION
# Applied during training only
# =========================================

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.25),
    tf.keras.layers.RandomZoom(0.10),
], name="data_augmentation")

# =========================================
# 3. LOAD PRETRAINED MOBILENETV2
# =========================================

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze pretrained layers for the baseline
base_model.trainable = False

# =========================================
# 4. BUILD OUR CLASSIFIER
# =========================================

inputs = tf.keras.Input(shape=(224, 224, 3))

x = data_augmentation(inputs)

# MobileNetV2 expects values scaled to approximately [-1, 1]
x = tf.keras.layers.Rescaling(
    scale=1.0 / 127.5,
    offset=-1
)(x)

x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.30)(x)

outputs = tf.keras.layers.Dense(
    len(CLASSES),
    activation="softmax",
    name="class_predictions"
)(x)

model = tf.keras.Model(inputs, outputs, name="BACH_MobileNetV2")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("\nModel architecture:")
model.summary()

# =========================================
# 5. CALLBACKS — SAVE PROGRESS TO DRIVE
# =========================================

callbacks = [
    # Preserve the model with the best validation loss
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    # Save the latest completed epoch as well
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(LATEST_MODEL_PATH),
        save_best_only=False,
        verbose=0
    ),

    # Stop if validation stops improving
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),

    # Reduce learning rate when validation progress stalls
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),

    # Record epoch metrics in a CSV on Drive
    tf.keras.callbacks.CSVLogger(
        str(LOG_PATH),
        append=False
    )
]

# =========================================
# 6. TRAIN
# =========================================

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks
)

# =========================================
# 7. CONFIRM SAVED ARTIFACTS
# =========================================

print("\n========== TRAINING FINISHED ==========")
print("Best model saved:", BEST_MODEL_PATH.exists())
print("Latest model saved:", LATEST_MODEL_PATH.exists())
print("Training log saved:", LOG_PATH.exists())

print("\nTraining log:")
print(tf.io.gfile.GFile(str(LOG_PATH)).read()[:1500])

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Persistent project paths
PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")
LOG_PATH = PROJECT_DIR / "baseline_training_log.csv"

FIGURE_DIR = PROJECT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Confirm the saved training log exists
if not LOG_PATH.exists():
    raise FileNotFoundError(
        f"Training log not found: {LOG_PATH}. "
        "Check that Google Drive is mounted."
    )

# Read metrics from the completed training run
history_df = pd.read_csv(LOG_PATH)
epochs = history_df["epoch"] + 1

print("Epochs recorded:", len(history_df))
print("Log columns:", list(history_df.columns))

# Find epoch with lowest validation loss
best_idx = history_df["val_loss"].idxmin()
best_row = history_df.loc[best_idx]

print(f"\nBest validation-loss epoch: {int(best_row['epoch']) + 1}")
print(f"Training accuracy:   {best_row['accuracy']:.4f}")
print(f"Validation accuracy: {best_row['val_accuracy']:.4f}")
print(f"Training loss:       {best_row['loss']:.4f}")
print(f"Validation loss:     {best_row['val_loss']:.4f}")

# Plot both learning curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(epochs, history_df["accuracy"], marker="o",
             label="Training accuracy")
axes[0].plot(epochs, history_df["val_accuracy"], marker="o",
             label="Validation accuracy")
axes[0].set_title("Baseline Model — Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0, 1)
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Loss
axes[1].plot(epochs, history_df["loss"], marker="o",
             label="Training loss")
axes[1].plot(epochs, history_df["val_loss"], marker="o",
             label="Validation loss")
axes[1].set_title("Baseline Model — Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()

# Save the plots permanently in Google Drive
plot_path = FIGURE_DIR / "baseline_learning_curves.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()

print("\nLearning curves saved to:")
print(plot_path)

In [ ]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    accuracy_score
)

# -----------------------------------------
# 1. Define paths
# -----------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")

BEST_MODEL_PATH = (
    PROJECT_DIR / "models" / "baseline_mobilenetv2_best.keras"
)

DRIVE_PROCESSED = PROJECT_DIR / "processed_224"
LOCAL_PROCESSED = Path("/content/BACH_processed_224")

CLASSES = ["Benign", "InSitu", "Invasive", "Normal"]
BATCH_SIZE = 16

# Restore temporary data copy if the runtime was reset
if not LOCAL_PROCESSED.exists():
    shutil.copytree(DRIVE_PROCESSED, LOCAL_PROCESSED)

if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Saved best model not found: {BEST_MODEL_PATH}"
    )

# -----------------------------------------
# 2. Rebuild validation dataset in fixed order
# -----------------------------------------

val_ds = tf.keras.utils.image_dataset_from_directory(
    str(LOCAL_PROCESSED / "validation"),
    labels="inferred",
    label_mode="int",
    class_names=CLASSES,
    color_mode="rgb",
    image_size=(224, 224),
    batch_size=BATCH_SIZE,
    shuffle=False
).prefetch(tf.data.AUTOTUNE)

# -----------------------------------------
# 3. Load saved best model
# -----------------------------------------

model = tf.keras.models.load_model(
    BEST_MODEL_PATH,
    compile=False
)

print("Loaded:", BEST_MODEL_PATH.name)

# -----------------------------------------
# 4. Predict validation images
# -----------------------------------------

y_true = np.concatenate([
    labels.numpy() for _, labels in val_ds
])

probabilities = model.predict(val_ds, verbose=1)
y_pred = np.argmax(probabilities, axis=1)

print(f"\nValidation images evaluated: {len(y_true)}")
print(f"Validation accuracy: {accuracy_score(y_true, y_pred):.4f}")

# -----------------------------------------
# 5. Classification report
# -----------------------------------------

report = classification_report(
    y_true,
    y_pred,
    labels=[0, 1, 2, 3],
    target_names=CLASSES,
    digits=4,
    zero_division=0
)

print("\nCLASSIFICATION REPORT")
print(report)

# -----------------------------------------
# 6. Confusion matrix
# -----------------------------------------

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1, 2, 3]
)

print("Confusion matrix:")
print(cm)

fig, ax = plt.subplots(figsize=(8, 7))

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=CLASSES
)

display.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=False
)

ax.set_title("Baseline MobileNetV2 — Validation Set")
plt.tight_layout()

FIGURE_DIR = PROJECT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plot_path = FIGURE_DIR / "baseline_validation_confusion_matrix.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()

# -----------------------------------------
# 7. Save results permanently to Drive
# -----------------------------------------

pd.DataFrame(
    cm,
    index=CLASSES,
    columns=CLASSES
).to_csv(PROJECT_DIR / "baseline_validation_confusion_matrix.csv")

pd.DataFrame(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3],
        target_names=CLASSES,
        output_dict=True,
        zero_division=0
    )
).T.to_csv(PROJECT_DIR / "baseline_validation_report.csv")

print("\nResults saved in:", PROJECT_DIR)

In [ ]:
import shutil
from pathlib import Path
import tensorflow as tf

# ==========================================
# 1. PATHS AND CONFIGURATION
# ==========================================

tf.keras.utils.set_random_seed(42)

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")
DRIVE_PROCESSED = PROJECT_DIR / "processed_224"
LOCAL_PROCESSED = Path("/content/BACH_processed_224")

BASELINE_PATH = PROJECT_DIR / "models" / "baseline_mobilenetv2_best.keras"

MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODEL_DIR / "mobilenetv2_finetune_best.keras"
LATEST_MODEL_PATH = MODEL_DIR / "mobilenetv2_finetune_last.keras"
LOG_PATH = PROJECT_DIR / "finetune_training_log.csv"

CLASSES = ["Benign", "InSitu", "Invasive", "Normal"]
BATCH_SIZE = 16

if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError("No GPU detected. Stop before training.")

if not BASELINE_PATH.exists():
    raise FileNotFoundError(f"Baseline model missing: {BASELINE_PATH}")

# Re-create temporary copy if runtime storage was reset
if not LOCAL_PROCESSED.exists():
    print("Restoring processed images from Google Drive...")
    shutil.copytree(DRIVE_PROCESSED, LOCAL_PROCESSED)

print("GPU available.")
print("Baseline model found.")
print("Temporary dataset:", LOCAL_PROCESSED)

# ==========================================
# 2. REBUILD DATA PIPELINE
# ==========================================

def load_split(split, shuffle=False):
    return tf.keras.utils.image_dataset_from_directory(
        str(LOCAL_PROCESSED / split),
        labels="inferred",
        label_mode="int",
        class_names=CLASSES,
        color_mode="rgb",
        image_size=(224, 224),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=42
    ).prefetch(tf.data.AUTOTUNE)

train_ds = load_split("train", shuffle=True)
val_ds = load_split("validation", shuffle=False)

# ==========================================
# 3. LOAD THE BEST BASELINE
# ==========================================

model = tf.keras.models.load_model(
    BASELINE_PATH,
    compile=False
)

# Find the pretrained MobileNetV2 feature extractor
base_model = next(
    layer for layer in model.layers
    if layer.name.startswith("mobilenetv2")
)

# ==========================================
# 4. UNFREEZE ONLY THE LAST 30 LAYERS
# ==========================================

base_model.trainable = True

fine_tune_at = max(0, len(base_model.layers) - 30)

for index, layer in enumerate(base_model.layers):
    if index < fine_tune_at:
        layer.trainable = False
    elif isinstance(layer, tf.keras.layers.BatchNormalization):
        # Keep Batch Normalization statistics frozen
        layer.trainable = False
    else:
        layer.trainable = True

print("\nTotal MobileNetV2 layers:", len(base_model.layers))
print("Fine-tuning starts at layer:", fine_tune_at)
print(
    "Trainable MobileNetV2 parameters:",
    sum(
        int(tf.size(variable))
        for variable in base_model.trainable_variables
    )
)

# ==========================================
# 5. COMPILE WITH A SMALL LEARNING RATE
# ==========================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# ==========================================
# 6. SAVE PROGRESS TO GOOGLE DRIVE
# ==========================================

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(LATEST_MODEL_PATH),
        save_best_only=False,
        verbose=0
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        str(LOG_PATH),
        append=False
    ),
    tf.keras.callbacks.TerminateOnNaN()
]

# ==========================================
# 7. TRAIN
# ==========================================

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=callbacks
)

print("\n========== FINE-TUNING COMPLETE ==========")
print("Best model saved:", BEST_MODEL_PATH.exists())
print("Latest model saved:", LATEST_MODEL_PATH.exists())
print("Training log saved:", LOG_PATH.exists())

print("\nBest validation accuracy in this run:",
      f"{max(history_ft.history['val_accuracy']):.4f}")

print("Best validation loss in this run:",
      f"{min(history_ft.history['val_loss']):.4f}")

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")
LOCAL_PROCESSED = Path("/content/BACH_processed_224")

CLASSES = ["Benign", "InSitu", "Invasive", "Normal"]
BATCH_SIZE = 16

# Restore temporary data if Colab restarted
if not LOCAL_PROCESSED.exists():
    import shutil
    shutil.copytree(
        PROJECT_DIR / "processed_224",
        LOCAL_PROCESSED
    )

# Fixed validation dataset; no shuffling
val_ds = tf.keras.utils.image_dataset_from_directory(
    str(LOCAL_PROCESSED / "validation"),
    labels="inferred",
    label_mode="int",
    class_names=CLASSES,
    image_size=(224, 224),
    batch_size=BATCH_SIZE,
    shuffle=False
).prefetch(tf.data.AUTOTUNE)

# Ground-truth validation labels
y_true = np.concatenate(
    [labels.numpy() for _, labels in val_ds]
)

MODEL_PATHS = {
    "Baseline": PROJECT_DIR / "models" /
                "baseline_mobilenetv2_best.keras",
    "Fine-tuned": PROJECT_DIR / "models" /
                  "mobilenetv2_finetune_best.keras"
}

results = {}
reports = {}

for name, model_path in MODEL_PATHS.items():
    if not model_path.exists():
        raise FileNotFoundError(f"Missing model: {model_path}")

    model = tf.keras.models.load_model(
        model_path,
        compile=False
    )

    probabilities = model.predict(val_ds, verbose=0)
    y_pred = np.argmax(probabilities, axis=1)

    accuracy = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(
        y_true, y_pred, labels=range(len(CLASSES))
    )

    results[name] = {
        "accuracy": accuracy,
        "confusion_matrix": cm
    }

    reports[name] = classification_report(
        y_true,
        y_pred,
        labels=range(len(CLASSES)),
        target_names=CLASSES,
        output_dict=True,
        zero_division=0
    )

    print(f"\n===== {name} =====")
    print(f"Validation accuracy: {accuracy:.4f}")
    print(classification_report(
        y_true,
        y_pred,
        labels=range(len(CLASSES)),
        target_names=CLASSES,
        digits=3,
        zero_division=0
    ))
    print("Confusion matrix:\n", cm)

# Comparison plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, name in zip(axes, MODEL_PATHS):
    disp = ConfusionMatrixDisplay(
        confusion_matrix=results[name]["confusion_matrix"],
        display_labels=CLASSES
    )
    disp.plot(
        ax=ax,
        cmap="Blues",
        values_format="d",
        colorbar=False
    )
    ax.set_title(f"{name} — Validation")

plt.tight_layout()

FIGURE_DIR = PROJECT_DIR / "figures"
FIGURE_DIR.mkdir(exist_ok=True)

plt.savefig(
    FIGURE_DIR / "baseline_vs_finetuned_validation.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()

# Save comparison report to Drive
comparison = pd.DataFrame({
    name: {
        "validation_accuracy": results[name]["accuracy"],
        **{
            f"{class_name}_F1":
                reports[name][class_name]["f1-score"]
            for class_name in CLASSES
        },
        "macro_F1": reports[name]["macro avg"]["f1-score"]
    }
    for name in MODEL_PATHS
}).T

comparison.to_csv(
    PROJECT_DIR / "baseline_vs_finetuned_comparison.csv"
)

print("\n===== MODEL COMPARISON =====")
print(comparison.round(4))
print("\nComparison saved in Google Drive.")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import shutil
import tensorflow as tf

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")
DRIVE_PROCESSED = PROJECT_DIR / "processed_224"
LOCAL_PROCESSED = Path("/content/BACH_processed_224")
MANIFEST_PATH = PROJECT_DIR / "split_manifest.csv"

# Verify persistent files
required = [
    PROJECT_DIR / "dataset",
    DRIVE_PROCESSED,
    MANIFEST_PATH,
    PROJECT_DIR / "models" / "baseline_mobilenetv2_best.keras",
    PROJECT_DIR / "models" / "mobilenetv2_finetune_best.keras",
]

for path in required:
    if not path.exists():
        raise FileNotFoundError(f"Required project file missing: {path}")
    print("FOUND:", path.name)

# Verify GPU
print("\nTensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError("GPU unavailable. Stop before training.")

# Restore the temporary dataset copy only if necessary
if not LOCAL_PROCESSED.exists():
    print("\nCopying processed images from Drive...")
    shutil.copytree(DRIVE_PROCESSED, LOCAL_PROCESSED)

# Verify actual counts
classes = ["Benign", "InSitu", "Invasive", "Normal"]
expected = {"train": 70, "validation": 15, "internal_test": 15}

for split, per_class in expected.items():
    total = 0
    for class_name in classes:
        folder = LOCAL_PROCESSED / split / class_name
        count = len(list(folder.glob("*.jpg")))
        print(f"{split:14s} {class_name:10s}: {count}")
        if count != per_class:
            raise RuntimeError(
                f"Unexpected count in {split}/{class_name}: {count}"
            )
        total += count
    print(f"{split:14s} TOTAL: {total}\n")

print("✅ Environment and dataset verified.")

In [ ]:
import tensorflow as tf
from pathlib import Path

tf.keras.utils.set_random_seed(42)

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")
LOCAL_PROCESSED = Path("/content/BACH_processed_224")
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODEL_DIR / "densenet121_best.keras"
LOG_PATH = PROJECT_DIR / "densenet121_training_log.csv"

CLASSES = ["Benign", "InSitu", "Invasive", "Normal"]
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

# Load the same fixed training and validation splits
def load_split(split, shuffle=False):
    return tf.keras.utils.image_dataset_from_directory(
        str(LOCAL_PROCESSED / split),
        labels="inferred",
        label_mode="int",
        class_names=CLASSES,
        color_mode="rgb",
        image_size=(224, 224),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=42
    ).prefetch(AUTOTUNE)

train_ds = load_split("train", shuffle=True)
val_ds = load_split("validation", shuffle=False)

# Data augmentation: training images only
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.10)
], name="data_augmentation")

# Pretrained DenseNet121 feature extractor
base_model = tf.keras.applications.DenseNet121(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)
base_model.trainable = False

# Build the four-class classifier
inputs = tf.keras.Input(shape=(224, 224, 3))

x = augmentation(inputs)
x = tf.keras.applications.densenet.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.40)(x)
outputs = tf.keras.layers.Dense(
    4, activation="softmax", name="class_predictions"
)(x)

model = tf.keras.Model(inputs, outputs, name="BACH_DenseNet121")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Trainable parameters:",
      sum(int(tf.size(v)) for v in model.trainable_variables))
model.summary()

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        str(BEST_MODEL_PATH),
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(str(LOG_PATH)),
    tf.keras.callbacks.TerminateOnNaN()
]

history_dn = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks
)

# Save selected weights/model separately in Drive
model.save(MODEL_DIR / "densenet121_selected.keras")

print("\n========== DENSENET121 HEAD TRAINING COMPLETE ==========")
print("Best checkpoint exists:", BEST_MODEL_PATH.exists())
print("Training log exists:", LOG_PATH.exists())
print("Best validation accuracy observed:",
      round(max(history_dn.history["val_accuracy"]), 4))
print("Best validation loss observed:",
      round(min(history_dn.history["val_loss"]), 4))

In [ ]:

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")
LOCAL_PROCESSED = Path("/content/BACH_processed_224")

CLASSES = ["Benign", "InSitu", "Invasive", "Normal"]

# Restore temporary dataset if necessary
if not LOCAL_PROCESSED.exists():
    import shutil
    shutil.copytree(
        PROJECT_DIR / "processed_224",
        LOCAL_PROCESSED
    )

# Fixed validation dataset
val_ds = tf.keras.utils.image_dataset_from_directory(
    str(LOCAL_PROCESSED / "validation"),
    labels="inferred",
    label_mode="int",
    class_names=CLASSES,
    image_size=(224, 224),
    batch_size=16,
    shuffle=False
)

y_true = np.concatenate(
    [labels.numpy() for _, labels in val_ds]
)

models = {
    "MobileNetV2 baseline":
        PROJECT_DIR / "models/baseline_mobilenetv2_best.keras",

    "MobileNetV2 fine-tuned":
        PROJECT_DIR / "models/mobilenetv2_finetune_best.keras",

    "DenseNet121 head-only":
        PROJECT_DIR / "models/densenet121_best.keras"
}

results = {}
reports = {}

for name, model_path in models.items():

    if not model_path.exists():
        raise FileNotFoundError(model_path)

    print(f"\nEvaluating {name}...")

    model = tf.keras.models.load_model(
        model_path,
        compile=False
    )

    predictions = model.predict(val_ds, verbose=0)
    y_pred = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(y_true, y_pred)

    cm = confusion_matrix(
        y_true, y_pred,
        labels=range(4)
    )

    report = classification_report(
        y_true,
        y_pred,
        target_names=CLASSES,
        output_dict=True,
        zero_division=0
    )

    results[name] = {
        "accuracy": accuracy,
        "macro_f1": report["macro avg"]["f1-score"],
        "cm": cm
    }

    reports[name] = report

    print(f"Validation accuracy: {accuracy:.4f}")
    print(f"Macro F1: {report['macro avg']['f1-score']:.4f}")
    print(classification_report(
        y_true,
        y_pred,
        target_names=CLASSES,
        digits=3,
        zero_division=0
    ))

    del model, predictions, y_pred

# Summary table
summary = pd.DataFrame({
    name: {
        "Accuracy": result["accuracy"],
        "Macro F1": result["macro_f1"]
    }
    for name, result in results.items()
}).T

print("\n========== MODEL COMPARISON ==========")
print(summary.round(4))

summary.to_csv(
    PROJECT_DIR / "model_comparison_validation.csv"
)

# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, result) in zip(axes, results.items()):

    ax.imshow(result["cm"], cmap="Blues")

    ax.set_title(name)
    ax.set_xticks(range(4), CLASSES, rotation=45)
    ax.set_yticks(range(4), CLASSES)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

    for i in range(4):
        for j in range(4):
            ax.text(
                j, i, result["cm"][i, j],
                ha="center", va="center"
            )

plt.tight_layout()

FIGURE_DIR = PROJECT_DIR / "figures"
FIGURE_DIR.mkdir(exist_ok=True)

plt.savefig(
    FIGURE_DIR / "three_model_comparison.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print("\nComparison saved to Google Drive.")


In [ ]:

import tensorflow as tf
import pandas as pd
from pathlib import Path

tf.keras.utils.set_random_seed(42)

# ==========================================
# 1. CONFIGURATION
# ==========================================

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")
LOCAL_PROCESSED = Path("/content/BACH_processed_224")

MODEL_DIR = PROJECT_DIR / "models"

BEST_LOSS_PATH = MODEL_DIR / "densenet121_finetune_best_loss.keras"
BEST_ACC_PATH = MODEL_DIR / "densenet121_finetune_best_accuracy.keras"
LATEST_PATH = MODEL_DIR / "densenet121_finetune_last.keras"

LOG_PATH = PROJECT_DIR / "densenet121_finetune_log.csv"

CLASSES = ["Benign", "InSitu", "Invasive", "Normal"]
BATCH_SIZE = 8

MODEL_DIR.mkdir(parents=True, exist_ok=True)

if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError("GPU unavailable.")

# Restore temporary dataset if needed
if not LOCAL_PROCESSED.exists():
    import shutil
    shutil.copytree(
        PROJECT_DIR / "processed_224",
        LOCAL_PROCESSED
    )

# ==========================================
# 2. LOAD DATA
# ==========================================

def load_split(split, shuffle=False):
    return tf.keras.utils.image_dataset_from_directory(
        str(LOCAL_PROCESSED / split),
        labels="inferred",
        label_mode="int",
        class_names=CLASSES,
        image_size=(224, 224),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=42
    ).prefetch(tf.data.AUTOTUNE)

train_ds = load_split("train", shuffle=True)
val_ds = load_split("validation", shuffle=False)

# ==========================================
# 3. LOAD BEST DENSENET HEAD-ONLY MODEL
# ==========================================

HEAD_ONLY_PATH = MODEL_DIR / "densenet121_best.keras"

model = tf.keras.models.load_model(
    HEAD_ONLY_PATH,
    compile=False
)

base_model = next(
    layer for layer in model.layers
    if layer.name == "densenet121"
)

# ==========================================
# 4. UNFREEZE LAST 60 DENSENET LAYERS
# ==========================================

base_model.trainable = True

fine_tune_at = max(0, len(base_model.layers) - 60)

for index, layer in enumerate(base_model.layers):

    if index < fine_tune_at:
        layer.trainable = False

    elif isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

    else:
        layer.trainable = True

print("Total DenseNet layers:", len(base_model.layers))
print("Fine-tuning starts at:", fine_tune_at)

trainable_params = sum(
    int(tf.size(variable))
    for variable in model.trainable_variables
)

print("Trainable parameters:", trainable_params)

# ==========================================
# 5. COMPILE
# ==========================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# ==========================================
# 6. CHECKPOINTS AND CALLBACKS
# ==========================================

callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
        str(BEST_LOSS_PATH),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        str(BEST_ACC_PATH),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        str(LATEST_PATH),
        save_best_only=False,
        verbose=0
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),

    tf.keras.callbacks.CSVLogger(
        str(LOG_PATH),
        append=False
    ),

    tf.keras.callbacks.TerminateOnNaN()
]

# ==========================================
# 7. TRAIN
# ==========================================

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=callbacks
)

# ==========================================
# 8. REPORT
# ==========================================

print("\n========== DENSENET FINE-TUNING COMPLETE ==========")

print(
    "Highest observed validation accuracy:",
    round(max(history.history["val_accuracy"]), 4)
)

print(
    "Lowest observed validation loss:",
    round(min(history.history["val_loss"]), 4)
)

print("\nSaved checkpoints:")
print("Best loss:", BEST_LOSS_PATH.exists())
print("Best accuracy:", BEST_ACC_PATH.exists())
print("Latest:", LATEST_PATH.exists())
print("Training log:", LOG_PATH.exists())


In [ ]:

import tensorflow as tf
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")
LOCAL_PROCESSED = Path("/content/BACH_processed_224")

CLASSES = ["Benign", "InSitu", "Invasive", "Normal"]

# Recreate temporary dataset if necessary
if not LOCAL_PROCESSED.exists():
    import shutil
    shutil.copytree(
        PROJECT_DIR / "processed_224",
        LOCAL_PROCESSED
    )

# Fixed validation dataset
val_ds = tf.keras.utils.image_dataset_from_directory(
    str(LOCAL_PROCESSED / "validation"),
    labels="inferred",
    label_mode="int",
    class_names=CLASSES,
    image_size=(224, 224),
    batch_size=16,
    shuffle=False
)

y_true = np.concatenate([
    labels.numpy() for _, labels in val_ds
])

checkpoints = {
    "MobileNet baseline":
        PROJECT_DIR / "models/baseline_mobilenetv2_best.keras",

    "MobileNet fine-tuned":
        PROJECT_DIR / "models/mobilenetv2_finetune_best.keras",

    "DenseNet head-only":
        PROJECT_DIR / "models/densenet121_best.keras",

    "DenseNet fine-tuned - best accuracy":
        PROJECT_DIR / "models/densenet121_finetune_best_accuracy.keras",

    "DenseNet fine-tuned - best loss":
        PROJECT_DIR / "models/densenet121_finetune_best_loss.keras"
}

results = []

for name, path in checkpoints.items():

    if not path.exists():
        raise FileNotFoundError(path)

    model = tf.keras.models.load_model(path, compile=False)

    probabilities = model.predict(val_ds, verbose=0)
    predictions = np.argmax(probabilities, axis=1)

    accuracy = accuracy_score(y_true, predictions)

    report = classification_report(
        y_true,
        predictions,
        target_names=CLASSES,
        output_dict=True,
        zero_division=0
    )

    macro_f1 = report["macro avg"]["f1-score"]

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Macro_F1": macro_f1,
        "Correct": int((y_true == predictions).sum())
    })

    print(f"\n===== {name} =====")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")
    print(classification_report(
        y_true,
        predictions,
        target_names=CLASSES,
        digits=3,
        zero_division=0
    ))
    print("Confusion matrix:")
    print(confusion_matrix(y_true, predictions))

    del model, probabilities, predictions

comparison = pd.DataFrame(results).sort_values(
    ["Accuracy", "Macro_F1"],
    ascending=False
)

print("\n========== FINAL CHECKPOINT COMPARISON ==========")
print(comparison.to_string(index=False))

comparison.to_csv(
    PROJECT_DIR / "all_model_checkpoints_comparison.csv",
    index=False
)

print("\nSaved comparison to Google Drive.")


In [ ]:
import pandas as pd
from pathlib import Path

comparison_path = (
    Path("/content/drive/MyDrive/BACH_Project")
    / "all_model_checkpoints_comparison.csv"
)

if comparison_path.exists():
    comparison = pd.read_csv(comparison_path)
    print(comparison.to_string(index=False))
else:
    print("Comparison CSV not found. The previous cell may not have completed.")

In [ ]:

import tensorflow as tf
from pathlib import Path

tf.keras.utils.set_random_seed(42)

# ==========================================
# 1. CONFIGURATION
# ==========================================

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")
LOCAL_PROCESSED = Path("/content/BACH_processed_224")

MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BEST_LOSS_PATH = MODEL_DIR / "efficientnetb0_best_loss.keras"
BEST_ACC_PATH = MODEL_DIR / "efficientnetb0_best_accuracy.keras"
LATEST_PATH = MODEL_DIR / "efficientnetb0_last.keras"

LOG_PATH = PROJECT_DIR / "efficientnetb0_training_log.csv"

CLASSES = ["Benign", "InSitu", "Invasive", "Normal"]
BATCH_SIZE = 16

if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError("GPU unavailable!")

if not LOCAL_PROCESSED.exists():
    import shutil
    shutil.copytree(
        PROJECT_DIR / "processed_224",
        LOCAL_PROCESSED
    )

print("GPU ready.")
print("Dataset ready.")

# ==========================================
# 2. LOAD DATASETS
# ==========================================

def load_split(split, shuffle=False):
    return tf.keras.utils.image_dataset_from_directory(
        str(LOCAL_PROCESSED / split),
        labels="inferred",
        label_mode="int",
        class_names=CLASSES,
        image_size=(224, 224),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=42
    ).prefetch(tf.data.AUTOTUNE)

train_ds = load_split("train", shuffle=True)
val_ds = load_split("validation", shuffle=False)

# ==========================================
# 3. DATA AUGMENTATION
# ==========================================

augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.10)
], name="data_augmentation")

# ==========================================
# 4. LOAD PRETRAINED EFFICIENTNETB0
# ==========================================

base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base_model.trainable = False

# ==========================================
# 5. BUILD CLASSIFICATION MODEL
# ==========================================

inputs = tf.keras.Input(shape=(224, 224, 3))

x = augmentation(inputs)

# EfficientNetB0 performs its own input rescaling.
# Keep incoming pixels in the 0-255 range.
x = base_model(x, training=False)

x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.40)(x)

outputs = tf.keras.layers.Dense(
    4,
    activation="softmax",
    name="class_predictions"
)(x)

model = tf.keras.Model(
    inputs,
    outputs,
    name="BACH_EfficientNetB0"
)

# ==========================================
# 6. COMPILE
# ==========================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("\nModel summary:")
model.summary()

# ==========================================
# 7. CHECKPOINTS
# ==========================================

callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
        str(BEST_LOSS_PATH),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        str(BEST_ACC_PATH),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        str(LATEST_PATH),
        save_best_only=False,
        verbose=0
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),

    tf.keras.callbacks.CSVLogger(
        str(LOG_PATH)
    ),

    tf.keras.callbacks.TerminateOnNaN()
]

# ==========================================
# 8. TRAIN
# ==========================================

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks
)

# ==========================================
# 9. RESULTS
# ==========================================

print("\n========== EFFICIENTNETB0 COMPLETE ==========")

print(
    "Highest observed validation accuracy:",
    round(max(history.history["val_accuracy"]), 4)
)

print(
    "Lowest validation loss:",
    round(min(history.history["val_loss"]), 4)
)

print("\nSaved checkpoints:")
print("Best accuracy:", BEST_ACC_PATH.exists())
print("Best loss:", BEST_LOSS_PATH.exists())
print("Latest:", LATEST_PATH.exists())
print("Training log:", LOG_PATH.exists())


In [ ]:

import tensorflow as tf
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")
LOCAL_PROCESSED = Path("/content/BACH_processed_224")

CLASSES = ["Benign", "InSitu", "Invasive", "Normal"]

# Recreate temporary dataset if necessary
if not LOCAL_PROCESSED.exists():
    import shutil
    shutil.copytree(
        PROJECT_DIR / "processed_224",
        LOCAL_PROCESSED
    )

# Load the same validation data for all models
val_ds = tf.keras.utils.image_dataset_from_directory(
    str(LOCAL_PROCESSED / "validation"),
    labels="inferred",
    label_mode="int",
    class_names=CLASSES,
    image_size=(224, 224),
    batch_size=16,
    shuffle=False
)

y_true = np.concatenate([
    labels.numpy() for _, labels in val_ds
])

checkpoints = {
    "MobileNet baseline":
        "baseline_mobilenetv2_best.keras",

    "MobileNet fine-tuned":
        "mobilenetv2_finetune_best.keras",

    "DenseNet head-only":
        "densenet121_best.keras",

    "DenseNet fine-tuned - best accuracy":
        "densenet121_finetune_best_accuracy.keras",

    "DenseNet fine-tuned - best loss":
        "densenet121_finetune_best_loss.keras",

    "EfficientNet - best accuracy":
        "efficientnetb0_best_accuracy.keras",

    "EfficientNet - best loss":
        "efficientnetb0_best_loss.keras"
}

results = []
confusion_matrices = {}

for name, filename in checkpoints.items():

    model_path = PROJECT_DIR / "models" / filename

    if not model_path.exists():
        raise FileNotFoundError(model_path)

    print(f"\nEvaluating: {name}")

    model = tf.keras.models.load_model(
        model_path,
        compile=False
    )

    probabilities = model.predict(
        val_ds,
        verbose=0
    )

    predictions = np.argmax(probabilities, axis=1)

    report = classification_report(
        y_true,
        predictions,
        labels=[0, 1, 2, 3],
        target_names=CLASSES,
        output_dict=True,
        zero_division=0
    )

    accuracy = accuracy_score(y_true, predictions)
    macro_f1 = report["macro avg"]["f1-score"]

    cm = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1, 2, 3]
    )

    confusion_matrices[name] = cm

    row = {
        "Model": name,
        "Correct": int(np.sum(y_true == predictions)),
        "Accuracy": accuracy,
        "Macro_F1": macro_f1
    }

    for class_name in CLASSES:
        row[f"{class_name}_F1"] = report[class_name]["f1-score"]
        row[f"{class_name}_Recall"] = report[class_name]["recall"]

    results.append(row)

    print("Accuracy:", round(accuracy, 4))
    print("Macro F1:", round(macro_f1, 4))
    print("Confusion matrix:\n", cm)

    del model, probabilities, predictions

# Build leaderboard
leaderboard = pd.DataFrame(results)

leaderboard = leaderboard.sort_values(
    ["Accuracy", "Macro_F1"],
    ascending=False
).reset_index(drop=True)

print("\n========== COMPLETE MODEL LEADERBOARD ==========")
print(leaderboard.round(4).to_string(index=False))

# Save results in Drive
output_path = PROJECT_DIR / "complete_model_leaderboard.csv"

leaderboard.to_csv(output_path, index=False)

print("\nLeaderboard saved:", output_path)

# Save all confusion matrices
cm_path = PROJECT_DIR / "complete_model_confusion_matrices.csv"

cm_frames = []

for name, cm in confusion_matrices.items():
    frame = pd.DataFrame(
        cm,
        index=CLASSES,
        columns=CLASSES
    )
    frame.insert(0, "Actual class", frame.index)
    frame.insert(0, "Model", name)
    cm_frames.append(frame.reset_index(drop=True))

pd.concat(cm_frames, ignore_index=True).to_csv(
    cm_path,
    index=False
)

print("Confusion matrices saved:", cm_path)


In [ ]:

import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image
import math

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")

CLASSES = ["Benign", "InSitu", "Invasive", "Normal"]

MODEL_PATH = (
    PROJECT_DIR / "models" /
    "efficientnetb0_best_accuracy.keras"
)

LOCAL_PROCESSED = Path("/content/BACH_processed_224")

# Restore temporary data if necessary
if not LOCAL_PROCESSED.exists():
    import shutil
    shutil.copytree(
        PROJECT_DIR / "processed_224",
        LOCAL_PROCESSED
    )

# Load validation data without shuffling
val_ds = tf.keras.utils.image_dataset_from_directory(
    str(LOCAL_PROCESSED / "validation"),
    labels="inferred",
    label_mode="int",
    class_names=CLASSES,
    image_size=(224, 224),
    batch_size=16,
    shuffle=False
)

# Reconstruct the corresponding file order
image_paths = []

for class_name in CLASSES:
    image_paths.extend(sorted(
        (LOCAL_PROCESSED / "validation" / class_name).glob("*.jpg")
    ))

y_true = np.concatenate([
    labels.numpy() for _, labels in val_ds
])

assert len(image_paths) == len(y_true) == 60

# Load best EfficientNet checkpoint
model = tf.keras.models.load_model(
    MODEL_PATH,
    compile=False
)

# Predict
probabilities = model.predict(val_ds, verbose=1)
y_pred = np.argmax(probabilities, axis=1)

# Build detailed prediction table
records = []

for i, processed_path in enumerate(image_paths):

    actual = CLASSES[int(y_true[i])]
    predicted = CLASSES[int(y_pred[i])]

    original_path = (
        PROJECT_DIR / "dataset" /
        actual / f"{processed_path.stem}.tif"
    )

    records.append({
        "image": processed_path.stem,
        "actual": actual,
        "predicted": predicted,
        "correct": actual == predicted,
        "predicted_confidence": float(probabilities[i, y_pred[i]]),
        "actual_class_probability": float(probabilities[i, y_true[i]]),
        "original_path": str(original_path)
    })

pred_df = pd.DataFrame(records)

# Save all validation predictions
pred_df.to_csv(
    PROJECT_DIR / "efficientnet_validation_predictions.csv",
    index=False
)

# Extract incorrectly classified Benign images
benign_errors = pred_df[
    (pred_df["actual"] == "Benign") &
    (~pred_df["correct"])
].copy()

benign_errors.to_csv(
    PROJECT_DIR / "benign_misclassifications.csv",
    index=False
)

print("\n========== BENIGN ERROR ANALYSIS ==========")
print("Benign validation images:", 15)
print("Correctly classified:",
      int((pred_df.query("actual == 'Benign'")["correct"]).sum()))
print("Misclassified:", len(benign_errors))

print("\nMisclassified Benign images:")
print(
    benign_errors[
        ["image", "actual", "predicted",
         "predicted_confidence", "actual_class_probability"]
    ].to_string(index=False)
)

# Visualize original TIFFs
if len(benign_errors) > 0:

    ncols = 3
    nrows = math.ceil(len(benign_errors) / ncols)

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(15, 5 * nrows)
    )

    axes = np.array(axes).reshape(-1)

    for ax, (_, row) in zip(axes, benign_errors.iterrows()):

        image_path = Path(row["original_path"])

        if not image_path.exists():
            ax.set_title("Original image missing")
            ax.axis("off")
            continue

        with Image.open(image_path) as img:
            img = img.convert("RGB")
            img.thumbnail((450, 350))
            ax.imshow(img)

        ax.set_title(
            f"{row['image']}\n"
            f"Predicted: {row['predicted']}\n"
            f"Confidence: {row['predicted_confidence']:.1%}"
        )
        ax.axis("off")

    for ax in axes[len(benign_errors):]:
        ax.axis("off")

    plt.suptitle(
        "EfficientNetB0 — Misclassified Benign Images",
        fontsize=16
    )

    plt.tight_layout()

    FIGURE_DIR = PROJECT_DIR / "figures"
    FIGURE_DIR.mkdir(exist_ok=True)

    plt.savefig(
        FIGURE_DIR / "efficientnet_benign_errors.png",
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()

print("\nResults saved in Google Drive.")


In [ ]:

import tensorflow as tf
from pathlib import Path
import pandas as pd

tf.keras.utils.set_random_seed(42)

# ==========================================
# 1. CONFIGURATION
# ==========================================

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")
LOCAL_PROCESSED = Path("/content/BACH_processed_224")
MODEL_DIR = PROJECT_DIR / "models"

SOURCE_MODEL = MODEL_DIR / "efficientnetb0_best_accuracy.keras"

BEST_LOSS_PATH = MODEL_DIR / "efficientnetb0_finetune_best_loss.keras"
BEST_ACC_PATH = MODEL_DIR / "efficientnetb0_finetune_best_accuracy.keras"
LATEST_PATH = MODEL_DIR / "efficientnetb0_finetune_last.keras"

LOG_PATH = PROJECT_DIR / "efficientnetb0_finetune_log.csv"

CLASSES = ["Benign", "InSitu", "Invasive", "Normal"]

MODEL_DIR.mkdir(parents=True, exist_ok=True)

if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError("GPU unavailable.")

if not SOURCE_MODEL.exists():
    raise FileNotFoundError(SOURCE_MODEL)

# Restore temporary data if necessary
if not LOCAL_PROCESSED.exists():
    import shutil
    shutil.copytree(
        PROJECT_DIR / "processed_224",
        LOCAL_PROCESSED
    )

# ==========================================
# 2. LOAD DATASETS
# ==========================================

def load_split(split, shuffle=False):
    return tf.keras.utils.image_dataset_from_directory(
        str(LOCAL_PROCESSED / split),
        labels="inferred",
        label_mode="int",
        class_names=CLASSES,
        image_size=(224, 224),
        batch_size=8,
        shuffle=shuffle,
        seed=42
    ).prefetch(tf.data.AUTOTUNE)

train_ds = load_split("train", shuffle=True)
val_ds = load_split("validation", shuffle=False)

# ==========================================
# 3. LOAD BEST EFFICIENTNET CHECKPOINT
# ==========================================

model = tf.keras.models.load_model(
    SOURCE_MODEL,
    compile=False
)

base_model = model.get_layer("efficientnetb0")

# ==========================================
# 4. UNFREEZE LAST 40 LAYERS
# ==========================================

base_model.trainable = True

fine_tune_at = max(0, len(base_model.layers) - 40)

for index, layer in enumerate(base_model.layers):

    if index < fine_tune_at:
        layer.trainable = False

    elif isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

    else:
        layer.trainable = True

trainable_params = sum(
    int(tf.size(variable))
    for variable in model.trainable_variables
)

print("Total EfficientNet layers:", len(base_model.layers))
print("Fine-tuning starts at:", fine_tune_at)
print("Trainable parameters:", trainable_params)

# ==========================================
# 5. COMPILE
# ==========================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# ==========================================
# 6. CHECKPOINTS AND CALLBACKS
# ==========================================

callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
        str(BEST_LOSS_PATH),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        str(BEST_ACC_PATH),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        str(LATEST_PATH),
        save_best_only=False,
        verbose=0
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),

    tf.keras.callbacks.CSVLogger(
        str(LOG_PATH)
    ),

    tf.keras.callbacks.TerminateOnNaN()
]

# ==========================================
# 7. TRAIN
# ==========================================

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=callbacks
)

# ==========================================
# 8. RESULTS
# ==========================================

print("\n========== EFFICIENTNET FINE-TUNING COMPLETE ==========")

print(
    "Highest observed validation accuracy:",
    round(max(history.history["val_accuracy"]), 4)
)

print(
    "Lowest observed validation loss:",
    round(min(history.history["val_loss"]), 4)
)

print("\nSaved checkpoints:")
print("Best accuracy:", BEST_ACC_PATH.exists())
print("Best loss:", BEST_LOSS_PATH.exists())
print("Latest:", LATEST_PATH.exists())
print("Training log:", LOG_PATH.exists())


In [ ]:

import tensorflow as tf
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")
LOCAL_PROCESSED = Path("/content/BACH_processed_224")

CLASSES = ["Benign", "InSitu", "Invasive", "Normal"]

val_ds = tf.keras.utils.image_dataset_from_directory(
    str(LOCAL_PROCESSED / "validation"),
    labels="inferred",
    label_mode="int",
    class_names=CLASSES,
    image_size=(224, 224),
    batch_size=16,
    shuffle=False
)

y_true = np.concatenate([
    y.numpy() for _, y in val_ds
])

checkpoints = {
    "EfficientNet original - best accuracy":
        "efficientnetb0_best_accuracy.keras",

    "EfficientNet original - best loss":
        "efficientnetb0_best_loss.keras",

    "EfficientNet fine-tuned - best accuracy":
        "efficientnetb0_finetune_best_accuracy.keras",

    "EfficientNet fine-tuned - best loss":
        "efficientnetb0_finetune_best_loss.keras"
}

results = []

for name, filename in checkpoints.items():

    path = PROJECT_DIR / "models" / filename

    if not path.exists():
        raise FileNotFoundError(path)

    model = tf.keras.models.load_model(path, compile=False)

    probabilities = model.predict(val_ds, verbose=0)
    predictions = np.argmax(probabilities, axis=1)

    accuracy = accuracy_score(y_true, predictions)
    macro_f1 = f1_score(
        y_true, predictions,
        average="macro",
        zero_division=0
    )

    results.append({
        "Checkpoint": name,
        "Correct": int(np.sum(y_true == predictions)),
        "Accuracy": accuracy,
        "Macro_F1": macro_f1
    })

    print(f"\n{name}")
    print("Accuracy:", round(accuracy, 4))
    print("Macro F1:", round(macro_f1, 4))

    del model, probabilities, predictions

comparison = pd.DataFrame(results).sort_values(
    ["Accuracy", "Macro_F1"],
    ascending=False
)

print("\n===== EFFICIENTNET CHECKPOINT COMPARISON =====")
print(comparison.to_string(index=False))

comparison.to_csv(
    PROJECT_DIR / "efficientnet_checkpoint_comparison.csv",
    index=False
)

print("\nSaved to Google Drive.")


In [ ]:

from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")

image_path = PROJECT_DIR / "dataset" / "Benign" / "b022.tif"

PATCH_SIZE = 224

with Image.open(image_path) as img:
    original = img.convert("RGB")
    width, height = original.size

print("Original dimensions:", original.size)

columns = width // PATCH_SIZE
rows = height // PATCH_SIZE

print("Patch columns:", columns)
print("Patch rows:", rows)
print("Total patches:", columns * rows)

# Show all non-overlapping patches
fig, axes = plt.subplots(
    rows, columns,
    figsize=(18, 13)
)

for row in range(rows):
    for col in range(columns):

        x1 = col * PATCH_SIZE
        y1 = row * PATCH_SIZE

        x2 = x1 + PATCH_SIZE
        y2 = y1 + PATCH_SIZE

        patch = original.crop((x1, y1, x2, y2))

        ax = axes[row, col]
        ax.imshow(patch)
        ax.set_title(f"R{row+1} C{col+1}", fontsize=8)
        ax.axis("off")

plt.suptitle(
    "BACH b022 — 54 Non-overlapping Tissue Patches",
    fontsize=15
)

plt.tight_layout()
plt.show()


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")

image_path = PROJECT_DIR / "dataset/Benign/b022.tif"

PATCH = 224
STRIDE = 192

with Image.open(image_path) as img:
    original = img.convert("RGB")
    width, height = original.size

def get_positions(length, patch, stride):
    positions = list(range(0, length - patch + 1, stride))

    last_position = length - patch

    if positions[-1] != last_position:
        positions.append(last_position)

    return positions

x_positions = get_positions(width, PATCH, STRIDE)
y_positions = get_positions(height, PATCH, STRIDE)

# Verify pixel coverage
coverage = np.zeros((height, width), dtype=np.uint8)

for y in y_positions:
    for x in x_positions:
        coverage[y:y+PATCH, x:x+PATCH] += 1

uncovered = np.count_nonzero(coverage == 0)
overlapping = np.count_nonzero(coverage > 1)

print("Original image:", width, "×", height)
print("Patch size:", PATCH)
print("Stride:", STRIDE)
print("Columns:", len(x_positions))
print("Rows:", len(y_positions))
print("Total patches:", len(x_positions) * len(y_positions))
print("Uncovered pixels:", uncovered)
print("Overlapping pixels:", overlapping)

# Visualize boundaries
fig, ax = plt.subplots(figsize=(15, 9))

ax.imshow(original)

for row, y in enumerate(y_positions):
    for col, x in enumerate(x_positions):
        rect = Rectangle(
            (x, y),
            PATCH,
            PATCH,
            linewidth=0.7,
            edgecolor="lime",
            facecolor="none"
        )
        ax.add_patch(rect)

ax.set_title("BACH b022 — Corrected 88-Patch Coverage")
ax.axis("off")

plt.tight_layout()
plt.show()

assert uncovered == 0, "Some original pixels are not covered!"
print("✅ Full image coverage verified.")

In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")

print("Searching project for metadata files...\n")

matches = []

for path in PROJECT_DIR.rglob("*"):
    if path.is_file():
        name = path.name.lower()

        if any(term in name for term in [
            "patient",
            "groundtruth",
            "metadata"
        ]):
            matches.append(path)

if matches:
    for path in matches:
        print(path)
else:
    print("No patient metadata files found.")

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")

METADATA_PATH = (
    PROJECT_DIR / "metadata" /
    "ICIAR2018_BACH_dataset_micro_patient.xlsx"
)

MANIFEST_PATH = PROJECT_DIR / "split_manifest.csv"

if not METADATA_PATH.exists():
    raise FileNotFoundError(
        f"Metadata file not found: {METADATA_PATH}"
    )

# Read original split
manifest = pd.read_csv(MANIFEST_PATH)

# Read the microscopy patient workbook
raw = pd.read_excel(METADATA_PATH, header=None)

# Four blocks: Histology, Label, Patient
column_starts = [0, 4, 8, 12]

mapping = []

for start in column_starts:

    for _, row in raw.iloc[1:].iterrows():

        filename = row.iloc[start]
        label = row.iloc[start + 1]
        patient = row.iloc[start + 2]

        if pd.isna(filename):
            continue

        if pd.isna(patient):
            patient_id = None
        else:
            patient_id = str(int(patient))

        mapping.append({
            "filename": str(filename).strip(),
            "metadata_label": str(label).strip(),
            "patient_id": patient_id
        })

mapping_df = pd.DataFrame(mapping)

# Check uniqueness
if mapping_df["filename"].duplicated().any():
    raise RuntimeError("Duplicate filenames found in metadata.")

manifest["filename"] = manifest["path"].apply(
    lambda x: Path(x).name
)

audit = manifest.merge(
    mapping_df,
    on="filename",
    how="left",
    validate="one_to_one"
)

# Check labels agree
label_mismatches = audit[
    audit["metadata_label"] != audit["label"]
]

print("========== METADATA AUDIT ==========")
print("Manifest images:", len(manifest))
print("Metadata entries:", len(mapping_df))
print("Matched images:", audit["metadata_label"].notna().sum())
print("Missing patient IDs:", audit["patient_id"].isna().sum())
print("Label mismatches:", len(label_mismatches))

# Patient metadata coverage by split
print("\nPatient mapping by split:")

for split in ["train", "validation", "internal_test"]:

    subset = audit[audit["split"] == split]

    print(
        split,
        "| total:", len(subset),
        "| mapped:", subset["patient_id"].notna().sum(),
        "| missing:", subset["patient_id"].isna().sum()
    )

# Detect patients appearing across multiple splits
known = audit.dropna(subset=["patient_id"])

patient_split_counts = (
    known.groupby("patient_id")["split"]
    .nunique()
)

leaked_patients = patient_split_counts[
    patient_split_counts > 1
]

print("\n========== PATIENT OVERLAP ==========")
print("Identified patients:", patient_split_counts.size)
print("Patients appearing in multiple splits:", len(leaked_patients))

if len(leaked_patients):
    print("\nPotential patient leakage:")
    for patient_id in leaked_patients.index:
        records = known[known["patient_id"] == patient_id]

        print(
            "Patient", patient_id,
            "| splits:", sorted(records["split"].unique()),
            "| images:", len(records)
        )
else:
    print("No overlap found among identified patients.")

# Save audit results
audit_path = PROJECT_DIR / "patient_split_audit.csv"
audit.to_csv(audit_path, index=False)

print("\nAudit saved:", audit_path)


In [ ]:
from pathlib import Path

METADATA_PATH = (
    Path("/content/drive/MyDrive/BACH_Project")
    / "metadata"
    / "ICIAR2018_BACH_dataset_micro_patient.xlsx"
)

print("Metadata file exists:", METADATA_PATH.exists())
print("File location:", METADATA_PATH)

In [ ]:

import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold

# ==========================================
# 1. PATHS
# ==========================================

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")

METADATA_PATH = (
    PROJECT_DIR / "metadata" /
    "ICIAR2018_BACH_dataset_micro_patient.xlsx"
)

OLD_MANIFEST_PATH = PROJECT_DIR / "split_manifest.csv"

NEW_MANIFEST_PATH = (
    PROJECT_DIR / "split_manifest_patient_grouped.csv"
)

# ==========================================
# 2. LOAD ORIGINAL MANIFEST AND METADATA
# ==========================================

manifest = pd.read_csv(OLD_MANIFEST_PATH)
raw = pd.read_excel(METADATA_PATH, header=None)

column_starts = [0, 4, 8, 12]

mapping = []

for start in column_starts:

    for _, row in raw.iloc[1:].iterrows():

        filename = row.iloc[start]
        label = row.iloc[start + 1]
        patient = row.iloc[start + 2]

        if pd.isna(filename):
            continue

        if pd.isna(patient):
            patient_id = None
        else:
            patient_id = str(int(float(patient)))

        mapping.append({
            "filename": str(filename).strip(),
            "metadata_label": str(label).strip(),
            "patient_id": patient_id
        })

mapping_df = pd.DataFrame(mapping)

# ==========================================
# 3. VERIFY METADATA
# ==========================================

manifest["filename"] = manifest["path"].apply(
    lambda x: Path(x).name
)

if mapping_df["filename"].duplicated().any():
    raise RuntimeError("Duplicate metadata filenames detected.")

audit = manifest.merge(
    mapping_df,
    on="filename",
    how="left",
    validate="one_to_one"
)

if len(audit) != 400:
    raise RuntimeError("Expected exactly 400 images.")

if audit["patient_id"].isna().sum() != 255:
    raise RuntimeError("Patient metadata count differs from previous audit.")

mismatches = audit[
    audit["label"] != audit["metadata_label"]
]

if len(mismatches):
    raise RuntimeError(
        "Metadata label mismatches detected. Stop and investigate."
    )

# ==========================================
# 4. CREATE GROUP IDENTIFIERS
# ==========================================

def make_group_id(row):

    if pd.notna(row["patient_id"]):
        return "patient_" + row["patient_id"]

    # Unique identifier for an image with unavailable patient ID
    return "unidentified_" + row["filename"]


audit["group_id"] = audit.apply(
    make_group_id,
    axis=1
)

# ==========================================
# 5. CREATE SEVEN GROUPED FOLDS
# ==========================================

splitter = StratifiedGroupKFold(
    n_splits=7,
    shuffle=True,
    random_state=42
)

audit["fold"] = -1

for fold_number, (_, heldout_indices) in enumerate(
    splitter.split(
        audit,
        audit["label"],
        groups=audit["group_id"]
    )
):
    audit.loc[
        audit.index[heldout_indices],
        "fold"
    ] = fold_number

if (audit["fold"] == -1).any():
    raise RuntimeError("Some images were not assigned a fold.")

# Fold 1: validation
# Fold 2: internal test
# Remaining five folds: training

audit["new_split"] = "train"

audit.loc[audit["fold"] == 1, "new_split"] = "validation"
audit.loc[audit["fold"] == 2, "new_split"] = "internal_test"

# ==========================================
# 6. VERIFY PATIENT SEPARATION
# ==========================================

known = audit.dropna(subset=["patient_id"])

patient_split_counts = (
    known.groupby("patient_id")["new_split"].nunique()
)

overlaps = patient_split_counts[
    patient_split_counts > 1
]

print("========== PATIENT GROUP VERIFICATION ==========")
print("Identified patients:", known["patient_id"].nunique())
print("Patients appearing across splits:", len(overlaps))

if len(overlaps) > 0:
    raise RuntimeError("Known patient leakage remains!")

print("✅ All identified patient groups stay within one split.")

# ==========================================
# 7. DISPLAY NEW SPLIT DISTRIBUTION
# ==========================================

print("\n========== NEW SPLIT SIZES ==========")

print(
    audit["new_split"].value_counts().to_string()
)

print("\n========== CLASS DISTRIBUTION ==========")

distribution = pd.crosstab(
    audit["new_split"],
    audit["label"]
)

print(distribution)

print("\n========== PATIENT METADATA COVERAGE ==========")

for split in ["train", "validation", "internal_test"]:

    subset = audit[audit["new_split"] == split]

    print(
        split,
        "| total:", len(subset),
        "| identified:", subset["patient_id"].notna().sum(),
        "| unidentified:", subset["patient_id"].isna().sum()
    )

# ==========================================
# 8. SAVE NEW MANIFEST
# ==========================================

audit["split"] = audit["new_split"]

audit.to_csv(
    NEW_MANIFEST_PATH,
    index=False
)

print("\n✅ PATIENT-GROUPED MANIFEST SAVED")
print(NEW_MANIFEST_PATH)

print("\nOriginal manifest preserved:")
print(OLD_MANIFEST_PATH)


In [ ]:

import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold

# ==========================================
# 1. LOAD EXISTING PATIENT-GROUPED MANIFEST
# ==========================================

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")

INPUT_PATH = (
    PROJECT_DIR / "split_manifest_patient_grouped.csv"
)

OUTPUT_PATH = (
    PROJECT_DIR / "split_manifest_patient_grouped_balanced_candidate.csv"
)

df = pd.read_csv(INPUT_PATH)

CLASSES = ["Benign", "InSitu", "Invasive", "Normal"]
SPLITS = ["train", "validation", "internal_test"]

assert len(df) == 400
assert "group_id" in df.columns
assert "patient_id" in df.columns

y = df["label"].to_numpy()
groups = df["group_id"].astype(str).to_numpy()

if set(y) != set(CLASSES):
    raise RuntimeError("Unexpected class names.")

# Target distribution
target_sizes = {
    "train": 280,
    "validation": 60,
    "internal_test": 60
}

target_classes = {
    "train": 70,
    "validation": 15,
    "internal_test": 15
}

target_matrix = np.array([
    [target_classes[s] for _ in CLASSES]
    for s in SPLITS
])

# ==========================================
# 2. SEARCH 500 RANDOM SEEDS
# ==========================================

best_score = float("inf")
best_split = None
best_details = None

print("Searching 500 random seeds...")

for seed in range(500):

    splitter = StratifiedGroupKFold(
        n_splits=7,
        shuffle=True,
        random_state=seed
    )

    folds = [
        test_indices
        for _, test_indices in splitter.split(
            np.zeros(len(df)),
            y,
            groups
        )
    ]

    # Try every ordered pair of different folds
    for val_fold in range(7):
        for test_fold in range(7):

            if val_fold == test_fold:
                continue

            candidate = np.full(
                len(df),
                "train",
                dtype=object
            )

            candidate[folds[val_fold]] = "validation"
            candidate[folds[test_fold]] = "internal_test"

            counts = pd.crosstab(
                pd.Series(candidate, name="split"),
                pd.Series(y, name="label")
            ).reindex(
                index=SPLITS,
                columns=CLASSES,
                fill_value=0
            ).to_numpy()

            sizes = np.array([
                np.sum(candidate == split)
                for split in SPLITS
            ])

            target_size_array = np.array([
                target_sizes[s] for s in SPLITS
            ])

            # Class balance is the main priority
            class_error = np.abs(
                counts - target_matrix
            ).sum()

            size_error = np.abs(
                sizes - target_size_array
            ).sum()

            # Lower score is better
            score = class_error + 0.5 * size_error

            if score < best_score:
                best_score = score
                best_split = candidate.copy()
                best_details = {
                    "seed": seed,
                    "validation_fold": val_fold,
                    "test_fold": test_fold,
                    "class_error": class_error,
                    "size_error": size_error
                }

# ==========================================
# 3. CREATE CANDIDATE MANIFEST
# ==========================================

candidate_df = df.copy()
candidate_df["split"] = best_split

print("\n========== BEST CANDIDATE ==========")
print(best_details)

print("\nSplit sizes:")
print(candidate_df["split"].value_counts().reindex(SPLITS))

print("\nClass distribution:")

distribution = pd.crosstab(
    candidate_df["split"],
    candidate_df["label"]
).reindex(
    index=SPLITS,
    columns=CLASSES,
    fill_value=0
)

print(distribution)

# ==========================================
# 4. VERIFY PATIENT SEPARATION
# ==========================================

known = candidate_df.dropna(subset=["patient_id"])

patient_overlaps = (
    known.groupby("patient_id")["split"].nunique()
)

patient_overlaps = patient_overlaps[
    patient_overlaps > 1
]

group_overlaps = (
    candidate_df.groupby("group_id")["split"].nunique()
)

group_overlaps = group_overlaps[group_overlaps > 1]

print("\nIdentified patient overlaps:", len(patient_overlaps))
print("Group overlaps:", len(group_overlaps))

if len(patient_overlaps) or len(group_overlaps):
    raise RuntimeError("Group leakage detected!")

# ==========================================
# 5. SAVE CANDIDATE SEPARATELY
# ==========================================

candidate_df.to_csv(OUTPUT_PATH, index=False)

print("\n✅ Candidate manifest saved:")
print(OUTPUT_PATH)

print("\nOriginal manifests have not been modified.")


In [ ]:

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")

CANDIDATE_PATH = (
    PROJECT_DIR /
    "split_manifest_patient_grouped_balanced_candidate.csv"
)

FINAL_PATH = (
    PROJECT_DIR /
    "split_manifest_patient_grouped_final.csv"
)

df = pd.read_csv(CANDIDATE_PATH)

# Validate the candidate before accepting it
assert len(df) == 400
assert df["path"].is_unique
assert df["group_id"].notna().all()

assert set(df["split"]) == {
    "train", "validation", "internal_test"
}

# Verify every original image still exists
missing_files = [
    p for p in df["path"]
    if not Path(p).exists()
]

if missing_files:
    raise FileNotFoundError(
        f"{len(missing_files)} original images are missing."
    )

# Verify class labels agree with metadata
mismatches = df[
    df["label"] != df["metadata_label"]
]

if len(mismatches):
    raise ValueError("Metadata label mismatch detected.")

# Verify no group crosses splits
group_overlaps = (
    df.groupby("group_id")["split"].nunique()
)

assert (group_overlaps <= 1).all()

# Create a clean permanent manifest
final_df = df[
    [
        "path",
        "filename",
        "label",
        "patient_id",
        "group_id",
        "split"
    ]
].copy()

final_df.to_csv(FINAL_PATH, index=False)

# Final verification
print("========== FINAL MANIFEST ==========")
print("Total images:", len(final_df))

print("\nSplit sizes:")
print(final_df["split"].value_counts())

print("\nClass distribution:")
print(pd.crosstab(final_df["split"], final_df["label"]))

print("\nIdentified patients:",
      final_df["patient_id"].nunique())

print("Patient/group overlaps:",
      int((group_overlaps > 1).sum()))

print("\nMissing patient IDs:",
      final_df["patient_id"].isna().sum())

print("\nSaved permanently:")
print(FINAL_PATH)

print("\n✅ Patient-grouped split finalized.")


In [ ]:

from pathlib import Path
import pandas as pd
from PIL import Image
import os

# ==========================================
# 1. CONFIGURATION
# ==========================================

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")

FINAL_MANIFEST = (
    PROJECT_DIR / "split_manifest_patient_grouped_final.csv"
)

OUTPUT_DIR = (
    PROJECT_DIR / "processed_224_patient_grouped"
)

CLASSES = ["Benign", "InSitu", "Invasive", "Normal"]

SPLITS = ["train", "validation", "internal_test"]

# ==========================================
# 2. LOAD FINAL PATIENT-GROUPED MANIFEST
# ==========================================

df = pd.read_csv(FINAL_MANIFEST)

if len(df) != 400:
    raise ValueError("Expected exactly 400 images.")

if df["group_id"].isna().any():
    raise ValueError("Missing group IDs.")

print("Images in manifest:", len(df))

print("\nSplit distribution:")
print(pd.crosstab(df["split"], df["label"]))

# ==========================================
# 3. CREATE OUTPUT FOLDERS
# ==========================================

for split in SPLITS:
    for class_name in CLASSES:
        (OUTPUT_DIR / split / class_name).mkdir(
            parents=True,
            exist_ok=True
        )

# ==========================================
# 4. PROCESS ORIGINAL TIFF IMAGES
# ==========================================

records = []
errors = []

for index, row in df.iterrows():

    source = Path(row["path"])
    class_name = row["label"]
    split = row["split"]

    if not source.exists():
        errors.append((str(source), "Original file missing"))
        continue

    output = (
        OUTPUT_DIR
        / split
        / class_name
        / f"{source.stem}.jpg"
    )

    try:
        # Read original image, not an existing 224 JPEG
        with Image.open(source) as img:

            img = img.convert("RGB")

            if img.size != (2048, 1536):
                raise ValueError(
                    f"Unexpected original dimensions: {img.size}"
                )

            # Preserve the full image aspect ratio
            img.thumbnail(
                (224, 224),
                Image.Resampling.LANCZOS
            )

            # White square canvas
            canvas = Image.new(
                "RGB",
                (224, 224),
                (255, 255, 255)
            )

            x = (224 - img.width) // 2
            y = (224 - img.height) // 2

            canvas.paste(img, (x, y))

            # Save atomically to Drive
            temporary = output.with_name(
                output.stem + ".tmp.jpg"
            )

            canvas.save(
                temporary,
                "JPEG",
                quality=95
            )

            os.replace(temporary, output)

        records.append({
            "filename": source.name,
            "original_path": str(source),
            "processed_path": str(output),
            "label": class_name,
            "split": split,
            "patient_id": row["patient_id"],
            "group_id": row["group_id"]
        })

    except Exception as exc:
        errors.append((str(source), str(exc)))

    if (index + 1) % 50 == 0:
        print(f"Progress: {index + 1}/400")

# ==========================================
# 5. VERIFY PROCESSED DATASET
# ==========================================

print("\n========== PROCESSING RESULTS ==========")

print("Processed images:", len(records))
print("Errors:", len(errors))

for split in SPLITS:

    split_total = 0

    print(f"\n{split.upper()}")

    for class_name in CLASSES:

        folder = OUTPUT_DIR / split / class_name

        files = list(folder.glob("*.jpg"))
        count = len(files)

        split_total += count

        print(f"{class_name:10s}: {count}")

        for file in files:
            with Image.open(file) as img:
                if img.size != (224, 224) or img.mode != "RGB":
                    raise RuntimeError(
                        f"Invalid processed image: {file}"
                    )

    print("TOTAL:", split_total)

# Save processed-image mapping
processed_manifest = pd.DataFrame(records)

processed_manifest.to_csv(
    PROJECT_DIR / "processed_patient_grouped_manifest.csv",
    index=False
)

if errors:
    print("\nERRORS:")
    for error in errors[:10]:
        print(error)

    raise RuntimeError(
        "Processing incomplete. Check errors before continuing."
    )

if len(records) != 400:
    raise RuntimeError("Expected exactly 400 processed images.")

print("\n✅ PATIENT-GROUPED DATASET PREPARATION COMPLETE")
print("Saved in:", OUTPUT_DIR)


In [ ]:
from pathlib import Path
import shutil

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")

DRIVE_DATA = PROJECT_DIR / "processed_224_patient_grouped"

LOCAL_DATA = Path(
    "/content/BACH_processed_224_patient_grouped"
)

# Copy from persistent Drive if necessary
if LOCAL_DATA.exists():
    shutil.rmtree(LOCAL_DATA)

shutil.copytree(DRIVE_DATA, LOCAL_DATA)

# Verify every class in every split
CLASSES = ["Benign", "InSitu", "Invasive", "Normal"]

EXPECTED = {
    "train": 280,
    "validation": 60,
    "internal_test": 60
}

grand_total = 0

for split, expected_total in EXPECTED.items():

    split_total = 0

    print(f"\n{split.upper()}")

    for class_name in CLASSES:

        folder = LOCAL_DATA / split / class_name
        files = list(folder.glob("*.jpg"))

        count = len(files)
        split_total += count

        print(f"{class_name:10s}: {count}")

    print("TOTAL:", split_total)

    if split_total != expected_total:
        raise RuntimeError(
            f"Incorrect image count in {split}"
        )

    grand_total += split_total

print("\nTotal images:", grand_total)

assert grand_total == 400

print("✅ Patient-grouped working dataset verified.")

In [ ]:

from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import json

# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")
DATA_DIR = Path("/content/BACH_processed_224_patient_grouped")
MODEL_DIR = PROJECT_DIR / "models"
LOG_DIR = PROJECT_DIR / "training_logs"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["Benign", "InSitu", "Invasive", "Normal"]

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 20
SEED = 42

tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

# Confirm the expected folders exist
for split in ["train", "validation", "internal_test"]:
    split_path = DATA_DIR / split
    assert split_path.exists(), f"Missing folder: {split_path}"

# --------------------------------------------------
# 2. Load patient-grouped datasets
# --------------------------------------------------

train_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR / "train",
    labels="inferred",
    label_mode="int",
    class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR / "validation",
    labels="inferred",
    label_mode="int",
    class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Keep internal_test completely untouched for now.
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

print("\nClass order:", CLASS_NAMES)
print("Training batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("Validation batches:", tf.data.experimental.cardinality(val_ds).numpy())

# --------------------------------------------------
# 3. Data augmentation
# --------------------------------------------------

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.10),
], name="data_augmentation")

# --------------------------------------------------
# 4. Fresh EfficientNetB0 from ImageNet
# --------------------------------------------------

base_model = keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

# Freeze the backbone for the initial training stage
base_model.trainable = False

inputs = keras.Input(shape=(224, 224, 3))

x = data_augmentation(inputs)

# EfficientNetB0 includes its own input rescaling.
# Do NOT add Rescaling(1./255) here.
x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.40)(x)

outputs = layers.Dense(
    len(CLASS_NAMES),
    activation="softmax",
    name="class_predictions"
)(x)

model = keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="BACH_EfficientNetB0_PatientGrouped"
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# --------------------------------------------------
# 5. Checkpoint and training callbacks
# --------------------------------------------------

best_loss_path = MODEL_DIR / "efficientnetb0_patient_grouped_head_best_loss.keras"
best_accuracy_path = MODEL_DIR / "efficientnetb0_patient_grouped_head_best_accuracy.keras"
restored_path = MODEL_DIR / "efficientnetb0_patient_grouped_head_earlystop_restored.keras"

csv_log_path = LOG_DIR / "efficientnetb0_patient_grouped_head_log.csv"

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(best_loss_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.ModelCheckpoint(
        filepath=str(best_accuracy_path),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),

    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.CSVLogger(
        filename=str(csv_log_path),
        append=False
    )
]

# --------------------------------------------------
# 6. Train
# --------------------------------------------------

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

# Save the model with the best validation-loss weights
# restored by EarlyStopping.
model.save(restored_path)

# --------------------------------------------------
# 7. Report training results
# --------------------------------------------------

best_val_accuracy = max(history.history["val_accuracy"])
best_val_loss = min(history.history["val_loss"])

print("\n" + "=" * 55)
print("PATIENT-GROUPED EFFICIENTNETB0 — HEAD TRAINING")
print("=" * 55)

print(f"Best validation accuracy: {best_val_accuracy:.4f} "
      f"({best_val_accuracy * 100:.2f}%)")

print(f"Best validation loss: {best_val_loss:.4f}")

print("\nSaved models:")
print(best_loss_path)
print(best_accuracy_path)
print(restored_path)

print("\nTraining log:")
print(csv_log_path)

print("\n✅ Patient-grouped EfficientNetB0 head training complete.")


In [ ]:

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from pathlib import Path

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# --------------------------------------------------
# 1. Paths and configuration
# --------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")
DATA_DIR = Path("/content/BACH_processed_224_patient_grouped")
MODEL_DIR = PROJECT_DIR / "models"
LOG_DIR = PROJECT_DIR / "training_logs"

CLASS_NAMES = ["Benign", "InSitu", "Invasive", "Normal"]
NUM_CLASSES = len(CLASS_NAMES)

# --------------------------------------------------
# 2. Load validation data in a fixed order
# --------------------------------------------------

val_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR / "validation",
    labels="inferred",
    label_mode="int",
    class_names=CLASS_NAMES,
    image_size=(224, 224),
    batch_size=16,
    shuffle=False
)

# True labels
y_true = np.concatenate([
    labels.numpy() for images, labels in val_ds
])

# Obtain filenames in the same order, when available
file_paths = getattr(val_ds, "file_paths", [])

print("Validation images:", len(y_true))
print("Class order:", CLASS_NAMES)

# --------------------------------------------------
# 3. Evaluate both checkpoints
# --------------------------------------------------

checkpoints = {
    "best_accuracy": MODEL_DIR / "efficientnetb0_patient_grouped_head_best_accuracy.keras",
    "best_loss": MODEL_DIR / "efficientnetb0_patient_grouped_head_best_loss.keras"
}

all_results = {}
prediction_tables = {}

for checkpoint_name, checkpoint_path in checkpoints.items():

    print("\n" + "=" * 65)
    print("CHECKPOINT:", checkpoint_name)
    print("=" * 65)

    model = keras.models.load_model(checkpoint_path)

    probabilities = model.predict(val_ds, verbose=0)

    y_pred = np.argmax(probabilities, axis=1)
    confidence = np.max(probabilities, axis=1)

    accuracy = accuracy_score(y_true, y_pred)
    balanced_acc = balanced_accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(
        y_true, y_pred,
        average="macro",
        zero_division=0
    )

    print(f"\nAccuracy:          {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"Balanced accuracy: {balanced_acc:.4f}")
    print(f"Macro F1-score:    {macro_f1:.4f}")

    print("\nCLASSIFICATION REPORT:\n")

    report = classification_report(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES)),
        target_names=CLASS_NAMES,
        zero_division=0
    )

    print(report)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(NUM_CLASSES))
    )

    print("CONFUSION MATRIX")
    print("Rows = Actual | Columns = Predicted")
    print(pd.DataFrame(
        cm,
        index=CLASS_NAMES,
        columns=CLASS_NAMES
    ))

    all_results[checkpoint_name] = {
        "accuracy": float(accuracy),
        "balanced_accuracy": float(balanced_acc),
        "macro_f1": float(macro_f1),
        "confusion_matrix": cm.tolist()
    }

    # Per-image predictions
    table = pd.DataFrame({
        "filename": (
            [Path(p).name for p in file_paths]
            if len(file_paths) == len(y_true)
            else [f"validation_image_{i:03d}" for i in range(len(y_true))]
        ),
        "actual": [CLASS_NAMES[i] for i in y_true],
        "predicted": [CLASS_NAMES[i] for i in y_pred],
        "confidence": confidence
    })

    for i, class_name in enumerate(CLASS_NAMES):
        table[f"prob_{class_name}"] = probabilities[:, i]

    table["correct"] = table["actual"] == table["predicted"]

    prediction_tables[checkpoint_name] = table

    print("\nMISCLASSIFIED IMAGES:")
    print(
        table.loc[
            ~table["correct"],
            ["filename", "actual", "predicted", "confidence"]
        ].to_string(index=False)
    )

# --------------------------------------------------
# 4. Compare checkpoints
# --------------------------------------------------

comparison = pd.DataFrame(all_results).T[
    ["accuracy", "balanced_accuracy", "macro_f1"]
]

print("\n" + "=" * 65)
print("CHECKPOINT COMPARISON")
print("=" * 65)
print(comparison)

# --------------------------------------------------
# 5. Save validation results to Google Drive
# --------------------------------------------------

for name, table in prediction_tables.items():
    table.to_csv(
        LOG_DIR / f"efficientnetb0_patient_grouped_{name}_validation_predictions.csv",
        index=False
    )

comparison.to_csv(
    LOG_DIR / "efficientnetb0_patient_grouped_checkpoint_comparison.csv"
)

print("\n✅ Validation evaluation complete.")
print("Results saved in:", LOG_DIR)


In [ ]:

from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")
DATA_DIR = Path("/content/BACH_processed_224_patient_grouped")

MODEL_DIR = PROJECT_DIR / "models"
LOG_DIR = PROJECT_DIR / "training_logs"

CLASS_NAMES = ["Benign", "InSitu", "Invasive", "Normal"]

BATCH_SIZE = 16
EPOCHS = 15
SEED = 42

tf.keras.utils.set_random_seed(SEED)

# --------------------------------------------------
# 2. Load datasets
# --------------------------------------------------

train_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR / "train",
    labels="inferred",
    label_mode="int",
    class_names=CLASS_NAMES,
    image_size=(224, 224),
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR / "validation",
    labels="inferred",
    label_mode="int",
    class_names=CLASS_NAMES,
    image_size=(224, 224),
    batch_size=BATCH_SIZE,
    shuffle=False
)

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

# --------------------------------------------------
# 3. Load best head-only checkpoint
# --------------------------------------------------

initial_checkpoint = (
    MODEL_DIR /
    "efficientnetb0_patient_grouped_head_best_accuracy.keras"
)

model = keras.models.load_model(initial_checkpoint)

print("Loaded:", initial_checkpoint)

# --------------------------------------------------
# 4. Unfreeze the final 40 backbone layers
# --------------------------------------------------

base_model = model.get_layer("efficientnetb0")

base_model.trainable = True

# Initially freeze every backbone layer
for layer in base_model.layers:
    layer.trainable = False

# Unfreeze only the last 40 top-level layers
for layer in base_model.layers[-40:]:
    layer.trainable = True

# Freeze BatchNormalization layers recursively
# to reduce instability on this relatively small dataset.
def freeze_batch_norm(layer):
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False

    for sublayer in getattr(layer, "layers", []):
        freeze_batch_norm(sublayer)

freeze_batch_norm(base_model)

print("\nTotal backbone layers:", len(base_model.layers))
print("Last 40 backbone layers selected for fine-tuning.")

print(
    "Trainable backbone layers:",
    sum(layer.trainable for layer in base_model.layers)
)

print(
    "Trainable parameters in full model:",
    sum(
        tf.size(variable).numpy()
        for variable in model.trainable_variables
    )
)

# --------------------------------------------------
# 5. Recompile with a low learning rate
# --------------------------------------------------

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# --------------------------------------------------
# 6. Save to new checkpoint names
# --------------------------------------------------

best_loss_path = (
    MODEL_DIR /
    "efficientnetb0_patient_grouped_ft_best_loss.keras"
)

best_accuracy_path = (
    MODEL_DIR /
    "efficientnetb0_patient_grouped_ft_best_accuracy.keras"
)

restored_path = (
    MODEL_DIR /
    "efficientnetb0_patient_grouped_ft_earlystop_restored.keras"
)

csv_log_path = (
    LOG_DIR /
    "efficientnetb0_patient_grouped_ft_log.csv"
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(best_loss_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.ModelCheckpoint(
        filepath=str(best_accuracy_path),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),

    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.CSVLogger(
        filename=str(csv_log_path),
        append=False
    )
]

# --------------------------------------------------
# 7. Fine-tune
# --------------------------------------------------

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

model.save(restored_path)

# --------------------------------------------------
# 8. Report results
# --------------------------------------------------

best_val_accuracy = max(history_ft.history["val_accuracy"])
best_val_loss = min(history_ft.history["val_loss"])

print("\n" + "=" * 55)
print("PATIENT-GROUPED EFFICIENTNETB0 FINE-TUNING")
print("=" * 55)

print(
    f"Best validation accuracy: "
    f"{best_val_accuracy:.4f} ({best_val_accuracy*100:.2f}%)"
)

print(f"Best validation loss: {best_val_loss:.4f}")

print("\nSaved checkpoints:")
print(best_loss_path)
print(best_accuracy_path)
print(restored_path)

print("\n✅ Fine-tuning experiment complete.")


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
from tensorflow import keras
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")
DATA_DIR = Path("/content/BACH_processed_224_patient_grouped")
MODEL_DIR = PROJECT_DIR / "models"
LOG_DIR = PROJECT_DIR / "training_logs"

CLASS_NAMES = ["Benign", "InSitu", "Invasive", "Normal"]

# Load validation data once, without shuffling
val_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR / "validation",
    labels="inferred",
    label_mode="int",
    class_names=CLASS_NAMES,
    image_size=(224, 224),
    batch_size=16,
    shuffle=False
)

y_true = np.concatenate([
    labels.numpy() for images, labels in val_ds
])

file_paths = val_ds.file_paths

checkpoints = {
    "Head_only_best_accuracy":
        MODEL_DIR / "efficientnetb0_patient_grouped_head_best_accuracy.keras",

    "Fine_tuned_best_accuracy":
        MODEL_DIR / "efficientnetb0_patient_grouped_ft_best_accuracy.keras",

    "Fine_tuned_best_loss":
        MODEL_DIR / "efficientnetb0_patient_grouped_ft_best_loss.keras"
}

comparison_rows = []
class_rows = []

for name, path in checkpoints.items():

    print("\n" + "=" * 65)
    print(name)
    print("=" * 65)

    model = keras.models.load_model(path)

    probabilities = model.predict(val_ds, verbose=0)
    y_pred = np.argmax(probabilities, axis=1)

    accuracy = accuracy_score(y_true, y_pred)
    balanced_acc = balanced_accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(
        y_true, y_pred,
        average="macro",
        zero_division=0
    )

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Balanced accuracy: {balanced_acc:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")

    print("\nClassification report:")
    report = classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3],
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0
    )

    print(classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3],
        target_names=CLASS_NAMES,
        zero_division=0
    ))

    print("Confusion matrix:")
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3])
    print(pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES))

    comparison_rows.append({
        "model": name,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_acc,
        "macro_f1": macro_f1
    })

    for class_name in CLASS_NAMES:
        class_rows.append({
            "model": name,
            "class": class_name,
            "precision": report[class_name]["precision"],
            "recall": report[class_name]["recall"],
            "f1_score": report[class_name]["f1-score"],
            "support": report[class_name]["support"]
        })

    predictions = pd.DataFrame({
        "filename": [Path(p).name for p in file_paths],
        "actual": [CLASS_NAMES[i] for i in y_true],
        "predicted": [CLASS_NAMES[i] for i in y_pred],
        "confidence": np.max(probabilities, axis=1),
        "correct": y_true == y_pred
    })

    predictions.to_csv(
        LOG_DIR / f"{name}_predictions.csv",
        index=False
    )

comparison = pd.DataFrame(comparison_rows)
class_comparison = pd.DataFrame(class_rows)

comparison.to_csv(
    LOG_DIR / "patient_grouped_model_comparison.csv",
    index=False
)

class_comparison.to_csv(
    LOG_DIR / "patient_grouped_per_class_comparison.csv",
    index=False
)

print("\n" + "=" * 65)
print("OVERALL MODEL COMPARISON")
print("=" * 65)
print(comparison.to_string(index=False))

print("\n" + "=" * 65)
print("PER-CLASS COMPARISON")
print("=" * 65)
print(
    class_comparison.pivot(
        index="class",
        columns="model",
        values="recall"
    ).to_string()
)

print("\n✅ Comparison saved to:", LOG_DIR)


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

from PIL import Image

# =========================================================
# 1. CONFIGURATION
# =========================================================

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")
DATASET_DIR = PROJECT_DIR / "dataset"

MODEL_DIR = PROJECT_DIR / "models"
FEATURE_DIR = PROJECT_DIR / "patch_mil"

FEATURE_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_MANIFEST = (
    PROJECT_DIR / "split_manifest_patient_grouped_final.csv"
)

BACKBONE_CHECKPOINT = (
    MODEL_DIR /
    "efficientnetb0_patient_grouped_head_best_accuracy.keras"
)

FEATURE_PATH = (
    FEATURE_DIR /
    "efficientnetb0_patch_features_train_validation.npy"
)

FEATURE_MANIFEST_PATH = (
    FEATURE_DIR /
    "patch_mil_train_validation_manifest.csv"
)

COORDINATES_PATH = (
    FEATURE_DIR /
    "patch_coordinates.npy"
)

CLASS_NAMES = ["Benign", "InSitu", "Invasive", "Normal"]

PATCH_SIZE = 224
STRIDE = 192
PATCH_BATCH_SIZE = 16

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

# =========================================================
# 2. READ THE FINAL PATIENT-GROUPED MANIFEST
# =========================================================

assert SPLIT_MANIFEST.exists(), SPLIT_MANIFEST
assert BACKBONE_CHECKPOINT.exists(), BACKBONE_CHECKPOINT

manifest = pd.read_csv(SPLIT_MANIFEST)

required_columns = {
    "filename", "label", "split", "group_id"
}

assert required_columns.issubset(manifest.columns), (
    f"Missing columns: {required_columns - set(manifest.columns)}"
)

assert set(manifest["label"]).issubset(CLASS_NAMES)

print("\nOriginal manifest:")
print(manifest["split"].value_counts())

# Verify there is no group overlap between splits.
group_sets = {
    split: set(
        manifest.loc[
            manifest["split"] == split, "group_id"
        ].astype(str)
    )
    for split in ["train", "validation", "internal_test"]
}

assert group_sets["train"].isdisjoint(group_sets["validation"])
assert group_sets["train"].isdisjoint(group_sets["internal_test"])
assert group_sets["validation"].isdisjoint(group_sets["internal_test"])

print("\n✅ Group separation verified.")

# Exclude internal_test completely for this phase.
selected = manifest[
    manifest["split"].isin(["train", "validation"])
].copy().reset_index(drop=True)

expected_counts = {"train": 280, "validation": 60}

actual_counts = selected["split"].value_counts().to_dict()

assert actual_counts == expected_counts, (
    f"Unexpected split counts: {actual_counts}"
)

print("\nImages selected for feature extraction:")
print(selected["split"].value_counts())

# =========================================================
# 3. RESOLVE ORIGINAL TIFF PATHS
# =========================================================

def resolve_original_image(row):

    # Preferred persistent Drive location.
    candidate = (
        DATASET_DIR /
        str(row["label"]) /
        Path(str(row["filename"])).name
    )

    if candidate.exists():
        return candidate

    # Fallback to the path recorded in the manifest.
    if "path" in row.index and pd.notna(row["path"]):

        recorded = Path(str(row["path"]))

        if recorded.exists():
            return recorded

    raise FileNotFoundError(
        f"Original TIFF not found: {row['filename']}"
    )

original_paths = [
    resolve_original_image(row)
    for _, row in selected.iterrows()
]

print("\nResolved original images:", len(original_paths))

# =========================================================
# 4. CALCULATE OVERLAPPING PATCH COORDINATES
# =========================================================

def calculate_positions(length, patch_size, stride):

    positions = list(
        range(0, length - patch_size + 1, stride)
    )

    last_position = length - patch_size

    if positions[-1] != last_position:
        positions.append(last_position)

    return positions


# Check the dimensions of the original images.
with Image.open(original_paths[0]) as img:
    image_width, image_height = img.size

print("\nOriginal image dimensions:", image_width, "x", image_height)

assert image_width == 2048 and image_height == 1536, (
    "Unexpected dimensions. Inspect the original images before proceeding."
)

x_positions = calculate_positions(
    image_width, PATCH_SIZE, STRIDE
)

y_positions = calculate_positions(
    image_height, PATCH_SIZE, STRIDE
)

# Row-major ordering: left to right, then top to bottom.
patch_coordinates = np.array(
    [
        (x, y)
        for y in y_positions
        for x in x_positions
    ],
    dtype=np.int32
)

NUM_PATCHES = len(patch_coordinates)

print("Horizontal positions:", len(x_positions))
print("Vertical positions:", len(y_positions))
print("Patches per image:", NUM_PATCHES)

assert NUM_PATCHES == 88

# Explicitly verify that every original image pixel is covered.
coverage = np.zeros(
    (image_height, image_width),
    dtype=np.uint8
)

for x, y in patch_coordinates:
    coverage[
        y:y + PATCH_SIZE,
        x:x + PATCH_SIZE
    ] += 1

assert coverage.min() > 0, "Some original pixels are uncovered!"

print("✅ Full image coverage verified.")

np.save(COORDINATES_PATH, patch_coordinates)

# =========================================================
# 5. LOAD THE WINNING IMAGE-LEVEL EFFICIENTNET CHECKPOINT
# =========================================================

print("\nLoading the head-only checkpoint...")

whole_image_model = keras.models.load_model(
    BACKBONE_CHECKPOINT
)

# Reuse its frozen ImageNet EfficientNet backbone.
backbone = whole_image_model.get_layer("efficientnetb0")
backbone.trainable = False

# EfficientNet output: 7 x 7 x 1280.
# Pool each patch into one 1280-dimensional feature vector.
pooled_features = layers.GlobalAveragePooling2D()(
    backbone.output
)

feature_extractor = keras.Model(
    inputs=backbone.input,
    outputs=pooled_features,
    name="BACH_Patch_EfficientNetB0_FeatureExtractor"
)

feature_extractor.trainable = False

print("Feature extractor output:", feature_extractor.output_shape)

assert feature_extractor.output_shape[-1] == 1280

# =========================================================
# 6. CREATE FEATURE STORAGE
# =========================================================

NUM_IMAGES = len(selected)
FEATURE_DIM = 1280

# float32 gives approximately 146 MiB for these 340 images.
features = np.lib.format.open_memmap(
    FEATURE_PATH,
    mode="w+",
    dtype=np.float32,
    shape=(NUM_IMAGES, NUM_PATCHES, FEATURE_DIM)
)

# =========================================================
# 7. EXTRACT FEATURES IMAGE BY IMAGE
# =========================================================

@tf.function
def extract_batch(batch):
    return feature_extractor(batch, training=False)


for image_index, image_path in enumerate(original_paths):

    with Image.open(image_path) as img:
        image = np.asarray(img.convert("RGB"))

    assert image.shape == (1536, 2048, 3), (
        f"Unexpected image shape for {image_path.name}: {image.shape}"
    )

    # Process patches in small batches to limit GPU memory use.
    for start in range(0, NUM_PATCHES, PATCH_BATCH_SIZE):

        end = min(
            start + PATCH_BATCH_SIZE,
            NUM_PATCHES
        )

        coordinates_batch = patch_coordinates[start:end]

        patch_batch = np.stack([
            image[
                y:y + PATCH_SIZE,
                x:x + PATCH_SIZE,
                :
            ]
            for x, y in coordinates_batch
        ]).astype(np.float32)

        # EfficientNet handles its own input rescaling.
        # Values must remain in the original 0-255 range.
        batch_features = extract_batch(
            tf.convert_to_tensor(patch_batch)
        ).numpy()

        features[
            image_index,
            start:end,
            :
        ] = batch_features

    if (image_index + 1) % 10 == 0:
        features.flush()

    if (image_index + 1) % 20 == 0:
        print(
            f"Processed {image_index + 1}/{NUM_IMAGES} images"
        )

features.flush()

# =========================================================
# 8. SAVE FEATURE MANIFEST
# =========================================================

selected.to_csv(
    FEATURE_MANIFEST_PATH,
    index=False
)

# Release the memory-mapped feature array.
del features

print("\n" + "=" * 60)
print("PATCH FEATURE EXTRACTION COMPLETE")
print("=" * 60)

print("Images processed:", NUM_IMAGES)
print("Patches per image:", NUM_PATCHES)
print("Total patches:", NUM_IMAGES * NUM_PATCHES)
print("Feature dimensions:", FEATURE_DIM)

print("\nFeature shape:", (
    NUM_IMAGES,
    NUM_PATCHES,
    FEATURE_DIM
))

print("\nSaved files:")
print(FEATURE_PATH)
print(FEATURE_MANIFEST_PATH)
print(COORDINATES_PATH)

print("\nTraining images: 280")
print("Validation images: 60")
print("Internal test images processed: 0")

print("\n✅ Patch feature dataset successfully created.")


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers, regularizers

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# =========================================================
# 1. CONFIGURATION
# =========================================================

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")
FEATURE_DIR = PROJECT_DIR / "patch_mil"
MODEL_DIR = PROJECT_DIR / "models"
LOG_DIR = PROJECT_DIR / "training_logs"

FEATURE_PATH = (
    FEATURE_DIR /
    "efficientnetb0_patch_features_train_validation.npy"
)

MANIFEST_PATH = (
    FEATURE_DIR /
    "patch_mil_train_validation_manifest.csv"
)

CLASS_NAMES = ["Benign", "InSitu", "Invasive", "Normal"]

NUM_PATCHES = 88
FEATURE_DIM = 1280

BATCH_SIZE = 8
EPOCHS = 100
SEED = 42

MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

# =========================================================
# 2. LOAD FEATURE DATA AND MANIFEST
# =========================================================

assert FEATURE_PATH.exists(), FEATURE_PATH
assert MANIFEST_PATH.exists(), MANIFEST_PATH

# Memory-map first to avoid unnecessary copying.
feature_memmap = np.load(
    FEATURE_PATH,
    mmap_mode="r"
)

manifest = pd.read_csv(MANIFEST_PATH)

print("\nFeature tensor shape:", feature_memmap.shape)
print("Manifest rows:", len(manifest))

assert feature_memmap.shape == (
    340, NUM_PATCHES, FEATURE_DIM
)

assert len(manifest) == 340

# Verify the saved feature manifest contains only
# the training and validation examples.
assert set(manifest["split"]) == {"train", "validation"}

assert manifest["split"].value_counts().to_dict() == {
    "train": 280,
    "validation": 60
}

# =========================================================
# 3. VERIFY PATIENT-GROUPED SEPARATION
# =========================================================

train_groups = set(
    manifest.loc[
        manifest["split"] == "train",
        "group_id"
    ].astype(str)
)

validation_groups = set(
    manifest.loc[
        manifest["split"] == "validation",
        "group_id"
    ].astype(str)
)

assert train_groups.isdisjoint(validation_groups)

print("✅ No group overlap between train and validation.")

# =========================================================
# 4. ENCODE IMAGE-LEVEL LABELS
# =========================================================

label_to_id = {
    class_name: index
    for index, class_name in enumerate(CLASS_NAMES)
}

assert set(manifest["label"]).issubset(label_to_id)

all_labels = np.array([
    label_to_id[label]
    for label in manifest["label"]
], dtype=np.int32)

train_mask = (
    manifest["split"].to_numpy() == "train"
)

val_mask = (
    manifest["split"].to_numpy() == "validation"
)

# Each row is one image bag, containing 88 patch features.
X_train = np.asarray(
    feature_memmap[train_mask],
    dtype=np.float32
)

X_val = np.asarray(
    feature_memmap[val_mask],
    dtype=np.float32
)

y_train = all_labels[train_mask]
y_val = all_labels[val_mask]

print("\nTraining bags:", X_train.shape)
print("Validation bags:", X_val.shape)

print("\nTraining class distribution:")
print(
    manifest.loc[train_mask, "label"].value_counts()
)

print("\nValidation class distribution:")
print(
    manifest.loc[val_mask, "label"].value_counts()
)

assert X_train.shape == (280, 88, 1280)
assert X_val.shape == (60, 88, 1280)

# Class weights are not applied in this first experiment.
# The training class counts are already reasonably balanced.

# =========================================================
# 5. BUILD GATED ATTENTION MIL
# =========================================================

inputs = keras.Input(
    shape=(NUM_PATCHES, FEATURE_DIM),
    name="patch_features"
)

# Project each patch feature into a smaller representation.
patch_embeddings = layers.Dense(
    256,
    activation="relu",
    kernel_regularizer=regularizers.l2(1e-4),
    name="patch_projection"
)(inputs)

# Gated attention: two parallel transformations.
attention_v = layers.Dense(
    128,
    activation="tanh",
    name="attention_tanh"
)(patch_embeddings)

attention_u = layers.Dense(
    128,
    activation="sigmoid",
    name="attention_sigmoid"
)(patch_embeddings)

gated_features = layers.Multiply(
    name="attention_gating"
)([attention_v, attention_u])

# One attention score per patch.
attention_scores = layers.Dense(
    1,
    use_bias=False,
    name="attention_scores"
)(gated_features)

# Normalize the scores across the 88 patches.
attention_weights = layers.Softmax(
    axis=1,
    name="attention_weights"
)(attention_scores)

# Weighted aggregation:
# (batch, 88, 1) x (batch, 88, 256)
# -> (batch, 1, 256)
weighted_bag = layers.Dot(
    axes=1,
    name="attention_weighted_pooling"
)([attention_weights, patch_embeddings])

bag_representation = layers.Flatten(
    name="bag_representation"
)(weighted_bag)

# Image-level classifier.
x = layers.Dropout(
    0.40,
    name="bag_dropout"
)(bag_representation)

x = layers.Dense(
    128,
    activation="relu",
    kernel_regularizer=regularizers.l2(1e-4),
    name="classifier_hidden"
)(x)

x = layers.Dropout(
    0.30,
    name="classifier_dropout"
)(x)

outputs = layers.Dense(
    len(CLASS_NAMES),
    activation="softmax",
    name="class_predictions"
)(x)

mil_model = keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="BACH_GatedAttention_MIL"
)

mil_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

mil_model.summary()

# Separate model to inspect learned attention weights later.
attention_model = keras.Model(
    inputs=inputs,
    outputs=attention_weights,
    name="BACH_MIL_Attention_Inspector"
)

# =========================================================
# 6. CALLBACKS AND CHECKPOINTS
# =========================================================

best_accuracy_path = (
    MODEL_DIR /
    "bach_attention_mil_best_accuracy.keras"
)

best_loss_path = (
    MODEL_DIR /
    "bach_attention_mil_best_loss.keras"
)

restored_path = (
    MODEL_DIR /
    "bach_attention_mil_earlystop_restored.keras"
)

csv_log_path = (
    LOG_DIR /
    "bach_attention_mil_training_log.csv"
)

callbacks = [

    keras.callbacks.ModelCheckpoint(
        filepath=str(best_accuracy_path),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.ModelCheckpoint(
        filepath=str(best_loss_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),

    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=12,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.CSVLogger(
        filename=str(csv_log_path),
        append=False
    )
]

# =========================================================
# 7. TRAIN THE MIL MODEL
# =========================================================

history = mil_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    shuffle=True,
    callbacks=callbacks,
    verbose=1
)

# Save early-stopping-restored weights.
mil_model.save(restored_path)

# =========================================================
# 8. REPORT BEST VALIDATION RESULTS
# =========================================================

best_val_accuracy = max(
    history.history["val_accuracy"]
)

best_val_loss = min(
    history.history["val_loss"]
)

best_accuracy_epoch = (
    int(np.argmax(history.history["val_accuracy"])) + 1
)

best_loss_epoch = (
    int(np.argmin(history.history["val_loss"])) + 1
)

print("\n" + "=" * 65)
print("BACH GATED ATTENTION MIL — TRAINING RESULTS")
print("=" * 65)

print(
    f"Best validation accuracy: "
    f"{best_val_accuracy:.4f} "
    f"({best_val_accuracy * 100:.2f}%)"
)

print("Best validation accuracy epoch:", best_accuracy_epoch)

print(f"Best validation loss: {best_val_loss:.4f}")
print("Best validation loss epoch:", best_loss_epoch)

print("\nSaved models:")
print(best_accuracy_path)
print(best_loss_path)
print(restored_path)

print("\nTraining log:")
print(csv_log_path)

print("\n✅ Attention-based MIL training complete.")


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

from tensorflow import keras

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# =========================================================
# 1. PATHS
# =========================================================

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")

FEATURE_DIR = PROJECT_DIR / "patch_mil"
MODEL_DIR = PROJECT_DIR / "models"
LOG_DIR = PROJECT_DIR / "training_logs"

FEATURE_PATH = (
    FEATURE_DIR /
    "efficientnetb0_patch_features_train_validation.npy"
)

MANIFEST_PATH = (
    FEATURE_DIR /
    "patch_mil_train_validation_manifest.csv"
)

COORDINATES_PATH = FEATURE_DIR / "patch_coordinates.npy"

CLASS_NAMES = ["Benign", "InSitu", "Invasive", "Normal"]

# =========================================================
# 2. LOAD VALIDATION FEATURES
# =========================================================

features = np.load(FEATURE_PATH, mmap_mode="r")
manifest = pd.read_csv(MANIFEST_PATH)
coordinates = np.load(COORDINATES_PATH)

val_mask = manifest["split"].to_numpy() == "validation"

val_manifest = manifest.loc[val_mask].reset_index(drop=True)

X_val = np.asarray(features[val_mask], dtype=np.float32)

label_to_id = {
    name: index
    for index, name in enumerate(CLASS_NAMES)
}

y_true = np.array([
    label_to_id[label]
    for label in val_manifest["label"]
])

print("Validation bags:", X_val.shape)
print("Validation labels:", y_true.shape)
print("Patch coordinates:", coordinates.shape)

assert X_val.shape == (60, 88, 1280)
assert coordinates.shape == (88, 2)

# =========================================================
# 3. CHECKPOINTS
# =========================================================

checkpoints = {
    "MIL_best_accuracy":
        MODEL_DIR / "bach_attention_mil_best_accuracy.keras",

    "MIL_best_loss":
        MODEL_DIR / "bach_attention_mil_best_loss.keras"
}

overall_results = []

# =========================================================
# 4. EVALUATE EACH MODEL
# =========================================================

for model_name, model_path in checkpoints.items():

    print("\n" + "=" * 70)
    print(model_name)
    print("=" * 70)

    assert model_path.exists(), f"Missing checkpoint: {model_path}"

    model = keras.models.load_model(model_path)

    probabilities = model.predict(X_val, verbose=0)

    y_pred = np.argmax(probabilities, axis=1)

    accuracy = accuracy_score(y_true, y_pred)

    balanced_acc = balanced_accuracy_score(
        y_true,
        y_pred
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    print(f"\nAccuracy:          {accuracy:.4f}")
    print(f"Balanced accuracy: {balanced_acc:.4f}")
    print(f"Macro F1-score:    {macro_f1:.4f}")

    print("\nCLASSIFICATION REPORT:")

    print(classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3],
        target_names=CLASS_NAMES,
        zero_division=0
    ))

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3]
    )

    print("CONFUSION MATRIX:")
    print(pd.DataFrame(
        cm,
        index=CLASS_NAMES,
        columns=CLASS_NAMES
    ))

    overall_results.append({
        "model": model_name,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_acc,
        "macro_f1": macro_f1
    })

    # -----------------------------------------------------
    # 5. EXTRACT ATTENTION WEIGHTS
    # -----------------------------------------------------

    attention_model = keras.Model(
        inputs=model.input,
        outputs=model.get_layer("attention_weights").output
    )

    attention = attention_model.predict(
        X_val,
        verbose=0
    ).squeeze(-1)

    assert attention.shape == (60, 88)

    # Each image's attention weights should sum to 1.
    assert np.allclose(
        attention.sum(axis=1),
        1.0,
        atol=1e-5
    )

    print("\nAttention shape:", attention.shape)
    print(
        "Mean attention weight:",
        float(attention.mean())
    )

    # Save complete attention arrays.
    attention_path = (
        FEATURE_DIR /
        f"{model_name}_attention_weights.npy"
    )

    np.save(attention_path, attention)

    # -----------------------------------------------------
    # 6. SAVE TOP 10 PATCHES PER IMAGE
    # -----------------------------------------------------

    attention_records = []

    for image_index in range(len(val_manifest)):

        row = val_manifest.iloc[image_index]

        top_indices = np.argsort(
            attention[image_index]
        )[::-1][:10]

        for rank, patch_index in enumerate(top_indices, start=1):

            x, y = coordinates[patch_index]

            attention_records.append({
                "filename": row["filename"],
                "actual_class": row["label"],
                "predicted_class": CLASS_NAMES[y_pred[image_index]],
                "patch_index": int(patch_index),
                "x": int(x),
                "y": int(y),
                "attention_weight": float(
                    attention[image_index, patch_index]
                ),
                "rank": rank
            })

    attention_df = pd.DataFrame(attention_records)

    attention_csv_path = (
        FEATURE_DIR /
        f"{model_name}_top10_attention_patches.csv"
    )

    attention_df.to_csv(
        attention_csv_path,
        index=False
    )

    print("\nTop 10 attention patches saved:")
    print(attention_csv_path)

    print("\nExample: highest-attention patches")
    print(
        attention_df.head(10).to_string(index=False)
    )

    # -----------------------------------------------------
    # 7. SAVE IMAGE-LEVEL PREDICTIONS
    # -----------------------------------------------------

    prediction_df = pd.DataFrame({
        "filename": val_manifest["filename"],
        "actual": [CLASS_NAMES[i] for i in y_true],
        "predicted": [CLASS_NAMES[i] for i in y_pred],
        "confidence": np.max(probabilities, axis=1),
        "correct": y_true == y_pred
    })

    for index, class_name in enumerate(CLASS_NAMES):
        prediction_df[f"prob_{class_name}"] = probabilities[:, index]

    prediction_df.to_csv(
        LOG_DIR / f"{model_name}_validation_predictions.csv",
        index=False
    )

# =========================================================
# 8. COMPARE AGAINST WHOLE-IMAGE BASELINE
# =========================================================

comparison = pd.DataFrame([
    {
        "model": "Whole_image_EfficientNetB0",
        "accuracy": 0.733333,
        "balanced_accuracy": 0.722474,
        "macro_f1": 0.720728
    },
    *overall_results
])

comparison.to_csv(
    LOG_DIR / "whole_image_vs_attention_mil_comparison.csv",
    index=False
)

print("\n" + "=" * 70)
print("FINAL VALIDATION COMPARISON")
print("=" * 70)

print(comparison.to_string(index=False))

print("\n✅ MIL checkpoint evaluation complete.")
print("Attention weights and top patches saved in:", FEATURE_DIR)
print("Prediction reports saved in:", LOG_DIR)


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from matplotlib.patches import Rectangle

# =========================================================
# 1. PATHS
# =========================================================

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")

DATASET_DIR = PROJECT_DIR / "dataset"
FEATURE_DIR = PROJECT_DIR / "patch_mil"
LOG_DIR = PROJECT_DIR / "training_logs"

MANIFEST_PATH = (
    FEATURE_DIR /
    "patch_mil_train_validation_manifest.csv"
)

ATTENTION_PATH = (
    FEATURE_DIR /
    "MIL_best_accuracy_attention_weights.npy"
)

COORDINATES_PATH = (
    FEATURE_DIR /
    "patch_coordinates.npy"
)

PREDICTIONS_PATH = (
    LOG_DIR /
    "MIL_best_accuracy_validation_predictions.csv"
)

OUTPUT_DIR = FEATURE_DIR / "attention_visualizations"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PATCH_SIZE = 224

# =========================================================
# 2. LOAD DATA
# =========================================================

manifest = pd.read_csv(MANIFEST_PATH)
predictions = pd.read_csv(PREDICTIONS_PATH)

attention = np.load(ATTENTION_PATH)
coordinates = np.load(COORDINATES_PATH)

val_manifest = (
    manifest[manifest["split"] == "validation"]
    .reset_index(drop=True)
)

assert len(val_manifest) == 60
assert attention.shape == (60, 88)
assert coordinates.shape == (88, 2)

assert np.allclose(
    attention.sum(axis=1),
    1.0,
    atol=1e-5
)

# Merge predictions by filename to avoid relying on
# accidental row ordering.
val_manifest = val_manifest.merge(
    predictions[
        ["filename", "predicted", "confidence", "correct"]
    ],
    on="filename",
    how="left",
    validate="one_to_one"
)

assert val_manifest["predicted"].notna().all()

# =========================================================
# 3. SELECT REPRESENTATIVE VALIDATION IMAGES
# =========================================================

misclassified = val_manifest[
    ~val_manifest["correct"].astype(bool)
].copy()

# Prioritize confident mistakes.
misclassified = misclassified.sort_values(
    "confidence",
    ascending=False
)

# Take up to two errors from each actual class.
selected_rows = []

for class_name in ["Benign", "InSitu", "Invasive", "Normal"]:

    class_errors = misclassified[
        misclassified["label"] == class_name
    ]

    selected_rows.extend(
        class_errors.head(2).index.tolist()
    )

# Add one correctly classified example from each class.
for class_name in ["Benign", "InSitu", "Invasive", "Normal"]:

    class_correct = val_manifest[
        (val_manifest["label"] == class_name)
        & (val_manifest["correct"].astype(bool))
    ]

    if len(class_correct):
        selected_rows.append(class_correct.index[0])

selected_rows = list(dict.fromkeys(selected_rows))

print("Selected images:", len(selected_rows))
print("\nSelected examples:")

print(
    val_manifest.loc[
        selected_rows,
        ["filename", "label", "predicted", "confidence", "correct"]
    ].to_string(index=False)
)

# =========================================================
# 4. VISUALIZATION FUNCTION
# =========================================================

def visualize_attention(image_index):

    row = val_manifest.iloc[image_index]

    image_path = (
        DATASET_DIR /
        str(row["label"]) /
        Path(str(row["filename"])).name
    )

    if not image_path.exists():
        raise FileNotFoundError(image_path)

    with Image.open(image_path) as img:
        image = np.asarray(img.convert("RGB"))

    height, width = image.shape[:2]

    weights = attention[image_index]

    # Accumulate each patch's attention across its area.
    attention_map = np.zeros(
        (height, width),
        dtype=np.float32
    )

    coverage = np.zeros(
        (height, width),
        dtype=np.float32
    )

    for patch_index, (x, y) in enumerate(coordinates):

        attention_map[
            y:y + PATCH_SIZE,
            x:x + PATCH_SIZE
        ] += weights[patch_index]

        coverage[
            y:y + PATCH_SIZE,
            x:x + PATCH_SIZE
        ] += 1.0

    # Average scores where patches overlap.
    attention_map = attention_map / np.maximum(
        coverage, 1.0
    )

    # Normalize for visual comparison within this image.
    attention_min = attention_map.min()
    attention_max = attention_map.max()

    normalized_map = (
        (attention_map - attention_min)
        / (attention_max - attention_min + 1e-8)
    )

    top_indices = np.argsort(weights)[::-1][:5]

    # -----------------------------------------------------
    # Plot
    # -----------------------------------------------------

    fig, axes = plt.subplots(
        1, 3,
        figsize=(19, 7)
    )

    fig.suptitle(
        f"{row['filename']} | Actual: {row['label']} | "
        f"Predicted: {row['predicted']} | "
        f"Confidence: {row['confidence']:.3f}",
        fontsize=13
    )

    axes[0].imshow(image)
    axes[0].set_title("Original microscopy image")
    axes[0].axis("off")

    axes[1].imshow(image)
    heatmap = axes[1].imshow(
        normalized_map,
        cmap="jet",
        alpha=0.45,
        vmin=0,
        vmax=1
    )

    axes[1].set_title("Attention overlay")
    axes[1].axis("off")

    fig.colorbar(
        heatmap,
        ax=axes[1],
        fraction=0.046,
        pad=0.04,
        label="Relative attention"
    )

    axes[2].imshow(image)
    axes[2].set_title("Top 5 attended patches")

    for rank, patch_index in enumerate(top_indices, start=1):

        x, y = coordinates[patch_index]

        rect = Rectangle(
            (x, y),
            PATCH_SIZE,
            PATCH_SIZE,
            linewidth=2,
            edgecolor="yellow",
            facecolor="none"
        )

        axes[2].add_patch(rect)

        axes[2].text(
            x + 4,
            y + 18,
            str(rank),
            color="black",
            fontsize=10,
            weight="bold",
            bbox=dict(facecolor="yellow", alpha=0.85)
        )

    axes[2].axis("off")

    plt.tight_layout()

    output_path = (
        OUTPUT_DIR /
        f"{Path(row['filename']).stem}_attention.png"
    )

    plt.savefig(
        output_path,
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    print("Saved:", output_path)

    print("\nTop five patches:")

    for rank, patch_index in enumerate(top_indices, start=1):

        x, y = coordinates[patch_index]

        print(
            f"Rank {rank}: "
            f"Patch {patch_index}, "
            f"x={x}, y={y}, "
            f"weight={weights[patch_index]:.5f}"
        )

# =========================================================
# 5. GENERATE VISUALIZATIONS
# =========================================================

for image_index in selected_rows:
    visualize_attention(image_index)

print("\n✅ Attention visualizations generated.")
print("Output directory:", OUTPUT_DIR)


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")
LOG_DIR = PROJECT_DIR / "training_logs"

CLASS_NAMES = ["Benign", "InSitu", "Invasive", "Normal"]
PROB_COLS = [f"prob_{name}" for name in CLASS_NAMES]

# ---------------------------------------------------------
# 1. Load prediction files
# ---------------------------------------------------------

whole_path = (
    LOG_DIR /
    "efficientnetb0_patient_grouped_best_accuracy_validation_predictions.csv"
)

mil_path = (
    LOG_DIR /
    "MIL_best_accuracy_validation_predictions.csv"
)

whole = pd.read_csv(whole_path)
mil = pd.read_csv(mil_path)

print("Whole-image rows:", len(whole))
print("MIL rows:", len(mil))

# ---------------------------------------------------------
# 2. Create extension-independent image identifiers
# ---------------------------------------------------------

whole["image_id"] = whole["filename"].apply(
    lambda x: Path(str(x)).stem
)

mil["image_id"] = mil["filename"].apply(
    lambda x: Path(str(x)).stem
)

# Check for duplicate image identifiers.
assert whole["image_id"].is_unique, "Duplicate whole-image IDs"
assert mil["image_id"].is_unique, "Duplicate MIL image IDs"

# Check both models have the same 60 images.
whole_ids = set(whole["image_id"])
mil_ids = set(mil["image_id"])

print("\nImages only in whole-image predictions:",
      sorted(whole_ids - mil_ids))

print("Images only in MIL predictions:",
      sorted(mil_ids - whole_ids))

assert whole_ids == mil_ids, (
    "The two prediction files contain different image sets."
)

# ---------------------------------------------------------
# 3. Merge predictions
# ---------------------------------------------------------

merged = whole.merge(
    mil,
    on="image_id",
    suffixes=("_whole", "_mil"),
    validate="one_to_one"
)

assert len(merged) == 60

# Confirm the actual labels agree.
assert (
    merged["actual_whole"] == merged["actual_mil"]
).all(), "Actual labels disagree between models."

print("\n✅ All 60 validation images matched successfully.")

# ---------------------------------------------------------
# 4. Obtain probability matrices
# ---------------------------------------------------------

whole_probs = merged[
    [f"{col}_whole" for col in PROB_COLS]
].to_numpy(dtype=np.float32)

mil_probs = merged[
    [f"{col}_mil" for col in PROB_COLS]
].to_numpy(dtype=np.float32)

assert whole_probs.shape == (60, 4)
assert mil_probs.shape == (60, 4)

# Equal-weight probability averaging.
ensemble_probs = (whole_probs + mil_probs) / 2.0

y_true = np.array([
    CLASS_NAMES.index(label)
    for label in merged["actual_whole"]
])

whole_pred = np.argmax(whole_probs, axis=1)
mil_pred = np.argmax(mil_probs, axis=1)
ensemble_pred = np.argmax(ensemble_probs, axis=1)

# ---------------------------------------------------------
# 5. Calculate metrics
# ---------------------------------------------------------

def calculate_metrics(y_true, predictions, name):

    return {
        "model": name,
        "accuracy": accuracy_score(y_true, predictions),
        "balanced_accuracy": balanced_accuracy_score(
            y_true, predictions
        ),
        "macro_f1": f1_score(
            y_true,
            predictions,
            average="macro",
            zero_division=0
        )
    }

comparison = pd.DataFrame([
    calculate_metrics(
        y_true, whole_pred, "Whole-image EfficientNetB0"
    ),
    calculate_metrics(
        y_true, mil_pred, "Attention MIL"
    ),
    calculate_metrics(
        y_true, ensemble_pred, "Equal probability ensemble"
    )
])

print("\n" + "=" * 65)
print("MODEL COMPARISON")
print("=" * 65)

print(comparison.to_string(index=False))

# ---------------------------------------------------------
# 6. Ensemble classification report
# ---------------------------------------------------------

print("\nENSEMBLE CLASSIFICATION REPORT")

print(classification_report(
    y_true,
    ensemble_pred,
    labels=[0, 1, 2, 3],
    target_names=CLASS_NAMES,
    zero_division=0
))

print("\nENSEMBLE CONFUSION MATRIX")

ensemble_cm = confusion_matrix(
    y_true,
    ensemble_pred,
    labels=[0, 1, 2, 3]
)

print(pd.DataFrame(
    ensemble_cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
))

# ---------------------------------------------------------
# 7. Compare complementary predictions
# ---------------------------------------------------------

whole_correct = whole_pred == y_true
mil_correct = mil_pred == y_true
ensemble_correct = ensemble_pred == y_true

print("\nCORRECTNESS OVERLAP")

print("Both correct:",
      int(np.sum(whole_correct & mil_correct)))

print("Only whole-image correct:",
      int(np.sum(whole_correct & ~mil_correct)))

print("Only MIL correct:",
      int(np.sum(~whole_correct & mil_correct)))

print("Both wrong:",
      int(np.sum(~whole_correct & ~mil_correct)))

print("Ensemble correct:",
      int(np.sum(ensemble_correct)))

# ---------------------------------------------------------
# 8. Save predictions and results
# ---------------------------------------------------------

merged["ensemble_prediction"] = [
    CLASS_NAMES[i] for i in ensemble_pred
]

merged["ensemble_confidence"] = np.max(
    ensemble_probs,
    axis=1
)

merged["ensemble_correct"] = ensemble_correct

for i, class_name in enumerate(CLASS_NAMES):
    merged[f"ensemble_prob_{class_name}"] = ensemble_probs[:, i]

comparison.to_csv(
    LOG_DIR / "whole_image_mil_ensemble_comparison.csv",
    index=False
)

merged.to_csv(
    LOG_DIR / "whole_image_mil_ensemble_predictions.csv",
    index=False
)

print("\n✅ Ensemble evaluation complete.")
print("Results saved in:", LOG_DIR)


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image

# =====================================================
# 1. CONFIGURATION
# =====================================================

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")

DATA_DIR = Path("/content/BACH_processed_224_patient_grouped")

FEATURE_DIR = PROJECT_DIR / "patch_mil"
MODEL_DIR = PROJECT_DIR / "models"

FEATURE_DIR.mkdir(parents=True, exist_ok=True)

PATCH_MANIFEST = (
    FEATURE_DIR /
    "patch_mil_train_validation_manifest.csv"
)

CHECKPOINT = (
    MODEL_DIR /
    "efficientnetb0_patient_grouped_head_best_accuracy.keras"
)

OUTPUT_FEATURES = (
    FEATURE_DIR /
    "efficientnetb0_global_features_train_validation.npy"
)

OUTPUT_MANIFEST = (
    FEATURE_DIR /
    "hybrid_global_local_train_validation_manifest.csv"
)

BATCH_SIZE = 16
IMAGE_SIZE = (224, 224)
FEATURE_DIM = 1280

# =====================================================
# 2. LOAD THE EXISTING MANIFEST
# =====================================================

manifest = pd.read_csv(PATCH_MANIFEST)

assert len(manifest) == 340

assert manifest["split"].value_counts().to_dict() == {
    "train": 280,
    "validation": 60
}

print("Images:", len(manifest))
print(manifest["split"].value_counts())

# =====================================================
# 3. RESOLVE PROCESSED WHOLE-IMAGE FILES
# =====================================================

def resolve_processed_path(row):

    folder = (
        DATA_DIR /
        str(row["split"]) /
        str(row["label"])
    )

    stem = Path(str(row["filename"])).stem

    for extension in [".jpg", ".jpeg", ".png"]:
        candidate = folder / f"{stem}{extension}"

        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        f"Processed image not found for {row['filename']} "
        f"in {folder}"
    )


image_paths = [
    resolve_processed_path(row)
    for _, row in manifest.iterrows()
]

print("\nResolved processed images:", len(image_paths))

# =====================================================
# 4. LOAD THE FROZEN EFFICIENTNET BACKBONE
# =====================================================

assert CHECKPOINT.exists()

whole_image_model = keras.models.load_model(CHECKPOINT)

backbone = whole_image_model.get_layer("efficientnetb0")
backbone.trainable = False

# Global-average-pool the full-image feature map.
global_output = layers.GlobalAveragePooling2D(
    name="global_feature_pooling"
)(backbone.output)

global_extractor = keras.Model(
    inputs=backbone.input,
    outputs=global_output,
    name="BACH_Global_EfficientNet_FeatureExtractor"
)

global_extractor.trainable = False

print("\nGlobal extractor output:", global_extractor.output_shape)

assert global_extractor.output_shape == (None, FEATURE_DIM)

# =====================================================
# 5. PREPARE GLOBAL FEATURE STORAGE
# =====================================================

global_features = np.lib.format.open_memmap(
    OUTPUT_FEATURES,
    mode="w+",
    dtype=np.float32,
    shape=(len(manifest), FEATURE_DIM)
)

# =====================================================
# 6. EXTRACT FEATURES IN BATCHES
# =====================================================

@tf.function
def extract_global_features(images):
    return global_extractor(images, training=False)


for start in range(0, len(image_paths), BATCH_SIZE):

    end = min(start + BATCH_SIZE, len(image_paths))

    image_batch = []

    for image_path in image_paths[start:end]:

        with Image.open(image_path) as img:

            img = img.convert("RGB")

            if img.size != IMAGE_SIZE:
                img = img.resize(
                    IMAGE_SIZE,
                    Image.Resampling.LANCZOS
                )

            image_array = np.asarray(img, dtype=np.float32)

            image_batch.append(image_array)

    image_batch = np.stack(image_batch)

    # Do not divide by 255.
    # EfficientNet contains its own input rescaling.
    batch_features = extract_global_features(
        tf.convert_to_tensor(image_batch)
    ).numpy()

    global_features[start:end] = batch_features

    if end % 32 == 0 or end == len(image_paths):
        global_features.flush()
        print(f"Processed {end}/{len(image_paths)} images")

global_features.flush()
del global_features

# =====================================================
# 7. SAVE MATCHING MANIFEST
# =====================================================

manifest.to_csv(
    OUTPUT_MANIFEST,
    index=False
)

# =====================================================
# 8. VERIFY SAVED FEATURES
# =====================================================

check = np.load(
    OUTPUT_FEATURES,
    mmap_mode="r"
)

assert check.shape == (340, 1280)

assert np.isfinite(check).all()

print("\n" + "=" * 55)
print("GLOBAL FEATURE EXTRACTION COMPLETE")
print("=" * 55)

print("Feature shape:", check.shape)
print("Training images:", 280)
print("Validation images:", 60)
print("Internal test images: 0")

print("\nSaved:")
print(OUTPUT_FEATURES)
print(OUTPUT_MANIFEST)

print("\n✅ Global feature extraction verified.")


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers, regularizers

# =========================================================
# 1. CONFIGURATION
# =========================================================

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")

FEATURE_DIR = PROJECT_DIR / "patch_mil"
MODEL_DIR = PROJECT_DIR / "models"
LOG_DIR = PROJECT_DIR / "training_logs"

GLOBAL_FEATURE_PATH = (
    FEATURE_DIR /
    "efficientnetb0_global_features_train_validation.npy"
)

LOCAL_FEATURE_PATH = (
    FEATURE_DIR /
    "efficientnetb0_patch_features_train_validation.npy"
)

GLOBAL_MANIFEST_PATH = (
    FEATURE_DIR /
    "hybrid_global_local_train_validation_manifest.csv"
)

LOCAL_MANIFEST_PATH = (
    FEATURE_DIR /
    "patch_mil_train_validation_manifest.csv"
)

CLASS_NAMES = ["Benign", "InSitu", "Invasive", "Normal"]

NUM_CLASSES = 4
NUM_PATCHES = 88
FEATURE_DIM = 1280

BATCH_SIZE = 8
EPOCHS = 100
SEED = 42

MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

# =========================================================
# 2. LOAD FEATURE ARRAYS
# =========================================================

assert GLOBAL_FEATURE_PATH.exists()
assert LOCAL_FEATURE_PATH.exists()

global_features = np.load(
    GLOBAL_FEATURE_PATH,
    mmap_mode="r"
)

local_features = np.load(
    LOCAL_FEATURE_PATH,
    mmap_mode="r"
)

global_manifest = pd.read_csv(GLOBAL_MANIFEST_PATH)
local_manifest = pd.read_csv(LOCAL_MANIFEST_PATH)

print("\nGlobal feature shape:", global_features.shape)
print("Local feature shape:", local_features.shape)

assert global_features.shape == (340, 1280)
assert local_features.shape == (340, 88, 1280)

# =========================================================
# 3. VERIFY FEATURE ALIGNMENT
# =========================================================

# Both feature extractors must refer to the same images
# in exactly the same order.

assert len(global_manifest) == len(local_manifest) == 340

for column in ["filename", "label", "split", "group_id"]:

    assert (
        global_manifest[column].astype(str).tolist()
        == local_manifest[column].astype(str).tolist()
    ), f"Manifest mismatch in column: {column}"

print("✅ Global and local feature alignment verified.")

assert set(global_manifest["split"]) == {
    "train", "validation"
}

split_counts = global_manifest["split"].value_counts().to_dict()

assert split_counts == {
    "train": 280,
    "validation": 60
}

# Verify no group crosses the train-validation boundary.
train_groups = set(
    global_manifest.loc[
        global_manifest["split"] == "train",
        "group_id"
    ].astype(str)
)

val_groups = set(
    global_manifest.loc[
        global_manifest["split"] == "validation",
        "group_id"
    ].astype(str)
)

assert train_groups.isdisjoint(val_groups)

print("✅ Patient-group separation verified.")

# =========================================================
# 4. ENCODE IMAGE-LEVEL LABELS
# =========================================================

label_to_id = {
    name: index
    for index, name in enumerate(CLASS_NAMES)
}

assert set(global_manifest["label"]).issubset(label_to_id)

all_labels = np.array([
    label_to_id[label]
    for label in global_manifest["label"]
], dtype=np.int32)

train_mask = (
    global_manifest["split"].to_numpy() == "train"
)

val_mask = (
    global_manifest["split"].to_numpy() == "validation"
)

# =========================================================
# 5. CREATE TRAINING AND VALIDATION ARRAYS
# =========================================================

X_global_train = np.asarray(
    global_features[train_mask],
    dtype=np.float32
)

X_local_train = np.asarray(
    local_features[train_mask],
    dtype=np.float32
)

X_global_val = np.asarray(
    global_features[val_mask],
    dtype=np.float32
)

X_local_val = np.asarray(
    local_features[val_mask],
    dtype=np.float32
)

y_train = all_labels[train_mask]
y_val = all_labels[val_mask]

print("\nTraining global:", X_global_train.shape)
print("Training local:", X_local_train.shape)

print("Validation global:", X_global_val.shape)
print("Validation local:", X_local_val.shape)

assert X_global_train.shape == (280, 1280)
assert X_local_train.shape == (280, 88, 1280)

assert X_global_val.shape == (60, 1280)
assert X_local_val.shape == (60, 88, 1280)

# =========================================================
# 6. BUILD THE GLOBAL BRANCH
# =========================================================

global_input = keras.Input(
    shape=(FEATURE_DIM,),
    name="global_features"
)

global_projection = layers.Dense(
    128,
    use_bias=False,
    kernel_regularizer=regularizers.l2(1e-4),
    name="global_projection"
)(global_input)

global_projection = layers.LayerNormalization(
    name="global_normalization"
)(global_projection)

global_embedding = layers.Activation(
    "relu",
    name="global_embedding"
)(global_projection)

global_embedding = layers.Dropout(
    0.35,
    name="global_dropout"
)(global_embedding)

# =========================================================
# 7. BUILD THE LOCAL ATTENTION BRANCH
# =========================================================

local_input = keras.Input(
    shape=(NUM_PATCHES, FEATURE_DIM),
    name="local_features"
)

patch_embeddings = layers.Dense(
    128,
    use_bias=False,
    kernel_regularizer=regularizers.l2(1e-4),
    name="local_projection"
)(local_input)

patch_embeddings = layers.LayerNormalization(
    name="local_normalization"
)(patch_embeddings)

patch_embeddings = layers.Activation(
    "relu",
    name="local_embedding"
)(patch_embeddings)

# Gated attention
attention_v = layers.Dense(
    64,
    activation="tanh",
    name="hybrid_attention_tanh"
)(patch_embeddings)

attention_u = layers.Dense(
    64,
    activation="sigmoid",
    name="hybrid_attention_sigmoid"
)(patch_embeddings)

gated_features = layers.Multiply(
    name="hybrid_attention_gating"
)([attention_v, attention_u])

attention_scores = layers.Dense(
    1,
    use_bias=False,
    name="hybrid_attention_scores"
)(gated_features)

attention_weights = layers.Softmax(
    axis=1,
    name="hybrid_attention_weights"
)(attention_scores)

# Aggregate all 88 patch embeddings.
weighted_local = layers.Dot(
    axes=1,
    name="hybrid_attention_pooling"
)([attention_weights, patch_embeddings])

local_embedding = layers.Flatten(
    name="local_bag_representation"
)(weighted_local)

# =========================================================
# 8. FUSE GLOBAL AND LOCAL REPRESENTATIONS
# =========================================================

combined = layers.Concatenate(
    name="global_local_fusion"
)([global_embedding, local_embedding])

fusion = layers.Dense(
    128,
    activation="relu",
    kernel_regularizer=regularizers.l2(1e-4),
    name="fusion_dense"
)(combined)

fusion = layers.LayerNormalization(
    name="fusion_normalization"
)(fusion)

fusion = layers.Dropout(
    0.45,
    name="fusion_dropout"
)(fusion)

fusion = layers.Dense(
    64,
    activation="relu",
    kernel_regularizer=regularizers.l2(1e-4),
    name="fusion_hidden"
)(fusion)

fusion = layers.Dropout(
    0.35,
    name="fusion_hidden_dropout"
)(fusion)

outputs = layers.Dense(
    NUM_CLASSES,
    activation="softmax",
    name="class_predictions"
)(fusion)

# =========================================================
# 9. CREATE HYBRID MODEL
# =========================================================

hybrid_model = keras.Model(
    inputs=[global_input, local_input],
    outputs=outputs,
    name="BACH_Hybrid_Global_Local_MIL"
)

hybrid_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

hybrid_model.summary()

# Separate model for extracting attention weights later.
hybrid_attention_model = keras.Model(
    inputs=hybrid_model.inputs,
    outputs=hybrid_model.get_layer(
        "hybrid_attention_weights"
    ).output,
    name="BACH_Hybrid_Attention_Inspector"
)

# =========================================================
# 10. CHECKPOINT PATHS
# =========================================================

best_accuracy_path = (
    MODEL_DIR /
    "bach_hybrid_global_local_best_accuracy.keras"
)

best_loss_path = (
    MODEL_DIR /
    "bach_hybrid_global_local_best_loss.keras"
)

restored_path = (
    MODEL_DIR /
    "bach_hybrid_global_local_earlystop_restored.keras"
)

log_path = (
    LOG_DIR /
    "bach_hybrid_global_local_training_log.csv"
)

callbacks = [

    keras.callbacks.ModelCheckpoint(
        filepath=str(best_accuracy_path),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.ModelCheckpoint(
        filepath=str(best_loss_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    ),

    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=12,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.CSVLogger(
        filename=str(log_path),
        append=False
    )
]

# =========================================================
# 11. TRAIN HYBRID MODEL
# =========================================================

history = hybrid_model.fit(
    {
        "global_features": X_global_train,
        "local_features": X_local_train
    },
    y_train,

    validation_data=(
        {
            "global_features": X_global_val,
            "local_features": X_local_val
        },
        y_val
    ),

    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    shuffle=True,
    callbacks=callbacks,
    verbose=1
)

# Save the early-stopping-restored model.
hybrid_model.save(restored_path)

# =========================================================
# 12. REPORT RESULTS
# =========================================================

best_val_accuracy = max(
    history.history["val_accuracy"]
)

best_val_loss = min(
    history.history["val_loss"]
)

best_accuracy_epoch = (
    int(np.argmax(history.history["val_accuracy"])) + 1
)

best_loss_epoch = (
    int(np.argmin(history.history["val_loss"])) + 1
)

print("\n" + "=" * 65)
print("BACH HYBRID GLOBAL-LOCAL MIL RESULTS")
print("=" * 65)

print(
    f"Best validation accuracy: "
    f"{best_val_accuracy:.4f} "
    f"({best_val_accuracy * 100:.2f}%)"
)

print("Best accuracy epoch:", best_accuracy_epoch)

print(f"Best validation loss: {best_val_loss:.4f}")
print("Best loss epoch:", best_loss_epoch)

print("\nSaved models:")
print(best_accuracy_path)
print(best_loss_path)
print(restored_path)

print("\nTraining log:")
print(log_path)

print("\n✅ Hybrid Global–Local training complete.")


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

from tensorflow import keras

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# =====================================================
# 1. PATHS
# =====================================================

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")

FEATURE_DIR = PROJECT_DIR / "patch_mil"
MODEL_DIR = PROJECT_DIR / "models"
LOG_DIR = PROJECT_DIR / "training_logs"

GLOBAL_FEATURE_PATH = (
    FEATURE_DIR /
    "efficientnetb0_global_features_train_validation.npy"
)

LOCAL_FEATURE_PATH = (
    FEATURE_DIR /
    "efficientnetb0_patch_features_train_validation.npy"
)

MANIFEST_PATH = (
    FEATURE_DIR /
    "hybrid_global_local_train_validation_manifest.csv"
)

CLASS_NAMES = ["Benign", "InSitu", "Invasive", "Normal"]

# =====================================================
# 2. LOAD FEATURES AND VALIDATION LABELS
# =====================================================

global_features = np.load(
    GLOBAL_FEATURE_PATH,
    mmap_mode="r"
)

local_features = np.load(
    LOCAL_FEATURE_PATH,
    mmap_mode="r"
)

manifest = pd.read_csv(MANIFEST_PATH)

val_mask = manifest["split"].to_numpy() == "validation"

val_manifest = (
    manifest.loc[val_mask]
    .reset_index(drop=True)
)

X_global_val = np.asarray(
    global_features[val_mask],
    dtype=np.float32
)

X_local_val = np.asarray(
    local_features[val_mask],
    dtype=np.float32
)

label_to_id = {
    name: index
    for index, name in enumerate(CLASS_NAMES)
}

y_true = np.array([
    label_to_id[label]
    for label in val_manifest["label"]
])

assert X_global_val.shape == (60, 1280)
assert X_local_val.shape == (60, 88, 1280)
assert len(y_true) == 60

print("Validation images:", len(y_true))

# =====================================================
# 3. EVALUATE BOTH CHECKPOINTS
# =====================================================

checkpoints = {
    "Hybrid_best_accuracy":
        MODEL_DIR / "bach_hybrid_global_local_best_accuracy.keras",

    "Hybrid_best_loss":
        MODEL_DIR / "bach_hybrid_global_local_best_loss.keras"
}

results = []
saved_probabilities = {}

for model_name, model_path in checkpoints.items():

    print("\n" + "=" * 65)
    print(model_name)
    print("=" * 65)

    assert model_path.exists(), model_path

    model = keras.models.load_model(model_path)

    inputs = {
        "global_features": X_global_val,
        "local_features": X_local_val
    }

    probabilities = model.predict(
        inputs,
        verbose=0
    )

    y_pred = np.argmax(probabilities, axis=1)

    accuracy = accuracy_score(y_true, y_pred)

    balanced_acc = balanced_accuracy_score(
        y_true, y_pred
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    print(f"\nAccuracy:          {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"Balanced accuracy: {balanced_acc:.4f}")
    print(f"Macro F1-score:    {macro_f1:.4f}")

    print("\nCLASSIFICATION REPORT")

    print(classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3],
        target_names=CLASS_NAMES,
        zero_division=0
    ))

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3]
    )

    print("CONFUSION MATRIX")
    print(pd.DataFrame(
        cm,
        index=CLASS_NAMES,
        columns=CLASS_NAMES
    ))

    results.append({
        "model": model_name,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_acc,
        "macro_f1": macro_f1
    })

    saved_probabilities[model_name] = probabilities

    # Save individual image predictions.
    prediction_table = pd.DataFrame({
        "filename": val_manifest["filename"],
        "actual": [CLASS_NAMES[i] for i in y_true],
        "predicted": [CLASS_NAMES[i] for i in y_pred],
        "confidence": probabilities.max(axis=1),
        "correct": y_true == y_pred
    })

    for index, class_name in enumerate(CLASS_NAMES):
        prediction_table[f"prob_{class_name}"] = probabilities[:, index]

    prediction_table.to_csv(
        LOG_DIR / f"{model_name}_validation_predictions.csv",
        index=False
    )

    # Extract and save hybrid attention weights.
    attention_model = keras.Model(
        inputs=model.inputs,
        outputs=model.get_layer(
            "hybrid_attention_weights"
        ).output
    )

    attention_weights = attention_model.predict(
        inputs,
        verbose=0
    ).squeeze(-1)

    assert attention_weights.shape == (60, 88)

    assert np.allclose(
        attention_weights.sum(axis=1),
        1.0,
        atol=1e-5
    )

    np.save(
        FEATURE_DIR / f"{model_name}_attention_weights.npy",
        attention_weights
    )

# =====================================================
# 4. COMPARE CHECKPOINTS
# =====================================================

comparison = pd.DataFrame(results)

print("\n" + "=" * 65)
print("HYBRID CHECKPOINT COMPARISON")
print("=" * 65)

print(comparison.to_string(index=False))

comparison.to_csv(
    LOG_DIR / "hybrid_checkpoint_comparison.csv",
    index=False
)

if len(saved_probabilities) == 2:

    difference = np.max(
        np.abs(
            saved_probabilities["Hybrid_best_accuracy"]
            -
            saved_probabilities["Hybrid_best_loss"]
        )
    )

    print("\nMaximum checkpoint probability difference:", difference)

print("\n✅ Hybrid validation evaluation completed.")
print("Results saved in:", LOG_DIR)


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image

# =====================================================
# 1. CONFIGURATION
# =====================================================

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")

ORIGINAL_DIR = PROJECT_DIR / "dataset"
PROCESSED_DIR = PROJECT_DIR / "processed_224_patient_grouped"

FEATURE_DIR = PROJECT_DIR / "patch_mil"
MODEL_DIR = PROJECT_DIR / "models"

FINAL_MANIFEST = (
    PROJECT_DIR / "split_manifest_patient_grouped_final.csv"
)

GLOBAL_OUTPUT = (
    FEATURE_DIR / "internal_test_global_features.npy"
)

LOCAL_OUTPUT = (
    FEATURE_DIR / "internal_test_local_features.npy"
)

TEST_MANIFEST_OUTPUT = (
    FEATURE_DIR / "internal_test_feature_manifest.csv"
)

COORDINATES_PATH = FEATURE_DIR / "patch_coordinates.npy"

BACKBONE_CHECKPOINT = (
    MODEL_DIR /
    "efficientnetb0_patient_grouped_head_best_accuracy.keras"
)

IMAGE_SIZE = (224, 224)
PATCH_SIZE = 224
PATCH_BATCH_SIZE = 16
FEATURE_DIM = 1280

CLASS_NAMES = ["Benign", "InSitu", "Invasive", "Normal"]

# =====================================================
# 2. LOAD AND VERIFY FINAL SPLIT
# =====================================================

manifest = pd.read_csv(FINAL_MANIFEST)

train_groups = set(
    manifest.loc[
        manifest["split"] == "train",
        "group_id"
    ].astype(str)
)

validation_groups = set(
    manifest.loc[
        manifest["split"] == "validation",
        "group_id"
    ].astype(str)
)

test_groups = set(
    manifest.loc[
        manifest["split"] == "internal_test",
        "group_id"
    ].astype(str)
)

assert train_groups.isdisjoint(validation_groups)
assert train_groups.isdisjoint(test_groups)
assert validation_groups.isdisjoint(test_groups)

test_manifest = (
    manifest[manifest["split"] == "internal_test"]
    .copy()
    .reset_index(drop=True)
)

assert len(test_manifest) == 60

expected_distribution = {
    "Benign": 12,
    "InSitu": 16,
    "Invasive": 17,
    "Normal": 15
}

actual_distribution = (
    test_manifest["label"].value_counts().to_dict()
)

assert actual_distribution == expected_distribution, (
    f"Unexpected test distribution: {actual_distribution}"
)

print("Internal-test images:", len(test_manifest))
print("Class distribution:", actual_distribution)
print("✅ Patient-group separation verified.")

# =====================================================
# 3. LOAD THE SAME PATCH COORDINATES
# =====================================================

coordinates = np.load(COORDINATES_PATH)

assert coordinates.shape == (88, 2)

print("\nPatch coordinates:", coordinates.shape)

# =====================================================
# 4. RESOLVE ORIGINAL TIFFS AND PROCESSED IMAGES
# =====================================================

def resolve_original_path(row):

    candidate = (
        ORIGINAL_DIR /
        str(row["label"]) /
        Path(str(row["filename"])).name
    )

    if candidate.exists():
        return candidate

    if "path" in row.index and pd.notna(row["path"]):
        recorded = Path(str(row["path"]))

        if recorded.exists():
            return recorded

    raise FileNotFoundError(candidate)


def resolve_processed_path(row):

    folder = (
        PROCESSED_DIR /
        "internal_test" /
        str(row["label"])
    )

    stem = Path(str(row["filename"])).stem

    for extension in [".jpg", ".jpeg", ".png"]:
        candidate = folder / f"{stem}{extension}"

        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        f"Processed image missing: {stem} in {folder}"
    )


original_paths = [
    resolve_original_path(row)
    for _, row in test_manifest.iterrows()
]

processed_paths = [
    resolve_processed_path(row)
    for _, row in test_manifest.iterrows()
]

print("\nOriginal TIFFs resolved:", len(original_paths))
print("Processed images resolved:", len(processed_paths))

# =====================================================
# 5. LOAD THE SAME FROZEN EFFICIENTNET BACKBONE
# =====================================================

assert BACKBONE_CHECKPOINT.exists()

whole_model = keras.models.load_model(
    BACKBONE_CHECKPOINT
)

backbone = whole_model.get_layer("efficientnetb0")
backbone.trainable = False

global_output = layers.GlobalAveragePooling2D()(
    backbone.output
)

feature_extractor = keras.Model(
    inputs=backbone.input,
    outputs=global_output,
    name="BACH_Final_Test_Feature_Extractor"
)

feature_extractor.trainable = False

print("\nFeature extractor:", feature_extractor.output_shape)

# =====================================================
# 6. CREATE FEATURE ARRAYS
# =====================================================

NUM_IMAGES = 60
NUM_PATCHES = 88

global_features = np.lib.format.open_memmap(
    GLOBAL_OUTPUT,
    mode="w+",
    dtype=np.float32,
    shape=(NUM_IMAGES, FEATURE_DIM)
)

local_features = np.lib.format.open_memmap(
    LOCAL_OUTPUT,
    mode="w+",
    dtype=np.float32,
    shape=(NUM_IMAGES, NUM_PATCHES, FEATURE_DIM)
)

# =====================================================
# 7. FEATURE EXTRACTION FUNCTION
# =====================================================

@tf.function
def extract_features(batch):
    return feature_extractor(batch, training=False)

# =====================================================
# 8. EXTRACT GLOBAL AND LOCAL FEATURES
# =====================================================

for image_index in range(NUM_IMAGES):

    # -------------------------------------------------
    # A. Global feature from processed whole image
    # -------------------------------------------------

    with Image.open(processed_paths[image_index]) as img:

        img = img.convert("RGB")

        if img.size != IMAGE_SIZE:
            img = img.resize(
                IMAGE_SIZE,
                Image.Resampling.LANCZOS
            )

        global_array = np.asarray(
            img,
            dtype=np.float32
        )

    global_embedding = extract_features(
        tf.convert_to_tensor(global_array[None, ...])
    ).numpy()[0]

    global_features[image_index] = global_embedding

    # -------------------------------------------------
    # B. Local features from original TIFF patches
    # -------------------------------------------------

    with Image.open(original_paths[image_index]) as img:
        original = np.asarray(img.convert("RGB"))

    assert original.shape == (1536, 2048, 3), (
        f"Unexpected dimensions for {original_paths[image_index].name}: "
        f"{original.shape}"
    )

    for start in range(
        0, NUM_PATCHES, PATCH_BATCH_SIZE
    ):

        end = min(
            start + PATCH_BATCH_SIZE,
            NUM_PATCHES
        )

        batch_coordinates = coordinates[start:end]

        patch_batch = np.stack([
            original[
                y:y + PATCH_SIZE,
                x:x + PATCH_SIZE,
                :
            ]
            for x, y in batch_coordinates
        ]).astype(np.float32)

        patch_embeddings = extract_features(
            tf.convert_to_tensor(patch_batch)
        ).numpy()

        local_features[
            image_index,
            start:end
        ] = patch_embeddings

    if (image_index + 1) % 5 == 0:
        global_features.flush()
        local_features.flush()

        print(
            f"Processed internal-test images: "
            f"{image_index + 1}/{NUM_IMAGES}"
        )

global_features.flush()
local_features.flush()

del global_features
del local_features

# =====================================================
# 9. SAVE TEST MANIFEST
# =====================================================

test_manifest.to_csv(
    TEST_MANIFEST_OUTPUT,
    index=False
)

# =====================================================
# 10. VERIFY FEATURES
# =====================================================

check_global = np.load(
    GLOBAL_OUTPUT,
    mmap_mode="r"
)

check_local = np.load(
    LOCAL_OUTPUT,
    mmap_mode="r"
)

assert check_global.shape == (60, 1280)
assert check_local.shape == (60, 88, 1280)

assert np.isfinite(check_global).all()
assert np.isfinite(check_local).all()

print("\n" + "=" * 60)
print("INTERNAL-TEST FEATURE EXTRACTION COMPLETE")
print("=" * 60)

print("Global features:", check_global.shape)
print("Local features:", check_local.shape)

print("Internal-test images processed:", 60)
print("Training images processed: 0")
print("Validation images processed: 0")

print("\nSaved:")
print(GLOBAL_OUTPUT)
print(LOCAL_OUTPUT)
print(TEST_MANIFEST_OUTPUT)

print("\n✅ Independent test features prepared.")


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from tensorflow import keras

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# =====================================================
# 1. PATHS
# =====================================================

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")

FEATURE_DIR = PROJECT_DIR / "patch_mil"
MODEL_DIR = PROJECT_DIR / "models"
RESULT_DIR = PROJECT_DIR / "final_evaluation"

RESULT_DIR.mkdir(parents=True, exist_ok=True)

GLOBAL_PATH = FEATURE_DIR / "internal_test_global_features.npy"
LOCAL_PATH = FEATURE_DIR / "internal_test_local_features.npy"

TEST_MANIFEST_PATH = (
    FEATURE_DIR / "internal_test_feature_manifest.csv"
)

FULL_MANIFEST_PATH = (
    PROJECT_DIR / "split_manifest_patient_grouped_final.csv"
)

MODEL_PATH = (
    MODEL_DIR /
    "bach_hybrid_global_local_best_accuracy.keras"
)

CLASS_NAMES = ["Benign", "InSitu", "Invasive", "Normal"]
NUM_CLASSES = len(CLASS_NAMES)

# =====================================================
# 2. LOAD TEST FEATURES
# =====================================================

X_global_test = np.asarray(
    np.load(GLOBAL_PATH, mmap_mode="r"),
    dtype=np.float32
)

X_local_test = np.asarray(
    np.load(LOCAL_PATH, mmap_mode="r"),
    dtype=np.float32
)

test_manifest = pd.read_csv(TEST_MANIFEST_PATH)
full_manifest = pd.read_csv(FULL_MANIFEST_PATH)

assert X_global_test.shape == (60, 1280)
assert X_local_test.shape == (60, 88, 1280)
assert len(test_manifest) == 60

assert test_manifest["split"].eq("internal_test").all()

assert np.isfinite(X_global_test).all()
assert np.isfinite(X_local_test).all()

print("Global test features:", X_global_test.shape)
print("Local test features:", X_local_test.shape)

# =====================================================
# 3. VERIFY PATIENT-GROUP SEPARATION
# =====================================================

train_groups = set(
    full_manifest.loc[
        full_manifest["split"] == "train",
        "group_id"
    ].astype(str)
)

validation_groups = set(
    full_manifest.loc[
        full_manifest["split"] == "validation",
        "group_id"
    ].astype(str)
)

test_groups = set(
    full_manifest.loc[
        full_manifest["split"] == "internal_test",
        "group_id"
    ].astype(str)
)

assert train_groups.isdisjoint(test_groups)
assert validation_groups.isdisjoint(test_groups)

print("\n✅ Patient-group separation reverified.")

# Verify feature manifest corresponds to the final split.
expected_test = (
    full_manifest[full_manifest["split"] == "internal_test"]
    .reset_index(drop=True)
)

for column in ["filename", "label", "group_id"]:
    assert (
        test_manifest[column].astype(str).tolist()
        == expected_test[column].astype(str).tolist()
    ), f"Test manifest mismatch: {column}"

# =====================================================
# 4. ENCODE TRUE LABELS
# =====================================================

label_to_id = {
    name: index
    for index, name in enumerate(CLASS_NAMES)
}

assert set(test_manifest["label"]).issubset(label_to_id)

y_true = np.array([
    label_to_id[label]
    for label in test_manifest["label"]
])

print("\nActual test distribution:")
print(test_manifest["label"].value_counts().reindex(
    CLASS_NAMES, fill_value=0
))

# =====================================================
# 5. LOAD THE SELECTED HYBRID CHECKPOINT
# =====================================================

assert MODEL_PATH.exists(), MODEL_PATH

print("\nLoading selected Hybrid Global-Local model...")

model = keras.models.load_model(MODEL_PATH)

print("Model:", model.name)

# =====================================================
# 6. ONE-TIME FINAL PREDICTION
# =====================================================

test_inputs = {
    "global_features": X_global_test,
    "local_features": X_local_test
}

probabilities = model.predict(
    test_inputs,
    batch_size=8,
    verbose=1
)

assert probabilities.shape == (60, NUM_CLASSES)

y_pred = np.argmax(probabilities, axis=1)

# =====================================================
# 7. FINAL PERFORMANCE METRICS
# =====================================================

accuracy = accuracy_score(y_true, y_pred)

balanced_acc = balanced_accuracy_score(
    y_true,
    y_pred
)

macro_f1 = f1_score(
    y_true,
    y_pred,
    average="macro",
    zero_division=0
)

weighted_f1 = f1_score(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

correct_count = int(np.sum(y_true == y_pred))
incorrect_count = len(y_true) - correct_count

# 95% Wilson confidence interval for accuracy.
n = len(y_true)
z = 1.959963984540054
p = accuracy

denominator = 1 + (z**2 / n)

center = (
    p + z**2 / (2 * n)
) / denominator

margin = (
    z * np.sqrt(
        p * (1 - p) / n
        + z**2 / (4 * n**2)
    ) / denominator
)

ci_lower = max(0.0, center - margin)
ci_upper = min(1.0, center + margin)

print("\n" + "=" * 70)
print("FINAL PATIENT-GROUPED INTERNAL-TEST RESULTS")
print("=" * 70)

print(f"\nCorrect predictions:   {correct_count}/{n}")
print(f"Incorrect predictions: {incorrect_count}/{n}")

print(f"\nAccuracy:          {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"95% Wilson CI:     {ci_lower*100:.2f}% to {ci_upper*100:.2f}%")
print(f"Balanced accuracy:{balanced_acc:.4f}")
print(f"Macro F1-score:    {macro_f1:.4f}")
print(f"Weighted F1-score: {weighted_f1:.4f}")

# =====================================================
# 8. CLASSIFICATION REPORT
# =====================================================

report_dict = classification_report(
    y_true,
    y_pred,
    labels=list(range(NUM_CLASSES)),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0
)

print("\nCLASSIFICATION REPORT\n")

print(classification_report(
    y_true,
    y_pred,
    labels=list(range(NUM_CLASSES)),
    target_names=CLASS_NAMES,
    zero_division=0
))

# =====================================================
# 9. CONFUSION MATRIX
# =====================================================

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(NUM_CLASSES))
)

cm_df = pd.DataFrame(
    cm,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
)

print("\nCONFUSION MATRIX")
print("Rows = Actual | Columns = Predicted\n")
print(cm_df)

# =====================================================
# 10. INDIVIDUAL TEST PREDICTIONS
# =====================================================

prediction_df = pd.DataFrame({
    "filename": test_manifest["filename"],
    "actual": [CLASS_NAMES[i] for i in y_true],
    "predicted": [CLASS_NAMES[i] for i in y_pred],
    "confidence": probabilities.max(axis=1),
    "correct": y_true == y_pred
})

for index, class_name in enumerate(CLASS_NAMES):
    prediction_df[f"prob_{class_name}"] = probabilities[:, index]

print("\nINDIVIDUAL PREDICTIONS")

print(
    prediction_df[
        ["filename", "actual", "predicted", "confidence", "correct"]
    ].to_string(index=False)
)

# =====================================================
# 11. SAVE FINAL RESULTS
# =====================================================

prediction_df.to_csv(
    RESULT_DIR / "hybrid_internal_test_predictions.csv",
    index=False
)

cm_df.to_csv(
    RESULT_DIR / "hybrid_internal_test_confusion_matrix.csv"
)

with open(
    RESULT_DIR / "hybrid_internal_test_metrics.json",
    "w"
) as f:

    json.dump({
        "model": "BACH_Hybrid_Global_Local_MIL",
        "checkpoint": MODEL_PATH.name,
        "test_images": n,
        "correct": correct_count,
        "incorrect": incorrect_count,
        "accuracy": accuracy,
        "wilson_95_ci": [ci_lower, ci_upper],
        "balanced_accuracy": balanced_acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "classification_report": report_dict,
        "confusion_matrix": cm.tolist(),
        "class_order": CLASS_NAMES
    }, f, indent=4)

print("\n" + "=" * 70)
print("FINAL EVALUATION SAVED")
print("=" * 70)

print("Results folder:", RESULT_DIR)
print("✅ Independent internal-test evaluation complete.")


PHASE 2 - PROJECT

LOSS FUNCTION AND OPTIMISER

In [ ]:

from google.colab import drive
drive.mount('/content/drive')


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf

# Main project directory
PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")

# Existing trained hybrid model
MODEL_PATH = (
    PROJECT_DIR /
    "models" /
    "bach_hybrid_global_local_best_accuracy.keras"
)

# Previously extracted global and local features
GLOBAL_FEATURE_PATH = (
    PROJECT_DIR /
    "patch_mil" /
    "efficientnetb0_global_features_train_validation.npy"
)

LOCAL_FEATURE_PATH = (
    PROJECT_DIR /
    "patch_mil" /
    "efficientnetb0_patch_features_train_validation.npy"
)

# Training/validation manifest
MANIFEST_PATH = (
    PROJECT_DIR /
    "patch_mil" /
    "hybrid_global_local_train_validation_manifest.csv"
)

# Original final patient-grouped manifest
FINAL_MANIFEST_PATH = (
    PROJECT_DIR /
    "split_manifest_patient_grouped_final.csv"
)

# Check required files
required_files = [
    MODEL_PATH,
    GLOBAL_FEATURE_PATH,
    LOCAL_FEATURE_PATH,
    MANIFEST_PATH,
    FINAL_MANIFEST_PATH
]

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

print("\nChecking project files...\n")

for file_path in required_files:
    status = "FOUND ✅" if file_path.exists() else "MISSING ❌"
    print(status, file_path)

assert all(path.exists() for path in required_files), (
    "One or more required project files are missing."
)

# Verify feature shapes
global_features = np.load(
    GLOBAL_FEATURE_PATH,
    mmap_mode="r"
)

local_features = np.load(
    LOCAL_FEATURE_PATH,
    mmap_mode="r"
)

manifest = pd.read_csv(MANIFEST_PATH)

print("\nGlobal features:", global_features.shape)
print("Local features:", local_features.shape)
print("Manifest rows:", len(manifest))

assert global_features.shape == (340, 1280)
assert local_features.shape == (340, 88, 1280)
assert len(manifest) == 340

print("\nTraining/validation distribution:")
print(manifest["split"].value_counts())

# Create a separate directory for our new experiment.
# Do not overwrite existing checkpoints.
EXPERIMENT_DIR = PROJECT_DIR / "cost_sensitive_loss"

(EXPERIMENT_DIR / "models").mkdir(
    parents=True, exist_ok=True
)

(EXPERIMENT_DIR / "logs").mkdir(
    parents=True, exist_ok=True
)

(EXPERIMENT_DIR / "results").mkdir(
    parents=True, exist_ok=True
)

print("\nNew experiment directory:", EXPERIMENT_DIR)

print("\n✅ DRIVE AND PROJECT VERIFICATION COMPLETE.")


In [ ]:

from pathlib import Path
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix
)

# ============================================================
# 1. CONFIGURATION
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")

FEATURE_DIR = PROJECT_DIR / "patch_mil"
ORIGINAL_MODEL_DIR = PROJECT_DIR / "models"

EXPERIMENT_DIR = PROJECT_DIR / "cost_sensitive_loss"

NEW_MODEL_DIR = EXPERIMENT_DIR / "models"
LOG_DIR = EXPERIMENT_DIR / "logs"
RESULT_DIR = EXPERIMENT_DIR / "results"

for folder in [NEW_MODEL_DIR, LOG_DIR, RESULT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

GLOBAL_PATH = (
    FEATURE_DIR /
    "efficientnetb0_global_features_train_validation.npy"
)

LOCAL_PATH = (
    FEATURE_DIR /
    "efficientnetb0_patch_features_train_validation.npy"
)

MANIFEST_PATH = (
    FEATURE_DIR /
    "hybrid_global_local_train_validation_manifest.csv"
)

FINAL_MANIFEST_PATH = (
    PROJECT_DIR /
    "split_manifest_patient_grouped_final.csv"
)

BASELINE_MODEL_PATH = (
    ORIGINAL_MODEL_DIR /
    "bach_hybrid_global_local_best_accuracy.keras"
)

CLASS_NAMES = ["Benign", "InSitu", "Invasive", "Normal"]

NUM_CLASSES = 4

# Initial experimental hyperparameters
PENALTY_WEIGHT = 0.1
LEARNING_RATE = 1e-5

BATCH_SIZE = 8
MAX_EPOCHS = 40
SEED = 42

tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

# ============================================================
# 2. DEFINE THE PENALTY MATRIX
# ============================================================

# Rows = actual class
# Columns = predicted class
#
# Class order:
# 0 = Benign
# 1 = InSitu
# 2 = Invasive
# 3 = Normal

COST_MATRIX = np.array([
    [0, 3, 5, 1],
    [3, 0, 1, 5],
    [5, 1, 0, 5],
    [1, 5, 5, 0]
], dtype=np.float32)

cost_df = pd.DataFrame(
    COST_MATRIX,
    index=CLASS_NAMES,
    columns=CLASS_NAMES
)

print("\nPENALTY MATRIX")
print(cost_df)

cost_df.to_csv(
    EXPERIMENT_DIR / "penalty_matrix.csv"
)

experiment_config = {
    "experiment": "Cost-Sensitive Hybrid Global-Local MIL",
    "class_order": CLASS_NAMES,
    "penalty_matrix": COST_MATRIX.tolist(),
    "penalty_weight": PENALTY_WEIGHT,
    "learning_rate": LEARNING_RATE,
    "optimizer": "Adam",
    "loss": "Sparse Categorical Crossentropy + Expected Cost",
    "batch_size": BATCH_SIZE,
    "maximum_epochs": MAX_EPOCHS,
    "initial_checkpoint": BASELINE_MODEL_PATH.name,
    "patient_grouped_split": True,
    "penalty_values_status": (
        "Initial experimental values; some class-pair costs "
        "remain provisional pending professor confirmation."
    )
}

with open(
    EXPERIMENT_DIR / "experiment_config.json",
    "w"
) as f:
    json.dump(experiment_config, f, indent=4)

# ============================================================
# 3. LOAD FEATURES AND MANIFEST
# ============================================================

assert GLOBAL_PATH.exists()
assert LOCAL_PATH.exists()
assert MANIFEST_PATH.exists()
assert FINAL_MANIFEST_PATH.exists()
assert BASELINE_MODEL_PATH.exists()

global_features = np.load(
    GLOBAL_PATH,
    mmap_mode="r"
)

local_features = np.load(
    LOCAL_PATH,
    mmap_mode="r"
)

manifest = pd.read_csv(MANIFEST_PATH)
full_manifest = pd.read_csv(FINAL_MANIFEST_PATH)

assert global_features.shape == (340, 1280)
assert local_features.shape == (340, 88, 1280)
assert len(manifest) == 340

# Both feature arrays must belong to identical image ordering.
expected_splits = {"train": 280, "validation": 60}

assert manifest["split"].value_counts().to_dict() == expected_splits

# Check that the original final split contains the same
# training and validation assignments.
expected_manifest = (
    full_manifest[
        full_manifest["split"].isin(["train", "validation"])
    ]
    .reset_index(drop=True)
)

for column in ["filename", "label", "split", "group_id"]:
    assert (
        manifest[column].astype(str).tolist()
        == expected_manifest[column].astype(str).tolist()
    ), f"Manifest does not match final split: {column}"

# Verify patient-group separation.
train_groups = set(
    manifest.loc[
        manifest["split"] == "train", "group_id"
    ].astype(str)
)

val_groups = set(
    manifest.loc[
        manifest["split"] == "validation", "group_id"
    ].astype(str)
)

assert train_groups.isdisjoint(val_groups)

print("\n✅ Feature alignment verified.")
print("✅ Patient-group separation verified.")

# ============================================================
# 4. ENCODE LABELS AND PREPARE ARRAYS
# ============================================================

label_to_id = {
    name: index
    for index, name in enumerate(CLASS_NAMES)
}

assert set(manifest["label"]).issubset(label_to_id)

labels = np.array([
    label_to_id[label]
    for label in manifest["label"]
], dtype=np.int32)

train_mask = manifest["split"].to_numpy() == "train"
val_mask = manifest["split"].to_numpy() == "validation"

X_global_train = np.asarray(
    global_features[train_mask],
    dtype=np.float32
)

X_local_train = np.asarray(
    local_features[train_mask],
    dtype=np.float32
)

X_global_val = np.asarray(
    global_features[val_mask],
    dtype=np.float32
)

X_local_val = np.asarray(
    local_features[val_mask],
    dtype=np.float32
)

y_train = labels[train_mask]
y_val = labels[val_mask]

print("\nTraining arrays:")
print("Global:", X_global_train.shape)
print("Local:", X_local_train.shape)
print("Labels:", y_train.shape)

print("\nValidation arrays:")
print("Global:", X_global_val.shape)
print("Local:", X_local_val.shape)
print("Labels:", y_val.shape)

assert X_global_train.shape == (280, 1280)
assert X_local_train.shape == (280, 88, 1280)
assert X_global_val.shape == (60, 1280)
assert X_local_val.shape == (60, 88, 1280)

# ============================================================
# 5. EVALUATE ORIGINAL MODEL BEFORE TRAINING
# ============================================================

print("\nLoading original hybrid baseline...")

baseline_model = keras.models.load_model(
    BASELINE_MODEL_PATH,
    compile=False
)

validation_inputs = {
    "global_features": X_global_val,
    "local_features": X_local_val
}

baseline_probs = baseline_model.predict(
    validation_inputs,
    batch_size=8,
    verbose=0
)

baseline_pred = np.argmax(
    baseline_probs,
    axis=1
)

baseline_accuracy = accuracy_score(
    y_val,
    baseline_pred
)

baseline_macro_f1 = f1_score(
    y_val,
    baseline_pred,
    average="macro",
    zero_division=0
)

# Hard decision cost:
# Cost for the class selected by argmax.
baseline_decision_cost = COST_MATRIX[
    y_val,
    baseline_pred
]

# Expected cost:
# Weighted sum of all four predicted probabilities.
baseline_expected_cost = np.sum(
    COST_MATRIX[y_val] * baseline_probs,
    axis=1
)

print("\nORIGINAL MODEL — VALIDATION BASELINE")
print("Accuracy:", round(baseline_accuracy, 4))
print("Macro F1:", round(baseline_macro_f1, 4))
print(
    "Mean decision cost:",
    round(float(np.mean(baseline_decision_cost)), 4)
)
print(
    "Mean expected cost:",
    round(float(np.mean(baseline_expected_cost)), 4)
)

# Save baseline predictions.
baseline_df = pd.DataFrame({
    "filename": manifest.loc[val_mask, "filename"].to_numpy(),
    "actual": [CLASS_NAMES[i] for i in y_val],
    "predicted": [CLASS_NAMES[i] for i in baseline_pred],
    "correct": y_val == baseline_pred,
    "decision_cost": baseline_decision_cost,
    "expected_cost": baseline_expected_cost
})

for i, class_name in enumerate(CLASS_NAMES):
    baseline_df[f"prob_{class_name}"] = baseline_probs[:, i]

baseline_df.to_csv(
    RESULT_DIR / "baseline_validation_predictions.csv",
    index=False
)

# ============================================================
# 6. IMPLEMENT CUSTOM COST-SENSITIVE LOSS
# ============================================================

@keras.utils.register_keras_serializable(package="BACH")
class CostSensitiveSparseCategoricalCrossentropy(keras.losses.Loss):

    def __init__(
        self,
        cost_matrix,
        penalty_weight=0.1,
        name="cost_sensitive_sparse_categorical_crossentropy",
        reduction="sum_over_batch_size",
        **kwargs
    ):

        super().__init__(
            name=name,
            reduction=reduction,
            **kwargs
        )

        # Keep a Python copy for model serialization.
        self.cost_matrix_values = [
            [float(value) for value in row]
            for row in cost_matrix
        ]

        self.penalty_weight = float(penalty_weight)

        # TensorFlow constant used during training.
        self.cost_matrix = tf.constant(
            self.cost_matrix_values,
            dtype=tf.float32
        )

    def call(self, y_true, y_pred):

        # Convert labels into a flat integer vector.
        y_true = tf.cast(
            tf.reshape(y_true, [-1]),
            tf.int32
        )

        cost_matrix = tf.cast(
            self.cost_matrix,
            y_pred.dtype
        )

        # ---------------------------------------------
        # A. Standard sparse categorical crossentropy
        # ---------------------------------------------

        ce_loss = keras.losses.sparse_categorical_crossentropy(
            y_true,
            y_pred
        )

        # ---------------------------------------------
        # B. Select the appropriate cost row for each
        #    actual class.
        # ---------------------------------------------

        sample_costs = tf.gather(
            cost_matrix,
            y_true
        )

        # ---------------------------------------------
        # C. Expected classification penalty
        # ---------------------------------------------

        expected_cost = tf.reduce_sum(
            sample_costs * y_pred,
            axis=-1
        )

        # ---------------------------------------------
        # D. Combined objective
        # ---------------------------------------------

        total_loss = (
            ce_loss
            + self.penalty_weight * expected_cost
        )

        return total_loss

    def get_config(self):

        config = super().get_config()

        config.update({
            "cost_matrix": self.cost_matrix_values,
            "penalty_weight": self.penalty_weight
        })

        return config


# ============================================================
# 7. VERIFY LOSS CALCULATION BEFORE TRAINING
# ============================================================

example_actual = tf.constant([2], dtype=tf.int32)

example_prediction = tf.constant(
    [[0.05, 0.10, 0.15, 0.70]],
    dtype=tf.float32
)

cost_loss = CostSensitiveSparseCategoricalCrossentropy(
    cost_matrix=COST_MATRIX,
    penalty_weight=PENALTY_WEIGHT
)

example_loss = cost_loss(
    example_actual,
    example_prediction
)

example_ce = keras.losses.sparse_categorical_crossentropy(
    example_actual,
    example_prediction
)

print("\nLOSS FUNCTION SANITY CHECK")

print("Illustrative actual class: Invasive")
print("Illustrative probabilities:", example_prediction.numpy()[0])

print("Crossentropy:", float(example_ce.numpy()[0]))
print("Total cost-sensitive loss:", float(example_loss.numpy()[0]))

# Expected cost for this example:
# (0.05*5) + (0.10*1) + (0.15*0) + (0.70*5)
# = 3.35
#
# Total loss = -ln(0.15) + 0.1*3.35

expected_example_cost = np.sum(
    COST_MATRIX[2] * example_prediction.numpy()[0]
)

manual_example_loss = (
    -np.log(0.15)
    + PENALTY_WEIGHT * expected_example_cost
)

assert np.isclose(
    float(example_loss.numpy()[0]),
    manual_example_loss,
    atol=1e-5
)

print("Expected cost:", expected_example_cost)
print("Manual loss verification:", manual_example_loss)
print("✅ Custom loss calculation verified.")

# ============================================================
# 8. LOAD ORIGINAL WEIGHTS AND COMPILE NEW OBJECTIVE
# ============================================================

# Load the selected original model's weights.
# Do not initialize EfficientNet or the hybrid network again.
model = keras.models.load_model(
    BASELINE_MODEL_PATH,
    compile=False
)

custom_loss = CostSensitiveSparseCategoricalCrossentropy(
    cost_matrix=COST_MATRIX,
    penalty_weight=PENALTY_WEIGHT
)

# Recompile using a fresh Adam optimizer.
# This resets optimizer state but preserves model weights.
model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=LEARNING_RATE
    ),
    loss=custom_loss,
    metrics=["accuracy"]
)

print("\nModel loaded from original Epoch 27 checkpoint.")
print("Loss: Crossentropy +", PENALTY_WEIGHT, "* Expected cost")
print("Learning rate:", LEARNING_RATE)

# ============================================================
# 9. CHECKPOINTS AND CALLBACKS
# ============================================================

best_loss_path = (
    NEW_MODEL_DIR /
    "hybrid_cost_sensitive_best_loss.keras"
)

best_accuracy_path = (
    NEW_MODEL_DIR /
    "hybrid_cost_sensitive_best_accuracy.keras"
)

restored_path = (
    NEW_MODEL_DIR /
    "hybrid_cost_sensitive_earlystop_restored.keras"
)

training_log_path = (
    LOG_DIR /
    "hybrid_cost_sensitive_training_log.csv"
)

callbacks = [

    keras.callbacks.ModelCheckpoint(
        filepath=str(best_loss_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.ModelCheckpoint(
        filepath=str(best_accuracy_path),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),

    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.CSVLogger(
        filename=str(training_log_path),
        append=False
    )
]

# ============================================================
# 10. TRAIN COST-SENSITIVE HYBRID MODEL
# ============================================================

print("\n" + "=" * 65)
print("STARTING COST-SENSITIVE HYBRID TRAINING")
print("=" * 65)

history = model.fit(
    {
        "global_features": X_global_train,
        "local_features": X_local_train
    },
    y_train,

    validation_data=(
        {
            "global_features": X_global_val,
            "local_features": X_local_val
        },
        y_val
    ),

    batch_size=BATCH_SIZE,
    epochs=MAX_EPOCHS,
    shuffle=True,
    callbacks=callbacks,
    verbose=1
)

# Save the early-stopping-restored model.
model.save(restored_path)

# ============================================================
# 11. TRAINING SUMMARY
# ============================================================

best_val_accuracy = max(
    history.history["val_accuracy"]
)

best_val_loss = min(
    history.history["val_loss"]
)

best_accuracy_epoch = (
    int(np.argmax(history.history["val_accuracy"])) + 1
)

best_loss_epoch = (
    int(np.argmin(history.history["val_loss"])) + 1
)

print("\n" + "=" * 65)
print("COST-SENSITIVE TRAINING COMPLETE")
print("=" * 65)

print(
    f"Best validation accuracy: "
    f"{best_val_accuracy:.4f} "
    f"({best_val_accuracy*100:.2f}%)"
)

print("Best accuracy epoch:", best_accuracy_epoch)

print(f"Best validation loss: {best_val_loss:.4f}")
print("Best loss epoch:", best_loss_epoch)

print("\nPenalty weight:", PENALTY_WEIGHT)
print("Learning rate:", LEARNING_RATE)

print("\nSaved checkpoints:")
print(best_loss_path)
print(best_accuracy_path)
print(restored_path)

print("\nTraining log:")
print(training_log_path)

print("\n✅ COST-SENSITIVE EXPERIMENT COMPLETE.")


In [ ]:

# ============================================================
# CORRECTED COST-SENSITIVE LOSS SANITY CHECK
# ============================================================

example_actual = tf.constant([2], dtype=tf.int32)

example_prediction = tf.constant(
    [[0.05, 0.10, 0.15, 0.70]],
    dtype=tf.float32
)

example_loss = cost_loss(
    example_actual,
    example_prediction
)

example_ce = keras.losses.sparse_categorical_crossentropy(
    example_actual,
    example_prediction
)

# Expected cost for actual Invasive:
# 0.05*5 + 0.10*1 + 0.15*0 + 0.70*5
#
# = 0.25 + 0.10 + 0 + 3.50
# = 3.85

expected_example_cost = np.sum(
    COST_MATRIX[2] * example_prediction.numpy()[0]
)

manual_example_loss = (
    -np.log(0.15)
    + PENALTY_WEIGHT * expected_example_cost
)

print("LOSS FUNCTION SANITY CHECK")
print("-----------------------------------")

print("Actual class: Invasive")
print("Predicted probabilities:", example_prediction.numpy()[0])

print("Crossentropy:", float(example_ce.numpy()[0]))

print("Expected classification cost:",
      float(expected_example_cost))

print("Penalty contribution:",
      float(PENALTY_WEIGHT * expected_example_cost))

print("Total cost-sensitive loss:",
      float(example_loss.numpy()))

print("Manual calculation:",
      float(manual_example_loss))

assert np.isclose(
    float(example_loss.numpy()),
    manual_example_loss,
    atol=1e-5
)

print("\n✅ CUSTOM LOSS VERIFIED.")


In [ ]:

# ============================================================
# COST-SENSITIVE HYBRID MODEL TRAINING
# Continue from the verified loss-function cell
# ============================================================

# 1. Load original hybrid model weights
model = keras.models.load_model(
    BASELINE_MODEL_PATH,
    compile=False
)

print("Loaded original hybrid checkpoint:")
print(BASELINE_MODEL_PATH)

# 2. Compile with cost-sensitive loss
custom_loss = CostSensitiveSparseCategoricalCrossentropy(
    cost_matrix=COST_MATRIX,
    penalty_weight=PENALTY_WEIGHT
)

model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=LEARNING_RATE
    ),
    loss=custom_loss,
    metrics=["accuracy"]
)

# 3. Define separate checkpoint paths
best_loss_path = (
    NEW_MODEL_DIR /
    "hybrid_cost_sensitive_best_loss.keras"
)

best_accuracy_path = (
    NEW_MODEL_DIR /
    "hybrid_cost_sensitive_best_accuracy.keras"
)

restored_path = (
    NEW_MODEL_DIR /
    "hybrid_cost_sensitive_earlystop_restored.keras"
)

training_log_path = (
    LOG_DIR /
    "hybrid_cost_sensitive_training_log.csv"
)

# 4. Callbacks
callbacks = [

    keras.callbacks.ModelCheckpoint(
        filepath=str(best_loss_path),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.ModelCheckpoint(
        filepath=str(best_accuracy_path),
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),

    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.CSVLogger(
        filename=str(training_log_path),
        append=False
    )
]

# 5. Train
print("\n" + "=" * 65)
print("COST-SENSITIVE HYBRID MIL TRAINING")
print("=" * 65)

history_cost = model.fit(
    {
        "global_features": X_global_train,
        "local_features": X_local_train
    },
    y_train,

    validation_data=(
        {
            "global_features": X_global_val,
            "local_features": X_local_val
        },
        y_val
    ),

    batch_size=BATCH_SIZE,
    epochs=MAX_EPOCHS,
    shuffle=True,
    callbacks=callbacks,
    verbose=1
)

# 6. Save early-stopping-restored model
model.save(restored_path)

# 7. Report results
best_accuracy = max(
    history_cost.history["val_accuracy"]
)

best_loss = min(
    history_cost.history["val_loss"]
)

best_accuracy_epoch = (
    int(np.argmax(history_cost.history["val_accuracy"])) + 1
)

best_loss_epoch = (
    int(np.argmin(history_cost.history["val_loss"])) + 1
)

print("\n" + "=" * 65)
print("COST-SENSITIVE TRAINING RESULTS")
print("=" * 65)

print(
    f"Best validation accuracy: "
    f"{best_accuracy:.4f} ({best_accuracy*100:.2f}%)"
)

print("Best accuracy epoch:", best_accuracy_epoch)

print(f"Best validation loss: {best_loss:.6f}")
print("Best loss epoch:", best_loss_epoch)

print("Penalty weight:", PENALTY_WEIGHT)
print("Learning rate:", LEARNING_RATE)

print("\nSaved checkpoints:")
print(best_loss_path)
print(best_accuracy_path)
print(restored_path)

print("\nTraining log:")
print(training_log_path)

print("\n✅ COST-SENSITIVE TRAINING COMPLETE.")


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ============================================================
# 1. PATHS AND CONFIGURATION
# ============================================================

PROJECT_DIR = Path("/content/drive/MyDrive/BACH_Project")

FEATURE_DIR = PROJECT_DIR / "patch_mil"
EXPERIMENT_DIR = PROJECT_DIR / "cost_sensitive_loss"

MODEL_DIR = EXPERIMENT_DIR / "models"
RESULT_DIR = EXPERIMENT_DIR / "results"

RESULT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["Benign", "InSitu", "Invasive", "Normal"]

COST_MATRIX = np.array([
    [0, 3, 5, 1],
    [3, 0, 1, 5],
    [5, 1, 0, 5],
    [1, 5, 5, 0]
], dtype=np.float32)

LAMBDA = 0.1

# ============================================================
# 2. LOAD VALIDATION FEATURES
# ============================================================

global_features = np.load(
    FEATURE_DIR / "efficientnetb0_global_features_train_validation.npy",
    mmap_mode="r"
)

local_features = np.load(
    FEATURE_DIR / "efficientnetb0_patch_features_train_validation.npy",
    mmap_mode="r"
)

manifest = pd.read_csv(
    FEATURE_DIR / "hybrid_global_local_train_validation_manifest.csv"
)

val_mask = manifest["split"].to_numpy() == "validation"

val_manifest = (
    manifest.loc[val_mask]
    .reset_index(drop=True)
)

X_global_val = np.asarray(
    global_features[val_mask],
    dtype=np.float32
)

X_local_val = np.asarray(
    local_features[val_mask],
    dtype=np.float32
)

label_to_id = {
    name: i for i, name in enumerate(CLASS_NAMES)
}

y_true = np.array([
    label_to_id[label]
    for label in val_manifest["label"]
], dtype=np.int32)

validation_inputs = {
    "global_features": X_global_val,
    "local_features": X_local_val
}

assert len(y_true) == 60
assert X_global_val.shape == (60, 1280)
assert X_local_val.shape == (60, 88, 1280)

print("Validation images:", len(y_true))

# ============================================================
# 3. DEFINE THE THREE CHECKPOINTS
# ============================================================

checkpoints = {
    "Original_Hybrid_Baseline":
        PROJECT_DIR / "models" /
        "bach_hybrid_global_local_best_accuracy.keras",

    "Cost_Sensitive_Best_Accuracy":
        MODEL_DIR /
        "hybrid_cost_sensitive_best_accuracy.keras",

    "Cost_Sensitive_Best_Loss":
        MODEL_DIR /
        "hybrid_cost_sensitive_best_loss.keras"
}

# ============================================================
# 4. EVALUATE EVERY MODEL
# ============================================================

overall_results = []
class_results = []
prediction_tables = {}

for model_name, model_path in checkpoints.items():

    print("\n" + "=" * 70)
    print(model_name)
    print("=" * 70)

    assert model_path.exists(), f"Missing model: {model_path}"

    model = keras.models.load_model(
        model_path,
        compile=False
    )

    probabilities = model.predict(
        validation_inputs,
        batch_size=8,
        verbose=0
    )

    y_pred = np.argmax(probabilities, axis=1)

    # --------------------------------------------------------
    # A. CLASSIFICATION METRICS
    # --------------------------------------------------------

    accuracy = accuracy_score(y_true, y_pred)

    balanced_acc = balanced_accuracy_score(
        y_true, y_pred
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0
    )

    # --------------------------------------------------------
    # B. HARD DECISION PENALTY
    # --------------------------------------------------------

    hard_cost = COST_MATRIX[
        y_true,
        y_pred
    ]

    total_hard_cost = float(np.sum(hard_cost))
    mean_hard_cost = float(np.mean(hard_cost))

    # Count different penalty levels among mistakes.
    low_errors = int(np.sum(hard_cost == 1))
    medium_errors = int(np.sum(hard_cost == 3))
    high_errors = int(np.sum(hard_cost == 5))

    # --------------------------------------------------------
    # C. EXPECTED CLASSIFICATION PENALTY
    # --------------------------------------------------------

    expected_cost_per_image = np.sum(
        COST_MATRIX[y_true] * probabilities,
        axis=1
    )

    mean_expected_cost = float(
        np.mean(expected_cost_per_image)
    )

    total_expected_cost = float(
        np.sum(expected_cost_per_image)
    )

    # --------------------------------------------------------
    # D. ORDINARY CROSSENTROPY FOR DIAGNOSTICS
    # --------------------------------------------------------

    correct_class_probabilities = probabilities[
        np.arange(len(y_true)),
        y_true
    ]

    ce_per_image = -np.log(
        np.clip(
            correct_class_probabilities,
            1e-7,
            1.0
        )
    )

    mean_ce = float(np.mean(ce_per_image))

    # --------------------------------------------------------
    # E. PRINT RESULTS
    # --------------------------------------------------------

    print(f"\nAccuracy:          {accuracy:.4f}")
    print(f"Balanced accuracy: {balanced_acc:.4f}")
    print(f"Macro F1:          {macro_f1:.4f}")
    print(f"Weighted F1:       {weighted_f1:.4f}")

    print("\nHARD DECISION COST")
    print("Total penalty:", total_hard_cost)
    print("Mean penalty:", mean_hard_cost)
    print("Low-cost errors (1):", low_errors)
    print("Medium-cost errors (3):", medium_errors)
    print("High-cost errors (5):", high_errors)

    print("\nEXPECTED COST")
    print("Total expected cost:", total_expected_cost)
    print("Mean expected cost:", mean_expected_cost)

    print("\nMean ordinary crossentropy:", mean_ce)

    # --------------------------------------------------------
    # F. CLASSIFICATION REPORT
    # --------------------------------------------------------

    report = classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3],
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0
    )

    print("\nCLASSIFICATION REPORT")

    print(classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3],
        target_names=CLASS_NAMES,
        zero_division=0
    ))

    for class_name in CLASS_NAMES:

        class_results.append({
            "model": model_name,
            "class": class_name,
            "precision": report[class_name]["precision"],
            "recall": report[class_name]["recall"],
            "f1_score": report[class_name]["f1-score"],
            "support": report[class_name]["support"]
        })

    # --------------------------------------------------------
    # G. CONFUSION MATRIX
    # --------------------------------------------------------

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3]
    )

    print("\nCONFUSION MATRIX")
    print(pd.DataFrame(
        cm,
        index=CLASS_NAMES,
        columns=CLASS_NAMES
    ))

    pd.DataFrame(
        cm,
        index=CLASS_NAMES,
        columns=CLASS_NAMES
    ).to_csv(
        RESULT_DIR / f"{model_name}_confusion_matrix.csv"
    )

    # --------------------------------------------------------
    # H. SAVE OVERALL METRICS
    # --------------------------------------------------------

    overall_results.append({
        "model": model_name,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "total_hard_cost": total_hard_cost,
        "mean_hard_cost": mean_hard_cost,
        "low_cost_errors": low_errors,
        "medium_cost_errors": medium_errors,
        "high_cost_errors": high_errors,
        "total_expected_cost": total_expected_cost,
        "mean_expected_cost": mean_expected_cost,
        "mean_crossentropy": mean_ce
    })

    # --------------------------------------------------------
    # I. SAVE INDIVIDUAL PREDICTIONS
    # --------------------------------------------------------

    prediction_table = pd.DataFrame({
        "filename": val_manifest["filename"],
        "actual": [CLASS_NAMES[i] for i in y_true],
        "predicted": [CLASS_NAMES[i] for i in y_pred],
        "confidence": probabilities.max(axis=1),
        "correct": y_true == y_pred,
        "hard_cost": hard_cost,
        "expected_cost": expected_cost_per_image
    })

    for i, class_name in enumerate(CLASS_NAMES):
        prediction_table[f"prob_{class_name}"] = probabilities[:, i]

    prediction_table.to_csv(
        RESULT_DIR / f"{model_name}_predictions.csv",
        index=False
    )

    prediction_tables[model_name] = prediction_table

# ============================================================
# 5. OVERALL MODEL COMPARISON
# ============================================================

comparison = pd.DataFrame(overall_results)

print("\n" + "=" * 80)
print("COST-SENSITIVE MODEL COMPARISON")
print("=" * 80)

print(comparison.to_string(index=False))

comparison.to_csv(
    RESULT_DIR / "cost_sensitive_model_comparison.csv",
    index=False
)

# ============================================================
# 6. PER-CLASS COMPARISON
# ============================================================

per_class = pd.DataFrame(class_results)

print("\n" + "=" * 80)
print("PER-CLASS RECALL COMPARISON")
print("=" * 80)

print(
    per_class.pivot(
        index="class",
        columns="model",
        values="recall"
    ).to_string()
)

per_class.to_csv(
    RESULT_DIR / "cost_sensitive_per_class_metrics.csv",
    index=False
)

# ============================================================
# 7. SAVE EXPERIMENT SUMMARY
# ============================================================

summary = {
    "penalty_matrix": COST_MATRIX.tolist(),
    "penalty_weight": LAMBDA,
    "class_order": CLASS_NAMES,
    "models": overall_results
}

with open(
    RESULT_DIR / "cost_sensitive_validation_summary.json",
    "w"
) as f:
    import json
    json.dump(summary, f, indent=4)

print("\n✅ COST-SENSITIVE VALIDATION COMPARISON COMPLETE.")
print("Saved results:", RESULT_DIR)


In [ ]:

# ============================================================
# COST-SENSITIVE PENALTY WEIGHT EXPERIMENTS
# LAMBDA = 0.2 AND 0.5
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from sklearn.metrics import accuracy_score, f1_score

# These variables were defined in our previous cells:
# BASELINE_MODEL_PATH
# COST_MATRIX
# CostSensitiveSparseCategoricalCrossentropy
# X_global_train, X_local_train, y_train
# X_global_val, X_local_val, y_val
# NEW_MODEL_DIR, LOG_DIR, RESULT_DIR
# LEARNING_RATE, BATCH_SIZE

LAMBDA_VALUES = [0.2, 0.5]

MAX_EPOCHS = 40
SEED = 42

sweep_results = []

for current_lambda in LAMBDA_VALUES:

    print("\n" + "=" * 70)
    print("STARTING EXPERIMENT: LAMBDA =", current_lambda)
    print("=" * 70)

    # Reset seed for a controlled comparison.
    tf.keras.utils.set_random_seed(SEED)

    # Always start from the ORIGINAL Epoch 27 checkpoint.
    experiment_model = keras.models.load_model(
        BASELINE_MODEL_PATH,
        compile=False
    )

    # New cost-sensitive loss for this lambda.
    loss_function = CostSensitiveSparseCategoricalCrossentropy(
        cost_matrix=COST_MATRIX,
        penalty_weight=current_lambda
    )

    experiment_model.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=LEARNING_RATE
        ),
        loss=loss_function,
        metrics=["accuracy"]
    )

    # Unique filenames for each experiment.
    tag = str(current_lambda).replace(".", "_")

    best_loss_path = (
        NEW_MODEL_DIR /
        f"hybrid_cost_sensitive_lambda_{tag}_best_loss.keras"
    )

    best_accuracy_path = (
        NEW_MODEL_DIR /
        f"hybrid_cost_sensitive_lambda_{tag}_best_accuracy.keras"
    )

    restored_path = (
        NEW_MODEL_DIR /
        f"hybrid_cost_sensitive_lambda_{tag}_restored.keras"
    )

    log_path = (
        LOG_DIR /
        f"hybrid_cost_sensitive_lambda_{tag}_log.csv"
    )

    callbacks = [

        keras.callbacks.ModelCheckpoint(
            filepath=str(best_loss_path),
            monitor="val_loss",
            mode="min",
            save_best_only=True,
            verbose=1
        ),

        keras.callbacks.ModelCheckpoint(
            filepath=str(best_accuracy_path),
            monitor="val_accuracy",
            mode="max",
            save_best_only=True,
            verbose=1
        ),

        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=1
        ),

        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=8,
            restore_best_weights=True,
            verbose=1
        ),

        keras.callbacks.CSVLogger(
            filename=str(log_path),
            append=False
        )
    ]

    # Train from original checkpoint.
    history = experiment_model.fit(
        {
            "global_features": X_global_train,
            "local_features": X_local_train
        },
        y_train,

        validation_data=(
            {
                "global_features": X_global_val,
                "local_features": X_local_val
            },
            y_val
        ),

        batch_size=BATCH_SIZE,
        epochs=MAX_EPOCHS,
        shuffle=True,
        callbacks=callbacks,
        verbose=1
    )

    experiment_model.save(restored_path)

    # --------------------------------------------------------
    # Evaluate the best-loss checkpoint on validation.
    # --------------------------------------------------------

    evaluation_model = keras.models.load_model(
        best_loss_path,
        compile=False
    )

    val_probs = evaluation_model.predict(
        {
            "global_features": X_global_val,
            "local_features": X_local_val
        },
        batch_size=8,
        verbose=0
    )

    val_pred = np.argmax(val_probs, axis=1)

    hard_cost = COST_MATRIX[y_val, val_pred]

    expected_cost = np.sum(
        COST_MATRIX[y_val] * val_probs,
        axis=1
    )

    accuracy = accuracy_score(y_val, val_pred)

    macro_f1 = f1_score(
        y_val,
        val_pred,
        average="macro",
        zero_division=0
    )

    high_cost_errors = int(
        np.sum(hard_cost == 5)
    )

    # Save predictions for this lambda.
    prediction_table = pd.DataFrame({
        "actual": y_val,
        "predicted": val_pred,
        "correct": y_val == val_pred,
        "hard_cost": hard_cost,
        "expected_cost": expected_cost
    })

    prediction_table.to_csv(
        RESULT_DIR /
        f"lambda_{tag}_validation_predictions.csv",
        index=False
    )

    sweep_results.append({
        "lambda": current_lambda,
        "best_accuracy_during_training": max(
            history.history["val_accuracy"]
        ),
        "best_combined_val_loss": min(
            history.history["val_loss"]
        ),
        "evaluation_accuracy": accuracy,
        "macro_f1": macro_f1,
        "total_hard_cost": float(hard_cost.sum()),
        "mean_hard_cost": float(hard_cost.mean()),
        "high_cost_errors": high_cost_errors,
        "mean_expected_cost": float(expected_cost.mean())
    })

    print("\nRESULT FOR LAMBDA =", current_lambda)
    print("Accuracy:", accuracy)
    print("Macro F1:", macro_f1)
    print("Total hard penalty:", hard_cost.sum())
    print("High-cost errors:", high_cost_errors)
    print("Mean expected cost:", expected_cost.mean())

    del evaluation_model
    del experiment_model

# ============================================================
# FINAL LAMBDA COMPARISON
# ============================================================

sweep_df = pd.DataFrame(sweep_results)

sweep_df.to_csv(
    RESULT_DIR / "penalty_weight_sweep_comparison.csv",
    index=False
)

print("\n" + "=" * 80)
print("PENALTY WEIGHT SWEEP RESULTS")
print("=" * 80)

print(sweep_df.to_string(index=False))

print("\n✅ PENALTY WEIGHT EXPERIMENTS COMPLETE.")
print("Results saved:", RESULT_DIR)


In [ ]:

# ============================================================
# EXPERIMENT: COST-SENSITIVE DECISION RULE
# Compare argmax vs minimum expected penalty
# Validation only. Do NOT evaluate internal test here.
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Load both models
baseline_model = keras.models.load_model(
    BASELINE_MODEL_PATH,
    compile=False
)

lambda_05_model = keras.models.load_model(
    NEW_MODEL_DIR /
    "hybrid_cost_sensitive_lambda_0_5_best_loss.keras",
    compile=False
)

val_inputs = {
    "global_features": X_global_val,
    "local_features": X_local_val
}

# Generate probabilities
baseline_probs = baseline_model.predict(
    val_inputs, batch_size=8, verbose=0
)

lambda_05_probs = lambda_05_model.predict(
    val_inputs, batch_size=8, verbose=0
)

# ------------------------------------------------------------
# Two decision strategies
# ------------------------------------------------------------

def minimum_expected_cost(probabilities, cost_matrix):

    # For every possible predicted class, calculate:
    # R(j|x) = sum_i P(actual=i|x) * C[i,j]

    expected_risk = probabilities @ cost_matrix

    # Select the class with minimum expected penalty.
    predictions = np.argmin(expected_risk, axis=1)

    return predictions, expected_risk


def evaluate_decision_rule(name, y_true, probabilities,
                           predictions, cost_matrix):

    hard_cost = cost_matrix[y_true, predictions]

    expected_risk = (
        probabilities @ cost_matrix
    )

    chosen_risk = expected_risk[
        np.arange(len(y_true)),
        predictions
    ]

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print("Accuracy:",
          accuracy_score(y_true, predictions))

    print("Balanced accuracy:",
          balanced_accuracy_score(y_true, predictions))

    print("Macro F1:",
          f1_score(
              y_true,
              predictions,
              average="macro",
              zero_division=0
          ))

    print("Total hard penalty:", hard_cost.sum())
    print("Mean hard penalty:", hard_cost.mean())

    print("Low-cost errors:",
          np.sum(hard_cost == 1))

    print("Medium-cost errors:",
          np.sum(hard_cost == 3))

    print("High-cost errors:",
          np.sum(hard_cost == 5))

    print("Mean expected risk of chosen decisions:",
          chosen_risk.mean())

    print("\nClassification report:")

    print(classification_report(
        y_true,
        predictions,
        target_names=[
            "Benign", "InSitu", "Invasive", "Normal"
        ],
        zero_division=0
    ))

    print("Confusion matrix:")

    print(confusion_matrix(
        y_true,
        predictions
    ))

    return {
        "accuracy": accuracy_score(y_true, predictions),
        "balanced_accuracy": balanced_accuracy_score(
            y_true, predictions
        ),
        "macro_f1": f1_score(
            y_true, predictions,
            average="macro",
            zero_division=0
        ),
        "total_hard_cost": float(hard_cost.sum()),
        "mean_hard_cost": float(hard_cost.mean()),
        "high_cost_errors": int(np.sum(hard_cost == 5)),
        "mean_chosen_risk": float(chosen_risk.mean())
    }


results = []

experiments = [
    ("Original", baseline_probs),
    ("Lambda_0_5", lambda_05_probs)
]

for model_name, probabilities in experiments:

    # Ordinary classification
    argmax_predictions = np.argmax(
        probabilities, axis=1
    )

    argmax_result = evaluate_decision_rule(
        model_name + " | ARGMAX",
        y_val,
        probabilities,
        argmax_predictions,
        COST_MATRIX
    )

    argmax_result["model"] = model_name
    argmax_result["decision_rule"] = "argmax"

    results.append(argmax_result)

    # Cost-sensitive decision
    cost_predictions, risks = minimum_expected_cost(
        probabilities,
        COST_MATRIX
    )

    cost_result = evaluate_decision_rule(
        model_name + " | MINIMUM EXPECTED COST",
        y_val,
        probabilities,
        cost_predictions,
        COST_MATRIX
    )

    cost_result["model"] = model_name
    cost_result["decision_rule"] = "minimum_expected_cost"

    results.append(cost_result)

# ------------------------------------------------------------
# Final comparison
# ------------------------------------------------------------

comparison = pd.DataFrame(results)

print("\n" + "=" * 80)
print("DECISION RULE COMPARISON")
print("=" * 80)

print(comparison.to_string(index=False))

comparison.to_csv(
    RESULT_DIR / "decision_rule_comparison.csv",
    index=False
)

# Save predictions for detailed inspection
prediction_results = pd.DataFrame({
    "actual": y_val,
    "baseline_argmax": np.argmax(baseline_probs, axis=1),
    "baseline_cost_decision": minimum_expected_cost(
        baseline_probs, COST_MATRIX
    )[0],
    "lambda05_argmax": np.argmax(lambda_05_probs, axis=1),
    "lambda05_cost_decision": minimum_expected_cost(
        lambda_05_probs, COST_MATRIX
    )[0]
})

prediction_results.to_csv(
    RESULT_DIR / "decision_rule_predictions.csv",
    index=False
)

print("\n✅ DECISION RULE EXPERIMENT COMPLETE.")
print("Saved results:", RESULT_DIR)


In [ ]:

# ============================================================
# GRADIENT CONFLICT ANALYSIS
# Cross-Entropy vs Cost-Sensitive Loss
# ============================================================

import numpy as np
import tensorflow as tf
from tensorflow import keras

# Load original hybrid checkpoint.
# This experiment only computes gradients; no weights are updated.

gradient_model = keras.models.load_model(
    BASELINE_MODEL_PATH,
    compile=False
)

cost_matrix_tf = tf.constant(
    COST_MATRIX,
    dtype=tf.float32
)

variables = gradient_model.trainable_variables

print("Trainable parameters:",
      sum(int(tf.size(v)) for v in variables))

# Use 10 random batches from TRAINING data only.
rng = np.random.default_rng(42)

batch_size = 8
number_of_batches = 10

indices = rng.permutation(len(y_train))[
    :batch_size * number_of_batches
]

batch_indices = indices.reshape(
    number_of_batches, batch_size
)

cosine_values = []
norm_ratios = []

for batch_number, batch_idx in enumerate(batch_indices, 1):

    global_batch = tf.convert_to_tensor(
        X_global_train[batch_idx],
        dtype=tf.float32
    )

    local_batch = tf.convert_to_tensor(
        X_local_train[batch_idx],
        dtype=tf.float32
    )

    labels = tf.convert_to_tensor(
        y_train[batch_idx],
        dtype=tf.int32
    )

    inputs = {
        "global_features": global_batch,
        "local_features": local_batch
    }

    with tf.GradientTape(persistent=True) as tape:

        probabilities = gradient_model(
            inputs,
            training=True
        )

        # Component 1: ordinary cross-entropy
        ce_per_sample = (
            keras.losses.sparse_categorical_crossentropy(
                labels,
                probabilities
            )
        )

        ce_loss = tf.reduce_mean(ce_per_sample)

        # Component 2: expected classification penalty
        true_class_costs = tf.gather(
            cost_matrix_tf,
            labels
        )

        cost_per_sample = tf.reduce_sum(
            true_class_costs * probabilities,
            axis=1
        )

        cost_loss = tf.reduce_mean(cost_per_sample)

    # Calculate gradients separately.
    ce_gradients = tape.gradient(
        ce_loss,
        variables
    )

    cost_gradients = tape.gradient(
        cost_loss,
        variables
    )

    del tape

    # Flatten and combine all trainable-variable gradients.
    def flatten_gradients(gradients):

        flattened = []

        for gradient in gradients:

            if gradient is None:
                continue

            gradient = tf.convert_to_tensor(gradient)

            flattened.append(
                tf.reshape(gradient, [-1])
            )

        if not flattened:
            raise ValueError(
                "No gradients were produced."
            )

        return tf.concat(flattened, axis=0)

    g_ce = flatten_gradients(ce_gradients)
    g_cost = flatten_gradients(cost_gradients)

    dot_product = tf.reduce_sum(g_ce * g_cost)

    ce_norm = tf.norm(g_ce)
    cost_norm = tf.norm(g_cost)

    cosine = dot_product / (
        ce_norm * cost_norm + 1e-12
    )

    cosine_value = float(cosine.numpy())

    # Relative influence of the penalty gradient
    # when lambda = 0.5.
    relative_cost_norm = float(
        (0.5 * cost_norm / (ce_norm + 1e-12)).numpy()
    )

    cosine_values.append(cosine_value)
    norm_ratios.append(relative_cost_norm)

    print(
        f"Batch {batch_number:02d} | "
        f"CE loss: {float(ce_loss.numpy()):.4f} | "
        f"Cost loss: {float(cost_loss.numpy()):.4f} | "
        f"Cosine: {cosine_value:+.4f} | "
        f"Lambda-weighted norm ratio: "
        f"{relative_cost_norm:.4f}"
    )

print("\n" + "=" * 70)
print("GRADIENT CONFLICT SUMMARY")
print("=" * 70)

print("Mean cosine similarity:",
      np.mean(cosine_values))

print("Negative cosine batches:",
      sum(c < 0 for c in cosine_values),
      "/", len(cosine_values))

print("Mean lambda-weighted gradient norm ratio:",
      np.mean(norm_ratios))

print("\n✅ Gradient analysis complete. No model weights were changed.")


In [ ]:

# ============================================================
# FULL TRAINING-SET GRADIENT ALIGNMENT AUDIT
# All 280 images | 35 batches | deterministic forward pass
# No training or weight updates
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from pathlib import Path

audit_model = keras.models.load_model(
    BASELINE_MODEL_PATH,
    compile=False
)

variables = audit_model.trainable_variables

cost_matrix_tf = tf.constant(
    COST_MATRIX,
    dtype=tf.float32
)

rng = np.random.default_rng(42)
indices = rng.permutation(len(y_train))

BATCH_SIZE_AUDIT = 8
records = []

for batch_number, start in enumerate(
    range(0, len(indices), BATCH_SIZE_AUDIT), 1
):

    batch_idx = indices[
        start:start + BATCH_SIZE_AUDIT
    ]

    global_batch = tf.convert_to_tensor(
        X_global_train[batch_idx],
        dtype=tf.float32
    )

    local_batch = tf.convert_to_tensor(
        X_local_train[batch_idx],
        dtype=tf.float32
    )

    labels = tf.convert_to_tensor(
        y_train[batch_idx],
        dtype=tf.int32
    )

    inputs = {
        "global_features": global_batch,
        "local_features": local_batch
    }

    # training=False disables dropout.
    with tf.GradientTape(persistent=True) as tape:

        probabilities = audit_model(
            inputs,
            training=False
        )

        ce_loss = tf.reduce_mean(
            keras.losses.sparse_categorical_crossentropy(
                labels,
                probabilities
            )
        )

        true_costs = tf.gather(
            cost_matrix_tf,
            labels
        )

        cost_loss = tf.reduce_mean(
            tf.reduce_sum(
                true_costs * probabilities,
                axis=1
            )
        )

    ce_grads = tape.gradient(
        ce_loss,
        variables
    )

    cost_grads = tape.gradient(
        cost_loss,
        variables
    )

    del tape

    # Keep matching variable-gradient pairs.
    paired = [
        (g1, g2)
        for g1, g2 in zip(ce_grads, cost_grads)
        if g1 is not None and g2 is not None
    ]

    g_ce = tf.concat(
        [
            tf.reshape(g1, [-1])
            for g1, g2 in paired
        ],
        axis=0
    )

    g_cost = tf.concat(
        [
            tf.reshape(g2, [-1])
            for g1, g2 in paired
        ],
        axis=0
    )

    dot = tf.reduce_sum(g_ce * g_cost)

    norm_ce = tf.norm(g_ce)
    norm_cost = tf.norm(g_cost)

    cosine = dot / (
        norm_ce * norm_cost + 1e-12
    )

    weighted_ratio = (
        0.5 * norm_cost / (norm_ce + 1e-12)
    )

    records.append({
        "batch": batch_number,
        "ce_loss": float(ce_loss.numpy()),
        "cost_loss": float(cost_loss.numpy()),
        "cosine_similarity": float(cosine.numpy()),
        "lambda_0_5_norm_ratio": float(
            weighted_ratio.numpy()
        )
    })

audit_df = pd.DataFrame(records)

print("\n" + "=" * 75)
print("FULL TRAINING GRADIENT AUDIT")
print("=" * 75)

print(audit_df.to_string(index=False))

print("\nSUMMARY")
print("Total batches:", len(audit_df))

print(
    "Mean cosine:",
    audit_df["cosine_similarity"].mean()
)

print(
    "Median cosine:",
    audit_df["cosine_similarity"].median()
)

print(
    "Negative batches:",
    (audit_df["cosine_similarity"] < 0).sum()
)

print(
    "Mean weighted norm ratio:",
    audit_df["lambda_0_5_norm_ratio"].mean()
)

RESULT_DIR = Path(RESULT_DIR)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

audit_df.to_csv(
    RESULT_DIR / "full_train_gradient_alignment_audit.csv",
    index=False
)

print("\n✅ Full gradient audit complete.")
print("No model weights were modified.")


In [ ]:

# ============================================================
# ADAM PRECONDITIONER DIRECTIONAL EFFECT AUDIT
# Fixed original hybrid checkpoint
# All 280 training images
# No model weight updates
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from pathlib import Path

# Load the original checkpoint.
direction_model = keras.models.load_model(
    BASELINE_MODEL_PATH,
    compile=False
)

variables = direction_model.trainable_variables

BETA_1 = 0.9
BETA_2 = 0.999
EPSILON = 1e-7
LAMBDA = 0.5

cost_matrix_tf = tf.constant(
    COST_MATRIX,
    dtype=tf.float32
)

# Initialize simulated Adam state.
m = [tf.zeros_like(var) for var in variables]
v = [tf.zeros_like(var) for var in variables]

rng = np.random.default_rng(42)

batch_indices = rng.permutation(len(y_train))

BATCH_SIZE = 8
records = []

def dot_product(grad_a, grad_b):
    terms = [
        tf.reduce_sum(a * b)
        for a, b in zip(grad_a, grad_b)
    ]
    return tf.add_n(terms)

def gradient_norm(gradients):
    return tf.sqrt(
        tf.maximum(
            dot_product(gradients, gradients),
            0.0
        )
    )

for step, start in enumerate(
    range(0, len(batch_indices), BATCH_SIZE),
    start=1
):

    idx = batch_indices[
        start:start + BATCH_SIZE
    ]

    x_global = tf.convert_to_tensor(
        X_global_train[idx],
        dtype=tf.float32
    )

    x_local = tf.convert_to_tensor(
        X_local_train[idx],
        dtype=tf.float32
    )

    labels = tf.convert_to_tensor(
        y_train[idx],
        dtype=tf.int32
    )

    inputs = {
        "global_features": x_global,
        "local_features": x_local
    }

    # Fixed weights; dropout disabled.
    with tf.GradientTape(persistent=True) as tape:

        probabilities = direction_model(
            inputs,
            training=False
        )

        ce_loss = tf.reduce_mean(
            keras.losses.sparse_categorical_crossentropy(
                labels,
                probabilities
            )
        )

        true_costs = tf.gather(
            cost_matrix_tf,
            labels
        )

        cost_loss = tf.reduce_mean(
            tf.reduce_sum(
                true_costs * probabilities,
                axis=1
            )
        )

    ce_grads = tape.gradient(
        ce_loss,
        variables
    )

    cost_grads = tape.gradient(
        cost_loss,
        variables
    )

    del tape

    # Replace disconnected gradients with zeros.
    ce_grads = [
        tf.zeros_like(var) if grad is None else grad
        for grad, var in zip(ce_grads, variables)
    ]

    cost_grads = [
        tf.zeros_like(var) if grad is None else grad
        for grad, var in zip(cost_grads, variables)
    ]

    combined_grads = [
        gce + LAMBDA * gcost
        for gce, gcost in zip(
            ce_grads, cost_grads
        )
    ]

    # Simulate Adam's moment updates.
    m = [
        BETA_1 * old_m + (1.0 - BETA_1) * grad
        for old_m, grad in zip(m, combined_grads)
    ]

    v = [
        BETA_2 * old_v + (1.0 - BETA_2) * tf.square(grad)
        for old_v, grad in zip(v, combined_grads)
    ]

    # Bias correction.
    m_hat = [
        moment / (1.0 - BETA_1 ** step)
        for moment in m
    ]

    v_hat = [
        moment / (1.0 - BETA_2 ** step)
        for moment in v
    ]

    # Adam's approximate applied direction.
    adam_direction = [
        mh / (tf.sqrt(vh) + EPSILON)
        for mh, vh in zip(m_hat, v_hat)
    ]

    # Compare each objective's gradient with
    # the direction Adam would apply.
    ce_dot = float(
        dot_product(
            ce_grads, adam_direction
        ).numpy()
    )

    cost_dot = float(
        dot_product(
            cost_grads, adam_direction
        ).numpy()
    )

    # Cosine similarity with the applied direction.
    adam_norm = gradient_norm(
        adam_direction
    )

    ce_cosine = float(
        (
            ce_dot /
            (gradient_norm(ce_grads) * adam_norm + 1e-12)
        ).numpy()
    )

    cost_cosine = float(
        (
            cost_dot /
            (gradient_norm(cost_grads) * adam_norm + 1e-12)
        ).numpy()
    )

    records.append({
        "batch": step,
        "ce_loss": float(ce_loss.numpy()),
        "cost_loss": float(cost_loss.numpy()),
        "ce_dot_adam_direction": ce_dot,
        "cost_dot_adam_direction": cost_dot,
        "ce_direction_cosine": ce_cosine,
        "cost_direction_cosine": cost_cosine
    })

audit_df = pd.DataFrame(records)

print("\n" + "=" * 80)
print("ADAM PRECONDITIONER DIRECTIONAL AUDIT")
print("=" * 80)

print(audit_df.to_string(index=False))

print("\nSUMMARY")

print(
    "CE direction negative batches:",
    (audit_df["ce_dot_adam_direction"] < 0).sum()
)

print(
    "Cost direction negative batches:",
    (audit_df["cost_dot_adam_direction"] < 0).sum()
)

print(
    "Mean CE direction cosine:",
    audit_df["ce_direction_cosine"].mean()
)

print(
    "Mean cost direction cosine:",
    audit_df["cost_direction_cosine"].mean()
)

print(
    "Minimum CE directional dot:",
    audit_df["ce_dot_adam_direction"].min()
)

print(
    "Minimum cost directional dot:",
    audit_df["cost_dot_adam_direction"].min()
)

save_path = (
    Path(RESULT_DIR) /
    "adam_preconditioner_directional_audit.csv"
)

audit_df.to_csv(save_path, index=False)

print("\n✅ Audit complete. Model weights were not modified.")
print("Saved:", save_path)


In [ ]:

# ============================================================
# ACTUAL ADAM TRAINING TRAJECTORY AUDIT
#
# Original Hybrid checkpoint
# Lambda = 0.5
# Learning rate = 1e-5
# Batch size = 8
# One actual training epoch
#
# The loaded model is a temporary experimental copy.
# The original checkpoint will not be overwritten.
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from pathlib import Path

tf.keras.utils.set_random_seed(42)

LAMBDA = 0.5
LEARNING_RATE = 1e-5
BATCH_SIZE = 8

# Load a fresh original checkpoint.
trajectory_model = keras.models.load_model(
    BASELINE_MODEL_PATH,
    compile=False
)

variables = trajectory_model.trainable_variables

optimizer = keras.optimizers.Adam(
    learning_rate=LEARNING_RATE,
    beta_1=0.9,
    beta_2=0.999,
    epsilon=1e-7
)

# Initialize optimizer state.
optimizer.build(variables)

cost_matrix_tf = tf.constant(
    COST_MATRIX,
    dtype=tf.float32
)

def calculate_components(probabilities, labels):

    ce = tf.reduce_mean(
        keras.losses.sparse_categorical_crossentropy(
            labels,
            probabilities
        )
    )

    true_costs = tf.gather(
        cost_matrix_tf,
        labels
    )

    cost = tf.reduce_mean(
        tf.reduce_sum(
            true_costs * probabilities,
            axis=1
        )
    )

    return ce, cost


def gradient_dot(a, b):

    valid_pairs = [
        (x, y)
        for x, y in zip(a, b)
        if x is not None and y is not None
    ]

    return tf.add_n([
        tf.reduce_sum(x * y)
        for x, y in valid_pairs
    ])


def gradient_norm(g):

    return tf.sqrt(
        tf.maximum(
            gradient_dot(g, g),
            0.0
        )
    )


# One shuffled pass through the training set.
rng = np.random.default_rng(42)

indices = rng.permutation(len(y_train))

records = []

for step, start in enumerate(
    range(0, len(indices), BATCH_SIZE),
    start=1
):

    batch_idx = indices[
        start:start + BATCH_SIZE
    ]

    inputs = {
        "global_features": tf.convert_to_tensor(
            X_global_train[batch_idx],
            dtype=tf.float32
        ),
        "local_features": tf.convert_to_tensor(
            X_local_train[batch_idx],
            dtype=tf.float32
        )
    }

    labels = tf.convert_to_tensor(
        y_train[batch_idx],
        dtype=tf.int32
    )

    # Deterministic pre-update evaluation.
    pre_probs = trajectory_model(
        inputs,
        training=False
    )

    pre_ce, pre_cost = calculate_components(
        pre_probs, labels
    )

    # Actual stochastic training forward pass.
    with tf.GradientTape(persistent=True) as tape:

        probabilities = trajectory_model(
            inputs,
            training=True
        )

        ce_loss, cost_loss = calculate_components(
            probabilities,
            labels
        )

        total_loss = (
            ce_loss + LAMBDA * cost_loss
        )

    ce_grads = tape.gradient(
        ce_loss,
        variables
    )

    cost_grads = tape.gradient(
        cost_loss,
        variables
    )

    total_grads = tape.gradient(
        total_loss,
        variables
    )

    del tape

    # Preserve the parameters immediately before the update.
    active = [
        (var, gce, gcost, gtotal)
        for var, gce, gcost, gtotal
        in zip(
            variables,
            ce_grads,
            cost_grads,
            total_grads
        )
        if gtotal is not None
    ]

    active_vars = [item[0] for item in active]

    active_ce = [item[1] for item in active]
    active_cost = [item[2] for item in active]
    active_total = [item[3] for item in active]

    before = [
        tf.identity(var)
        for var in active_vars
    ]

    # Perform the REAL Adam update.
    optimizer.apply_gradients(
        zip(active_total, active_vars)
    )

    # Descent direction = old parameters - new parameters.
    descent_direction = [
        old - new
        for old, new in zip(before, active_vars)
    ]

    ce_dot = float(
        gradient_dot(
            active_ce,
            descent_direction
        ).numpy()
    )

    cost_dot = float(
        gradient_dot(
            active_cost,
            descent_direction
        ).numpy()
    )

    ce_cos = float(
        (
            ce_dot /
            (
                gradient_norm(active_ce) *
                gradient_norm(descent_direction)
                + 1e-12
            )
        ).numpy()
    )

    cost_cos = float(
        (
            cost_dot /
            (
                gradient_norm(active_cost) *
                gradient_norm(descent_direction)
                + 1e-12
            )
        ).numpy()
    )

    # Evaluate both losses after the actual parameter update.
    post_probs = trajectory_model(
        inputs,
        training=False
    )

    post_ce, post_cost = calculate_components(
        post_probs, labels
    )

    records.append({

        "batch": step,

        "ce_before": float(pre_ce.numpy()),
        "ce_after": float(post_ce.numpy()),
        "ce_change": float(
            (post_ce - pre_ce).numpy()
        ),

        "cost_before": float(pre_cost.numpy()),
        "cost_after": float(post_cost.numpy()),
        "cost_change": float(
            (post_cost - pre_cost).numpy()
        ),

        "ce_update_dot": ce_dot,
        "cost_update_dot": cost_dot,

        "ce_update_cosine": ce_cos,
        "cost_update_cosine": cost_cos
    })

    if step % 5 == 0:
        print(
            f"Batch {step:02d} | "
            f"CE change: {float((post_ce-pre_ce).numpy()):+.6f} | "
            f"Cost change: {float((post_cost-pre_cost).numpy()):+.6f}"
        )


# Final report.
trajectory_df = pd.DataFrame(records)

print("\n" + "=" * 80)
print("ACTUAL ADAM TRAINING TRAJECTORY SUMMARY")
print("=" * 80)

print("Batches:", len(trajectory_df))

print(
    "CE uphill first-order updates:",
    (trajectory_df["ce_update_dot"] < 0).sum()
)

print(
    "Cost uphill first-order updates:",
    (trajectory_df["cost_update_dot"] < 0).sum()
)

print(
    "CE finite-step increases:",
    (trajectory_df["ce_change"] > 0).sum()
)

print(
    "Cost finite-step increases:",
    (trajectory_df["cost_change"] > 0).sum()
)

print(
    "Mean CE directional cosine:",
    trajectory_df["ce_update_cosine"].mean()
)

print(
    "Mean cost directional cosine:",
    trajectory_df["cost_update_cosine"].mean()
)

print(
    "Mean CE change:",
    trajectory_df["ce_change"].mean()
)

print(
    "Mean cost change:",
    trajectory_df["cost_change"].mean()
)

save_path = (
    Path(RESULT_DIR) /
    "actual_adam_training_trajectory_audit.csv"
)

trajectory_df.to_csv(
    save_path,
    index=False
)

print("\n✅ Actual trajectory audit complete.")
print("Experimental model was not saved.")
print("Original checkpoint remains unchanged.")
print("Results:", save_path)


In [ ]:

# ============================================================
# EXPERIMENT: SEPARATE-MOMENT ADAM (SM-ADAM)
#
# CE and cost objectives maintain independent Adam moments.
# Both updates are combined at the parameter-update stage.
#
# Starting point: ORIGINAL HYBRID CHECKPOINT
# Training: 280 images
# Validation: 60 images
# Internal test: untouched
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)
from pathlib import Path

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

SEED = 42

UPDATE_WEIGHT = 0.5
EVAL_COST_WEIGHT = 0.5

INITIAL_LR = 1e-5
MIN_LR = 1e-7

BETA_1 = 0.9
BETA_2 = 0.999
EPSILON = 1e-7

BATCH_SIZE = 8
MAX_EPOCHS = 40

EARLY_STOPPING_PATIENCE = 8
LR_PATIENCE = 3
LR_FACTOR = 0.5

tf.keras.utils.set_random_seed(SEED)

# ------------------------------------------------------------
# 2. Load fresh original model
# ------------------------------------------------------------

sm_model = keras.models.load_model(
    BASELINE_MODEL_PATH,
    compile=False
)

trainable_variables = sm_model.trainable_variables

print(
    "Trainable parameters:",
    sum(int(tf.size(v)) for v in trainable_variables)
)

# ------------------------------------------------------------
# 3. Implement Separate-Moment Adam
# ------------------------------------------------------------

class SeparateMomentAdam:

    def __init__(
        self,
        variables,
        learning_rate=1e-5,
        update_weight=0.5,
        beta1=0.9,
        beta2=0.999,
        epsilon=1e-7
    ):

        self.learning_rate = tf.Variable(
            learning_rate,
            dtype=tf.float32,
            trainable=False
        )

        self.update_weight = tf.constant(
            update_weight,
            dtype=tf.float32
        )

        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon

        self.iterations = tf.Variable(
            0,
            dtype=tf.int64,
            trainable=False
        )

        # Independent first and second moments
        # for cross-entropy gradients.
        self.m_ce = [
            tf.Variable(
                tf.zeros_like(v),
                trainable=False
            )
            for v in variables
        ]

        self.v_ce = [
            tf.Variable(
                tf.zeros_like(v),
                trainable=False
            )
            for v in variables
        ]

        # Independent first and second moments
        # for cost-sensitive gradients.
        self.m_cost = [
            tf.Variable(
                tf.zeros_like(v),
                trainable=False
            )
            for v in variables
        ]

        self.v_cost = [
            tf.Variable(
                tf.zeros_like(v),
                trainable=False
            )
            for v in variables
        ]

    def apply_gradients(
        self,
        gradients_ce,
        gradients_cost,
        variables
    ):

        self.iterations.assign_add(1)

        t = tf.cast(
            self.iterations,
            tf.float32
        )

        beta1 = tf.constant(
            self.beta1,
            dtype=tf.float32
        )

        beta2 = tf.constant(
            self.beta2,
            dtype=tf.float32
        )

        correction1 = 1.0 - tf.pow(beta1, t)
        correction2 = 1.0 - tf.pow(beta2, t)

        for i, variable in enumerate(variables):

            g_ce = gradients_ce[i]
            g_cost = gradients_cost[i]

            if g_ce is None:
                g_ce = tf.zeros_like(variable)

            if g_cost is None:
                g_cost = tf.zeros_like(variable)

            # Cross-entropy moment updates
            self.m_ce[i].assign(
                beta1 * self.m_ce[i]
                + (1.0 - beta1) * g_ce
            )

            self.v_ce[i].assign(
                beta2 * self.v_ce[i]
                + (1.0 - beta2) * tf.square(g_ce)
            )

            # Cost-loss moment updates
            self.m_cost[i].assign(
                beta1 * self.m_cost[i]
                + (1.0 - beta1) * g_cost
            )

            self.v_cost[i].assign(
                beta2 * self.v_cost[i]
                + (1.0 - beta2) * tf.square(g_cost)
            )

            # Bias-corrected estimates
            m_ce_hat = (
                self.m_ce[i] / correction1
            )

            v_ce_hat = (
                self.v_ce[i] / correction2
            )

            m_cost_hat = (
                self.m_cost[i] / correction1
            )

            v_cost_hat = (
                self.v_cost[i] / correction2
            )

            # Separate adaptive directions
            update_ce = (
                m_ce_hat /
                (tf.sqrt(v_ce_hat) + self.epsilon)
            )

            update_cost = (
                m_cost_hat /
                (tf.sqrt(v_cost_hat) + self.epsilon)
            )

            # Combined parameter update
            combined_update = (
                update_ce
                + self.update_weight * update_cost
            )

            variable.assign_sub(
                self.learning_rate * combined_update
            )


sm_optimizer = SeparateMomentAdam(
    variables=trainable_variables,
    learning_rate=INITIAL_LR,
    update_weight=UPDATE_WEIGHT,
    beta1=BETA_1,
    beta2=BETA_2,
    epsilon=EPSILON
)

# ------------------------------------------------------------
# 4. Prepare tensors
# ------------------------------------------------------------

Xg_train = tf.convert_to_tensor(
    X_global_train,
    dtype=tf.float32
)

Xl_train = tf.convert_to_tensor(
    X_local_train,
    dtype=tf.float32
)

yt_train = tf.convert_to_tensor(
    y_train,
    dtype=tf.int32
)

Xg_val = tf.convert_to_tensor(
    X_global_val,
    dtype=tf.float32
)

Xl_val = tf.convert_to_tensor(
    X_local_val,
    dtype=tf.float32
)

yt_val = tf.convert_to_tensor(
    y_val,
    dtype=tf.int32
)

cost_matrix_tf = tf.constant(
    COST_MATRIX,
    dtype=tf.float32
)

# ------------------------------------------------------------
# 5. Loss calculation
# ------------------------------------------------------------

def calculate_losses(probabilities, labels):

    ce_loss = tf.reduce_mean(
        keras.losses.sparse_categorical_crossentropy(
            labels,
            probabilities
        )
    )

    true_class_costs = tf.gather(
        cost_matrix_tf,
        labels
    )

    cost_loss = tf.reduce_mean(
        tf.reduce_sum(
            true_class_costs * probabilities,
            axis=1
        )
    )

    return ce_loss, cost_loss


# ------------------------------------------------------------
# 6. Output directories
# ------------------------------------------------------------

model_dir = Path(NEW_MODEL_DIR)
log_dir = Path(LOG_DIR)
result_dir = Path(RESULT_DIR)

best_objective_path = (
    model_dir /
    "hybrid_sm_adam_best_objective.keras"
)

best_accuracy_path = (
    model_dir /
    "hybrid_sm_adam_best_accuracy.keras"
)

# ------------------------------------------------------------
# 7. Training and validation loop
# ------------------------------------------------------------

rng = np.random.default_rng(SEED)

history_records = []

best_objective = np.inf
best_accuracy = -np.inf

early_wait = 0
lr_wait = 0

for epoch in range(1, MAX_EPOCHS + 1):

    print("\n" + "=" * 70)
    print(f"SM-ADAM EPOCH {epoch}/{MAX_EPOCHS}")
    print("=" * 70)

    # Shuffle training images.
    indices = rng.permutation(len(y_train))

    epoch_ce = 0.0
    epoch_cost = 0.0
    seen = 0

    for start in range(0, len(indices), BATCH_SIZE):

        idx = indices[start:start + BATCH_SIZE]

        global_batch = tf.gather(
            Xg_train, idx
        )

        local_batch = tf.gather(
            Xl_train, idx
        )

        label_batch = tf.gather(
            yt_train, idx
        )

        batch_inputs = {
            "global_features": global_batch,
            "local_features": local_batch
        }

        # Calculate both gradients independently.
        with tf.GradientTape(persistent=True) as tape:

            probabilities = sm_model(
                batch_inputs,
                training=True
            )

            ce_loss, cost_loss = calculate_losses(
                probabilities,
                label_batch
            )

        gradients_ce = tape.gradient(
            ce_loss,
            trainable_variables
        )

        gradients_cost = tape.gradient(
            cost_loss,
            trainable_variables
        )

        del tape

        # Apply the custom update.
        sm_optimizer.apply_gradients(
            gradients_ce,
            gradients_cost,
            trainable_variables
        )

        batch_size_actual = len(idx)

        epoch_ce += (
            float(ce_loss.numpy()) * batch_size_actual
        )

        epoch_cost += (
            float(cost_loss.numpy()) * batch_size_actual
        )

        seen += batch_size_actual

    train_ce = epoch_ce / seen
    train_cost = epoch_cost / seen

    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    val_probs = sm_model(
        {
            "global_features": Xg_val,
            "local_features": Xl_val
        },
        training=False
    ).numpy()

    val_ce = float(
        tf.reduce_mean(
            keras.losses.sparse_categorical_crossentropy(
                yt_val,
                val_probs
            )
        ).numpy()
    )

    val_cost = float(
        np.mean(
            np.sum(
                COST_MATRIX[y_val] * val_probs,
                axis=1
            )
        )
    )

    # Same validation objective used by the
    # combined-loss Adam experiment.
    val_objective = (
        val_ce + EVAL_COST_WEIGHT * val_cost
    )

    predictions = np.argmax(
        val_probs,
        axis=1
    )

    val_accuracy = accuracy_score(
        y_val,
        predictions
    )

    val_balanced_accuracy = balanced_accuracy_score(
        y_val,
        predictions
    )

    val_macro_f1 = f1_score(
        y_val,
        predictions,
        average="macro",
        zero_division=0
    )

    hard_cost = COST_MATRIX[
        y_val,
        predictions
    ]

    mean_hard_cost = float(
        np.mean(hard_cost)
    )

    mean_expected_cost = val_cost

    print(f"Train CE:           {train_ce:.5f}")
    print(f"Train cost:         {train_cost:.5f}")
    print(f"Validation CE:      {val_ce:.5f}")
    print(f"Validation cost:    {val_cost:.5f}")
    print(f"Combined objective: {val_objective:.5f}")
    print(f"Accuracy:           {val_accuracy:.4f}")
    print(f"Balanced accuracy:  {val_balanced_accuracy:.4f}")
    print(f"Macro F1:           {val_macro_f1:.4f}")
    print(f"Mean hard cost:     {mean_hard_cost:.5f}")
    print(f"High-cost errors:   {int(np.sum(hard_cost == 5))}")
    print(
        "Learning rate:",
        float(sm_optimizer.learning_rate.numpy())
    )

    history_records.append({
        "epoch": epoch,
        "train_ce": train_ce,
        "train_cost": train_cost,
        "val_ce": val_ce,
        "val_cost": val_cost,
        "val_objective": val_objective,
        "val_accuracy": val_accuracy,
        "val_balanced_accuracy": val_balanced_accuracy,
        "val_macro_f1": val_macro_f1,
        "mean_hard_cost": mean_hard_cost,
        "high_cost_errors": int(
            np.sum(hard_cost == 5)
        ),
        "learning_rate": float(
            sm_optimizer.learning_rate.numpy()
        )
    })

    # Save best combined validation objective.
    if val_objective < best_objective:

        best_objective = val_objective

        sm_model.save(
            best_objective_path
        )

        early_wait = 0
        lr_wait = 0

        print("✅ New best validation objective saved.")

    else:

        early_wait += 1
        lr_wait += 1

    # Save best accuracy checkpoint separately.
    if val_accuracy > best_accuracy:

        best_accuracy = val_accuracy

        sm_model.save(
            best_accuracy_path
        )

        print("✅ New best accuracy checkpoint saved.")

    # Reduce learning rate on plateau.
    if lr_wait >= LR_PATIENCE:

        new_lr = max(
            float(sm_optimizer.learning_rate.numpy())
            * LR_FACTOR,
            MIN_LR
        )

        sm_optimizer.learning_rate.assign(
            new_lr
        )

        lr_wait = 0

        print(
            "Learning rate reduced to:",
            new_lr
        )

    # Early stopping.
    if early_wait >= EARLY_STOPPING_PATIENCE:

        print(
            "\nEarly stopping triggered."
        )

        break


# ------------------------------------------------------------
# 8. Save history
# ------------------------------------------------------------

history_df = pd.DataFrame(
    history_records
)

history_path = (
    log_dir /
    "hybrid_sm_adam_training_log.csv"
)

history_df.to_csv(
    history_path,
    index=False
)

print("\n" + "=" * 80)
print("SM-ADAM EXPERIMENT COMPLETED")
print("=" * 80)

print("Best validation objective:", best_objective)
print("Best validation accuracy:", best_accuracy)

print("Best objective checkpoint:")
print(best_objective_path)

print("Best accuracy checkpoint:")
print(best_accuracy_path)

print("Training log:")
print(history_path)


In [ ]:

# ============================================================
# FINAL VALIDATION COMPARISON: ADAM vs SM-ADAM
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from pathlib import Path

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

model_dir = Path(NEW_MODEL_DIR)

models_to_compare = {
    "Original_Hybrid": BASELINE_MODEL_PATH,

    "Combined_Adam_Lambda_0_5": (
        model_dir /
        "hybrid_cost_sensitive_lambda_0_5_best_loss.keras"
    ),

    "SM_Adam_Best_Objective": (
        model_dir /
        "hybrid_sm_adam_best_objective.keras"
    ),

    "SM_Adam_Best_Accuracy": (
        model_dir /
        "hybrid_sm_adam_best_accuracy.keras"
    )
}

validation_inputs = {
    "global_features": tf.convert_to_tensor(
        X_global_val, dtype=tf.float32
    ),
    "local_features": tf.convert_to_tensor(
        X_local_val, dtype=tf.float32
    )
}

comparison_results = []
prediction_results = {}

for model_name, model_path in models_to_compare.items():

    print("\n" + "=" * 75)
    print(model_name)
    print("=" * 75)

    model_path = Path(model_path)

    if not model_path.exists():
        print("CHECKPOINT NOT FOUND:", model_path)
        continue

    model = keras.models.load_model(
        model_path,
        compile=False
    )

    probabilities = model(
        validation_inputs,
        training=False
    ).numpy()

    predictions = np.argmax(
        probabilities,
        axis=1
    )

    # Ordinary cross-entropy
    true_probabilities = probabilities[
        np.arange(len(y_val)),
        y_val
    ]

    ce = float(
        np.mean(
            -np.log(
                np.clip(true_probabilities, 1e-7, 1.0)
            )
        )
    )

    # Expected classification penalty
    expected_cost = float(
        np.mean(
            np.sum(
                COST_MATRIX[y_val] * probabilities,
                axis=1
            )
        )
    )

    # Hard decision penalty
    hard_cost = COST_MATRIX[
        y_val, predictions
    ]

    accuracy = accuracy_score(
        y_val, predictions
    )

    balanced_acc = balanced_accuracy_score(
        y_val, predictions
    )

    macro_f1 = f1_score(
        y_val,
        predictions,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        y_val,
        predictions,
        average="weighted",
        zero_division=0
    )

    combined_objective = (
        ce + 0.5 * expected_cost
    )

    print("Accuracy:", accuracy)
    print("Balanced accuracy:", balanced_acc)
    print("Macro F1:", macro_f1)
    print("Ordinary CE:", ce)
    print("Expected cost:", expected_cost)
    print("CE + 0.5 * expected cost:", combined_objective)

    print("Total hard penalty:", hard_cost.sum())
    print("Mean hard penalty:", hard_cost.mean())
    print("High-cost errors:", np.sum(hard_cost == 5))

    print("\nConfusion matrix:")
    print(confusion_matrix(y_val, predictions))

    print("\nClassification report:")
    print(
        classification_report(
            y_val,
            predictions,
            target_names=[
                "Benign",
                "InSitu",
                "Invasive",
                "Normal"
            ],
            zero_division=0
        )
    )

    comparison_results.append({
        "model": model_name,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "mean_crossentropy": ce,
        "mean_expected_cost": expected_cost,
        "combined_objective": combined_objective,
        "total_hard_cost": float(hard_cost.sum()),
        "mean_hard_cost": float(hard_cost.mean()),
        "high_cost_errors": int(
            np.sum(hard_cost == 5)
        )
    })

    prediction_results[model_name] = predictions

    del model

# Final comparison table
comparison_df = pd.DataFrame(
    comparison_results
)

print("\n" + "=" * 100)
print("COMPLETE OPTIMIZER COMPARISON")
print("=" * 100)

print(
    comparison_df.to_string(index=False)
)

# Save results
result_dir = Path(RESULT_DIR)

comparison_df.to_csv(
    result_dir / "adam_vs_sm_adam_comparison.csv",
    index=False
)

prediction_df = pd.DataFrame({
    "actual": y_val,
    **prediction_results
})

prediction_df.to_csv(
    result_dir / "adam_vs_sm_adam_predictions.csv",
    index=False
)

print("\n✅ Comparison complete.")
print("Saved:", result_dir)
